In [ ]:
# ============================================================
# CELL 1 — CONFIG, IMPORTS, LOAD CASE TABLES
# NEW dataset | box2deg | 2009–2025 UPDATED
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import gaussian_kde

from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier

# ============================================================
# CONFIG
# ============================================================

DATASET_NAME = "new"
LABEL_NAME = "box2deg"

CASE_DIR = Path(
    r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest\exactlead_box2deg_case_tables_new_2009_2025_UPDATED"
)

OUT_DIR = Path(
    r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest\ml_new_box2deg_2009_2025_UPDATED"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR = OUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CM_DIR = OUT_DIR / "confusion_matrices"
CM_DIR.mkdir(parents=True, exist_ok=True)

FI_DIR = OUT_DIR / "feature_importance"
FI_DIR.mkdir(parents=True, exist_ok=True)

LEADS = [6, 12, 24, 48]
RANDOM_STATE = 42
N_SPLITS = 5

FEATURE_COLS = [
    "lon", "lat", "month",
    "area_km2", "tb_mean", "tb_min",
    "speed_kmh", "direction_deg",
    "eta850_abs", "rh600", "wshear_200_850",
    "pi_vmax_ms", "gpi",
]

ID_COLS = [
    "track_id", "time",
    "centroid_lon", "centroid_lat",
]

print("CASE_DIR:", CASE_DIR)
print("OUT_DIR:", OUT_DIR)

# ============================================================
# LOAD CASE TABLES
# ============================================================

datasets = {}
qc_rows = []

for L in LEADS:
    fp = CASE_DIR / f"{LABEL_NAME}_case_table_{DATASET_NAME}_{L}h.csv"

    if not fp.exists():
        raise FileNotFoundError(fp)

    df = pd.read_csv(fp, low_memory=False)
    df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)
    df["y"] = df["y"].astype(int)

    datasets[L] = df.copy()

    qc_rows.append({
        "lead_h": L,
        "rows": len(df),
        "positive": int(df["y"].sum()),
        "negative": int((df["y"] == 0).sum()),
        "pos_frac": float(df["y"].mean()),
        "time_min": df["time"].min(),
        "time_max": df["time"].max(),
        "unique_tracks": df["track_id"].nunique(),
        "any_predictor_nan_frac": float(df[FEATURE_COLS].isna().any(axis=1).mean()),
        "speed_nan_frac": float(df["speed_kmh"].isna().mean()),
        "direction_nan_frac": float(df["direction_deg"].isna().mean()),
        "pi_nan_frac": float(df["pi_vmax_ms"].isna().mean()),
        "gpi_nan_frac": float(df["gpi"].isna().mean()),
    })

qc = pd.DataFrame(qc_rows)
qc.to_csv(OUT_DIR / "case_table_qc_new.csv", index=False)

display(qc)


In [ ]:
# ============================================================
# CELL 2 — MODELS AND HELPER FUNCTIONS
# ============================================================

def get_models(random_state=42):
    return {
        "Logistic Regression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="lbfgs",
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=600,
            random_state=random_state,
            class_weight="balanced",
            n_jobs=-1,
        ),

        "AdaBoost": AdaBoostClassifier(
            n_estimators=600,
            random_state=random_state,
        ),

        "MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            alpha=1e-4,
            learning_rate_init=0.001,
            max_iter=1000,
            random_state=random_state,
        ),

        "KNN": KNeighborsClassifier(
            n_neighbors=5,
        ),

        "Naive Bayes": GaussianNB(),

        "SVM": SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=random_state,
        ),

        "QDA": QuadraticDiscriminantAnalysis(
            reg_param=0.01,
        ),

        "Decision Tree": DecisionTreeClassifier(
            random_state=random_state,
            class_weight="balanced",
        ),
    }


def model_needs_scaling(model_name):
    return model_name in [
        "Logistic Regression",
        "MLP",
        "KNN",
        "Naive Bayes",
        "SVM",
        "QDA",
    ]


def make_pipeline(model_name, model):
    steps = [("imputer", SimpleImputer(strategy="median"))]

    if model_needs_scaling(model_name):
        steps.append(("scaler", StandardScaler()))

    steps.append(("model", model))
    return Pipeline(steps)


def get_cv(y, groups=None, n_splits=5, random_state=42):
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    n_splits_eff = min(n_splits, n_pos, n_neg)

    if n_splits_eff < 2:
        raise ValueError("Not enough positive/negative samples for CV.")

    if groups is not None:
        return StratifiedGroupKFold(
            n_splits=n_splits_eff,
            shuffle=True,
            random_state=random_state,
        ), "StratifiedGroupKFold", n_splits_eff

    return StratifiedKFold(
        n_splits=n_splits_eff,
        shuffle=True,
        random_state=random_state,
    ), "StratifiedKFold", n_splits_eff


def safe_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    out = {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_score)
    except Exception:
        out["roc_auc"] = np.nan

    try:
        out["pr_auc"] = average_precision_score(y_true, y_score)
    except Exception:
        out["pr_auc"] = np.nan

    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    except Exception:
        tn, fp, fn, tp = np.nan, np.nan, np.nan, np.nan

    out["tn"] = tn
    out["fp"] = fp
    out["fn"] = fn
    out["tp"] = tp

    return out


def threshold_scan(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    valid = np.isfinite(y_score)
    y_true = y_true[valid]
    y_score = y_score[valid]

    if len(y_true) == 0:
        return pd.DataFrame()

    thresholds = np.unique(np.nanpercentile(y_score, np.linspace(0, 100, 501)))

    rows = []
    for th in thresholds:
        rows.append(safe_metrics(y_true, y_score, threshold=th))

    scan = pd.DataFrame(rows)

    return (
        scan
        .sort_values(["f1", "recall", "precision"], ascending=False)
        .reset_index(drop=True)
    )


In [ ]:
# ============================================================
# CELL 3 — DEFINE MANUAL 5-FOLD CV RUNNER
# ============================================================

def run_one_model_cv(
    case_df,
    lead_h,
    model_name,
    model,
    dataset_name=DATASET_NAME,
    label_name=LABEL_NAME,
    random_state=RANDOM_STATE,
    n_splits=N_SPLITS,
):
    df = case_df.copy()
    df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

    X = df[FEATURE_COLS].copy()
    y = df["y"].astype(int).values
    groups = df["track_id"].astype(str).values if "track_id" in df.columns else None

    cv, cv_name, n_splits_eff = get_cv(
        y=y,
        groups=groups,
        n_splits=n_splits,
        random_state=random_state,
    )

    pipe = make_pipeline(model_name, model)

    fold_rows = []
    oof_rows = []

    splitter = cv.split(X, y, groups) if groups is not None else cv.split(X, y)

    for fold_id, (train_idx, test_idx) in enumerate(splitter, start=1):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        pipe.fit(X_train, y_train)

        if hasattr(pipe, "predict_proba"):
            y_score = pipe.predict_proba(X_test)[:, 1]
        else:
            y_score = pipe.decision_function(X_test)
            y_score = (
                (y_score - np.nanmin(y_score))
                / (np.nanmax(y_score) - np.nanmin(y_score) + 1e-12)
            )

        fold_metric = safe_metrics(y_test, y_score, threshold=0.5)

        fold_metric.update({
            "dataset": dataset_name,
            "label": label_name,
            "lead_h": lead_h,
            "model": model_name,
            "fold": fold_id,
            "cv": cv_name,
            "n_splits": n_splits_eff,
            "train_n": len(train_idx),
            "test_n": len(test_idx),
            "train_pos": int(y_train.sum()),
            "test_pos": int(y_test.sum()),
            "train_neg": int((y_train == 0).sum()),
            "test_neg": int((y_test == 0).sum()),
        })

        fold_rows.append(fold_metric)

        oof = df.iloc[test_idx][ID_COLS].copy()
        oof["dataset"] = dataset_name
        oof["label"] = label_name
        oof["lead_h"] = lead_h
        oof["model"] = model_name
        oof["fold"] = fold_id
        oof["y_true"] = y_test
        oof["y_score"] = y_score
        oof["y_pred_default"] = (y_score >= 0.5).astype(int)

        oof_rows.append(oof)

    fold_df = pd.DataFrame(fold_rows)
    oof_df = pd.concat(oof_rows, ignore_index=True)

    default_metrics = safe_metrics(
        oof_df["y_true"].values,
        oof_df["y_score"].values,
        threshold=0.5,
    )

    default_row = {
        "dataset": dataset_name,
        "label": label_name,
        "lead_h": lead_h,
        "model": model_name,
        "metric_type": "oof_default_0.5",
        **default_metrics,
    }

    scan = threshold_scan(
        oof_df["y_true"].values,
        oof_df["y_score"].values,
    )

    best_threshold = float(scan.iloc[0]["threshold"]) if len(scan) > 0 else 0.5

    oof_df["y_pred_tuned"] = (oof_df["y_score"] >= best_threshold).astype(int)

    tuned_metrics = safe_metrics(
        oof_df["y_true"].values,
        oof_df["y_score"].values,
        threshold=best_threshold,
    )

    tuned_row = {
        "dataset": dataset_name,
        "label": label_name,
        "lead_h": lead_h,
        "model": model_name,
        "metric_type": "oof_tuned_threshold",
        **tuned_metrics,
    }

    return fold_df, oof_df, pd.DataFrame([default_row, tuned_row]), scan


In [ ]:
# ============================================================
# CELL 4 — RUN ALL MODELS / ALL LEADS
# ============================================================

models = get_models(random_state=RANDOM_STATE)

all_fold_results = []
all_oof_results = []
all_oof_summary = []
all_threshold_scans = []
error_rows = []

for L in LEADS:
    case_df = datasets[L].copy()

    print("\n" + "=" * 80)
    print(
        f"LEAD {L}h | rows={len(case_df)} | "
        f"positives={int(case_df['y'].sum())} | "
        f"negatives={int((case_df['y'] == 0).sum())}"
    )
    print("=" * 80)

    for model_name, model in models.items():
        print(f"\nRunning model: {model_name}")

        try:
            fold_df, oof_df, oof_summary_df, scan_df = run_one_model_cv(
                case_df=case_df,
                lead_h=L,
                model_name=model_name,
                model=model,
                dataset_name=DATASET_NAME,
                label_name=LABEL_NAME,
                random_state=RANDOM_STATE,
                n_splits=N_SPLITS,
            )

            all_fold_results.append(fold_df)
            all_oof_results.append(oof_df)
            all_oof_summary.append(oof_summary_df)

            scan_df["dataset"] = DATASET_NAME
            scan_df["label"] = LABEL_NAME
            scan_df["lead_h"] = L
            scan_df["model"] = model_name
            all_threshold_scans.append(scan_df)

            safe_model_name = model_name.replace(" ", "_").replace("/", "_")

            oof_df.to_csv(
                OUT_DIR / f"oof_{DATASET_NAME}_{LABEL_NAME}_{L}h_{safe_model_name}.csv",
                index=False,
            )

            scan_df.to_csv(
                OUT_DIR / f"threshold_scan_{DATASET_NAME}_{LABEL_NAME}_{L}h_{safe_model_name}.csv",
                index=False,
            )

            default_f1 = float(
                oof_summary_df.loc[
                    oof_summary_df["metric_type"] == "oof_default_0.5",
                    "f1"
                ].iloc[0]
            )

            default_pr_auc = float(
                oof_summary_df.loc[
                    oof_summary_df["metric_type"] == "oof_default_0.5",
                    "pr_auc"
                ].iloc[0]
            )

            print("  F1:", default_f1, "| PR-AUC:", default_pr_auc)

        except Exception as e:
            print("[FAILED]", model_name, "|", repr(e))
            error_rows.append({
                "lead_h": L,
                "model": model_name,
                "error": repr(e),
            })

if not all_fold_results:
    raise RuntimeError("No model completed successfully.")

fold_results = pd.concat(all_fold_results, ignore_index=True)
oof_all = pd.concat(all_oof_results, ignore_index=True)
oof_summary = pd.concat(all_oof_summary, ignore_index=True)
threshold_scans = pd.concat(all_threshold_scans, ignore_index=True) if all_threshold_scans else pd.DataFrame()
errors = pd.DataFrame(error_rows)

fold_results.to_csv(OUT_DIR / "fold_results_all_models_new.csv", index=False)
oof_all.to_csv(OUT_DIR / "oof_predictions_all_models.csv", index=False)
oof_all.to_csv(OUT_DIR / "oof_predictions_allmodels_new.csv", index=False)
oof_summary.to_csv(OUT_DIR / "oof_summary_all_models_new.csv", index=False)
threshold_scans.to_csv(OUT_DIR / "threshold_scans_all_models_new.csv", index=False)
errors.to_csv(OUT_DIR / "model_errors_new.csv", index=False)

print("\nSaved:")
print(OUT_DIR / "fold_results_all_models_new.csv")
print(OUT_DIR / "oof_predictions_all_models.csv")
print(OUT_DIR / "oof_predictions_allmodels_new.csv")
print(OUT_DIR / "oof_summary_all_models_new.csv")
print(OUT_DIR / "threshold_scans_all_models_new.csv")


In [ ]:
# ============================================================
# CELL 5 — SUMMARY TABLES
# ============================================================

metric_cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]

summary_rows = []

for (lead_h, model), g in fold_results.groupby(["lead_h", "model"]):
    row = {
        "dataset": DATASET_NAME,
        "label": LABEL_NAME,
        "lead_h": lead_h,
        "model": model,
        "folds": g["fold"].nunique(),
        "samples_mean": g["test_n"].mean(),
        "test_pos_mean": g["test_pos"].mean(),
        "test_neg_mean": g["test_neg"].mean(),
    }

    for metric in metric_cols:
        row[f"{metric}_mean"] = g[metric].mean()
        row[f"{metric}_std"] = g[metric].std()

    summary_rows.append(row)

cv_summary = pd.DataFrame(summary_rows)
cv_summary = cv_summary.sort_values(["lead_h", "f1_mean"], ascending=[True, False]).reset_index(drop=True)

cv_summary.to_csv(OUT_DIR / "cv_summary_mean_std_all_models.csv", index=False)
cv_summary.to_csv(OUT_DIR / "cv_summary_mean_std_all_models_new.csv", index=False)

ranked_table = cv_summary[
    [
        "lead_h", "model",
        "precision_mean", "recall_mean", "f1_mean",
        "roc_auc_mean", "pr_auc_mean",
    ]
].copy()

ranked_table = ranked_table.rename(columns={
    "precision_mean": "precision",
    "recall_mean": "recall",
    "f1_mean": "f1",
    "roc_auc_mean": "roc_auc",
    "pr_auc_mean": "pr_auc",
})

ranked_table = ranked_table.sort_values(
    ["lead_h", "f1"],
    ascending=[True, False]
).reset_index(drop=True)

ranked_table.to_csv(OUT_DIR / "model_performance_ranked_by_lead_new.csv", index=False)

best_by_lead = (
    ranked_table
    .sort_values(["lead_h", "f1"], ascending=[True, False])
    .groupby("lead_h", as_index=False)
    .first()
)

best_by_lead.to_csv(OUT_DIR / "best_model_by_lead_new.csv", index=False)

print("\nRanked table:")
display(ranked_table)

print("\nBest model by lead:")
display(best_by_lead)


In [ ]:
# ============================================================
# CELL 6 — BAR PLOTS + TOP MODEL LINE PLOTS
# ============================================================

MODEL_ORDER = [
    "Logistic Regression",
    "Naive Bayes",
    "KNN",
    "SVM",
    "Decision Tree",
    "MLP",
    "QDA",
    "Random Forest",
    "AdaBoost",
]

summary_plot = cv_summary.copy()
summary_plot["model"] = pd.Categorical(
    summary_plot["model"],
    categories=MODEL_ORDER,
    ordered=True,
)
summary_plot = summary_plot.sort_values(["model", "lead_h"]).copy()

lead_order = [6, 12, 24, 48]


def make_bar_metric(metric_col, ylabel, title, outname):
    piv = summary_plot.pivot(index="model", columns="lead_h", values=metric_col)
    piv = piv.reindex(MODEL_ORDER)
    piv = piv[lead_order]

    x = np.arange(len(piv.index))
    width = 0.18

    fig, ax = plt.subplots(figsize=(12, 5.5))

    for i, lead in enumerate(lead_order):
        ax.bar(
            x + (i - 1.5) * width,
            piv[lead],
            width=width,
            label=f"{lead} h",
        )

    ax.set_xticks(x)
    ax.set_xticklabels(piv.index, rotation=30, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Model")
    ax.set_title(title)
    ax.legend(title="Lead")
    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(PLOT_DIR / outname, dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved:", PLOT_DIR / outname)


make_bar_metric(
    metric_col="f1_mean",
    ylabel="F1 score",
    title="NEW dataset: F1 score by model and lead time",
    outname="bar_f1_by_model_lead_new.png",
)

make_bar_metric(
    metric_col="precision_mean",
    ylabel="Precision",
    title="NEW dataset: Precision by model and lead time",
    outname="bar_precision_by_model_lead_new.png",
)

make_bar_metric(
    metric_col="recall_mean",
    ylabel="Recall",
    title="NEW dataset: Recall by model and lead time",
    outname="bar_recall_by_model_lead_new.png",
)

top_models = (
    cv_summary.groupby("model")["f1_mean"]
    .mean()
    .sort_values(ascending=False)
    .head(4)
    .index
    .tolist()
)

print("Top models:", top_models)


def make_line_top(metric_col, ylabel, title, outname):
    fig, ax = plt.subplots(figsize=(8, 5))

    for model in top_models:
        sub = cv_summary[cv_summary["model"] == model].sort_values("lead_h")

        ax.plot(
            sub["lead_h"],
            sub[metric_col],
            marker="o",
            linewidth=2,
            label=model,
        )

    ax.set_xticks(LEADS)
    ax.set_xlabel("Lead time (h)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(PLOT_DIR / outname, dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved:", PLOT_DIR / outname)


make_line_top(
    metric_col="recall_mean",
    ylabel="Recall",
    title="NEW dataset: Top models — Recall by lead time",
    outname="top_models_recall_new.png",
)

make_line_top(
    metric_col="precision_mean",
    ylabel="Precision",
    title="NEW dataset: Top models — Precision by lead time",
    outname="top_models_precision_new.png",
)

make_line_top(
    metric_col="f1_mean",
    ylabel="F1 score",
    title="NEW dataset: Top models — F1 by lead time",
    outname="top_models_f1_new.png",
)


In [ ]:
# ============================================================
# CELL 7 — GPI KDE + GPI THRESHOLD CLASSIFIER
# ============================================================

GPI_COL = "gpi"

# ------------------------------------------------------------
# GPI KDE
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes = axes.flatten()

for ax, lead in zip(axes, LEADS):
    df = datasets[lead].copy()
    df = df.dropna(subset=[GPI_COL])

    neg = df.loc[df["y"] == 0, GPI_COL].astype(float).values
    pos = df.loc[df["y"] == 1, GPI_COL].astype(float).values

    if len(neg) > 5 and np.nanmax(neg) > np.nanmin(neg):
        xs = np.linspace(np.nanmin(neg), np.nanmax(neg), 300)
        kde = gaussian_kde(neg)
        ax.plot(xs, kde(xs), linewidth=2, label="Negative")

    if len(pos) > 5 and np.nanmax(pos) > np.nanmin(pos):
        xs = np.linspace(np.nanmin(pos), np.nanmax(pos), 300)
        kde = gaussian_kde(pos)
        ax.plot(xs, kde(xs), linewidth=2, label="Positive")

    ax.set_title(f"Lead {lead} h")
    ax.set_xlabel("GPI")
    ax.set_ylabel("Density")
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle("NEW dataset: KDE of GPI for negative and positive cases", fontsize=14)
plt.savefig(PLOT_DIR / "gpi_kde_positive_negative_new.png", dpi=220, bbox_inches="tight")
plt.show()

print("Saved:", PLOT_DIR / "gpi_kde_positive_negative_new.png")

# ------------------------------------------------------------
# GPI threshold classifier
# ------------------------------------------------------------

rows = []

for lead in LEADS:
    df = datasets[lead].copy()
    df = df.dropna(subset=[GPI_COL])

    y_true = df["y"].astype(int).values
    gpi = df[GPI_COL].astype(float).values

    if len(df) == 0:
        continue

    thresholds = np.linspace(np.nanmin(gpi), np.nanmax(gpi), 200)

    for thr in thresholds:
        y_pred = (gpi >= thr).astype(int)

        rows.append({
            "lead_h": lead,
            "threshold": thr,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
        })

gpi_perf = pd.DataFrame(rows)
gpi_perf.to_csv(OUT_DIR / "gpi_threshold_performance_new.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

for ax, metric in zip(axes, ["f1", "precision", "recall"]):
    for lead in LEADS:
        sub = gpi_perf[gpi_perf["lead_h"] == lead]
        ax.plot(sub["threshold"], sub[metric], linewidth=2, label=f"{lead} h")

    ax.set_title(metric.upper())
    ax.set_xlabel("GPI threshold")
    ax.set_ylabel(metric.upper())
    ax.grid(True, alpha=0.3)

axes[0].legend(title="Lead")
fig.suptitle("NEW dataset: Performance of GPI threshold classifier", fontsize=14)
plt.savefig(PLOT_DIR / "gpi_threshold_classifier_performance_new.png", dpi=220, bbox_inches="tight")
plt.show()

best_gpi = (
    gpi_perf.sort_values(["lead_h", "f1"], ascending=[True, False])
    .groupby("lead_h", as_index=False)
    .first()
)

best_gpi.to_csv(OUT_DIR / "best_gpi_threshold_by_lead_new.csv", index=False)

print("\nBest GPI threshold by lead:")
display(best_gpi[["lead_h", "threshold", "f1", "precision", "recall"]])


In [ ]:
# ============================================================
# CELL 8 — CONFUSION MATRICES
# 1. Best model per lead
# 2. 6h 1x5 raw and normalized: SVM, AdaBoost, RF, Logistic, MLP
# ============================================================

# ------------------------------------------------------------
# Best model per lead
# ------------------------------------------------------------

cm_rows = []

for _, row in best_by_lead.iterrows():
    lead = int(row["lead_h"])
    model_name = row["model"]

    sub = oof_all[
        (oof_all["lead_h"] == lead)
        & (oof_all["model"] == model_name)
    ].copy()

    y_true = sub["y_true"].astype(int).values
    y_pred = sub["y_pred_default"].astype(int).values

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    cm_rows.append({
        "lead_h": lead,
        "model": model_name,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "total": int(cm.sum()),
    })

    fig, ax = plt.subplots(figsize=(5.5, 4.5))

    ConfusionMatrixDisplay(
        cm,
        display_labels=["Non-developing", "Developing"],
    ).plot(ax=ax, colorbar=False, values_format="d")

    ax.set_title(f"NEW dataset | {model_name} | Lead {lead} h")
    plt.tight_layout()

    safe_model = model_name.replace(" ", "_").replace("/", "_")
    out_png = CM_DIR / f"confusion_matrix_new_{safe_model}_{lead}h.png"

    plt.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.show()

    cm_norm = confusion_matrix(y_true, y_pred, labels=[0, 1], normalize="true")

    fig, ax = plt.subplots(figsize=(5.5, 4.5))

    ConfusionMatrixDisplay(
        cm_norm,
        display_labels=["Non-developing", "Developing"],
    ).plot(ax=ax, colorbar=False, values_format=".2f")

    ax.set_title(f"NEW dataset normalized | {model_name} | Lead {lead} h")
    plt.tight_layout()

    out_png_norm = CM_DIR / f"confusion_matrix_new_normalized_{safe_model}_{lead}h.png"

    plt.savefig(out_png_norm, dpi=220, bbox_inches="tight")
    plt.show()

cm_table = pd.DataFrame(cm_rows)
cm_table.to_csv(CM_DIR / "confusion_matrix_values_best_models_new.csv", index=False)

display(cm_table)

# ------------------------------------------------------------
# 6h 1x5 raw + normalized
# ------------------------------------------------------------

LEAD = 6

MODELS_6H = [
    "SVM",
    "AdaBoost",
    "Random Forest",
    "Logistic Regression",
    "MLP",
]

OUT_ONEFIG_DIR = CM_DIR / "combined_figures"
OUT_ONEFIG_DIR.mkdir(parents=True, exist_ok=True)

print("Available models in oof_all:")
print(sorted(oof_all["model"].unique()))

missing_models = [m for m in MODELS_6H if m not in oof_all["model"].unique()]

if missing_models:
    raise ValueError(
        f"Missing models in oof_all: {missing_models}\n"
        f"Available models: {sorted(oof_all['model'].unique())}"
    )

# Raw count
fig, axes = plt.subplots(1, 5, figsize=(22.5, 4.5))
axes = axes.flatten()

cm_rows_6h = []

for ax, model_name in zip(axes, MODELS_6H):
    sub = oof_all[
        (oof_all["lead_h"] == LEAD)
        & (oof_all["model"] == model_name)
    ].copy()

    y_true = sub["y_true"].astype(int).values
    y_pred = sub["y_pred_default"].astype(int).values

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    cm_rows_6h.append({
        "dataset": "new",
        "model": model_name,
        "lead_h": LEAD,
        "rows": len(sub),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    })

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Non-dev", "Dev"],
    ).plot(
        ax=ax,
        colorbar=False,
        values_format="d",
    )

    ax.set_title(model_name, fontsize=13)
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)

fig.suptitle("NEW dataset — 6h confusion matrices", fontsize=16, y=1.03)
plt.tight_layout()

out_png = OUT_ONEFIG_DIR / "new_6h_confusion_matrices_1x5_SVM_AdaBoost_RandomForest_Logistic_MLP.png"
plt.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_png)

cm_summary_new_6h = pd.DataFrame(cm_rows_6h)
cm_summary_new_6h.to_csv(
    OUT_ONEFIG_DIR / "new_6h_confusion_matrix_values_1x5.csv",
    index=False,
)

display(cm_summary_new_6h)

# Normalized
fig, axes = plt.subplots(1, 5, figsize=(22.5, 4.5))
axes = axes.flatten()

cm_rows_6h_norm = []

for ax, model_name in zip(axes, MODELS_6H):
    sub = oof_all[
        (oof_all["lead_h"] == LEAD)
        & (oof_all["model"] == model_name)
    ].copy()

    y_true = sub["y_true"].astype(int).values
    y_pred = sub["y_pred_default"].astype(int).values

    cm_count = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_norm = confusion_matrix(y_true, y_pred, labels=[0, 1], normalize="true")

    tn, fp, fn, tp = cm_count.ravel()

    cm_rows_6h_norm.append({
        "dataset": "new",
        "model": model_name,
        "lead_h": LEAD,
        "rows": len(sub),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "true_nondev_correct_rate": cm_norm[0, 0],
        "false_alarm_rate": cm_norm[0, 1],
        "miss_rate": cm_norm[1, 0],
        "true_dev_detection_rate": cm_norm[1, 1],
    })

    ConfusionMatrixDisplay(
        confusion_matrix=cm_norm,
        display_labels=["Non-dev", "Dev"],
    ).plot(
        ax=ax,
        colorbar=False,
        values_format=".2f",
    )

    ax.set_title(model_name, fontsize=13)
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)

fig.suptitle("NEW dataset — 6h normalized confusion matrices", fontsize=16, y=1.03)
plt.tight_layout()

out_png_norm = OUT_ONEFIG_DIR / "new_6h_confusion_matrices_1x5_normalized_SVM_AdaBoost_RandomForest_Logistic_MLP.png"
plt.savefig(out_png_norm, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_png_norm)

cm_summary_new_6h_norm = pd.DataFrame(cm_rows_6h_norm)
cm_summary_new_6h_norm.to_csv(
    OUT_ONEFIG_DIR / "new_6h_confusion_matrix_values_1x5_normalized.csv",
    index=False,
)

display(cm_summary_new_6h_norm)


TUNED THRESHOLD

In [ ]:
# ============================================================
# RELOAD ML OUTPUTS AND RECREATE best_by_lead
# Use this if best_by_lead is not defined
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Choose dataset
# ------------------------------------------------------------

DATASET_NAME = "new"   # change to "old" for OLD dataset
LABEL_NAME = "box2deg"

if DATASET_NAME == "new":
    OUT_DIR = Path(
        r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest\ml_new_box2deg_2009_2025_UPDATED"
    )
else:
    OUT_DIR = Path(
        r"C:\Users\_s2218026\Documents\lab\mcs_output\ml_old_box2deg_2009_2025_UPDATED"
    )

CM_DIR = OUT_DIR / "confusion_matrices"
CM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load OOF predictions
# ------------------------------------------------------------

oof_candidates = [
    OUT_DIR / f"oof_predictions_allmodels_{DATASET_NAME}.csv",
    OUT_DIR / "oof_predictions_all_models.csv",
    OUT_DIR / "oof_predictions_allmodels.csv",
]

oof_fp = None

for fp in oof_candidates:
    if fp.exists():
        oof_fp = fp
        break

if oof_fp is None:
    raise FileNotFoundError(
        "No OOF prediction file found. Checked:\n"
        + "\n".join(str(fp) for fp in oof_candidates)
    )

oof_all = pd.read_csv(oof_fp)

print("Loaded OOF predictions:")
print(oof_fp)
print("Rows:", len(oof_all))
print("Models:", sorted(oof_all["model"].unique()))

# ------------------------------------------------------------
# Load ranked model-performance table if it exists
# ------------------------------------------------------------

ranked_candidates = [
    OUT_DIR / f"model_performance_ranked_by_lead_{DATASET_NAME}.csv",
    OUT_DIR / "model_performance_ranked_by_lead.csv",
]

ranked_fp = None

for fp in ranked_candidates:
    if fp.exists():
        ranked_fp = fp
        break

if ranked_fp is not None:
    ranked_table = pd.read_csv(ranked_fp)

    print("\nLoaded ranked table:")
    print(ranked_fp)

else:
    # fallback: build ranked_table from oof_summary
    summary_candidates = [
        OUT_DIR / f"oof_summary_all_models_{DATASET_NAME}.csv",
        OUT_DIR / "oof_summary_all_models.csv",
    ]

    summary_fp = None

    for fp in summary_candidates:
        if fp.exists():
            summary_fp = fp
            break

    if summary_fp is None:
        raise FileNotFoundError(
            "No ranked table or oof summary found. Checked:\n"
            + "\n".join(str(fp) for fp in ranked_candidates + summary_candidates)
        )

    oof_summary = pd.read_csv(summary_fp)

    print("\nLoaded OOF summary:")
    print(summary_fp)

    # Use default 0.5 metrics as main table
    ranked_table = oof_summary[
        oof_summary["metric_type"] == "oof_default_0.5"
    ].copy()

    ranked_table = ranked_table[
        [
            "lead_h", "model",
            "precision", "recall", "f1",
            "roc_auc", "pr_auc"
        ]
    ].copy()

    ranked_table = ranked_table.sort_values(
        ["lead_h", "f1"],
        ascending=[True, False]
    ).reset_index(drop=True)

# ------------------------------------------------------------
# Recreate best_by_lead
# ------------------------------------------------------------

best_by_lead = (
    ranked_table
    .sort_values(["lead_h", "f1"], ascending=[True, False])
    .groupby("lead_h", as_index=False)
    .first()
)

print("\nRanked table:")
display(ranked_table)

print("\nBest model by lead:")
display(best_by_lead)

In [ ]:
# ============================================================
# CELL 8 — CONFUSION MATRICES WITH TUNED THRESHOLD
# 1. Best model per lead
# 2. 6h 1x5 raw and normalized: SVM, AdaBoost, RF, Logistic, MLP
#
# Uses tuned threshold selected from OOF y_score by max F1
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)

# ------------------------------------------------------------
# Dataset naming
# ------------------------------------------------------------

DATASET_DISPLAY = "NEW"     # change to "OLD" if running old dataset
DATASET_SLUG = "new"        # change to "old" if running old dataset

# ------------------------------------------------------------
# Helper: find best threshold by F1
# ------------------------------------------------------------

def get_best_threshold_from_oof(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

    rows = []

    for thr in thresholds:
        y_pred = (y_score >= thr).astype(int)

        rows.append({
            "threshold": thr,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
        })

    scan = pd.DataFrame(rows)

    scan = (
        scan
        .sort_values(["f1", "recall", "precision"], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    best_threshold = float(scan.loc[0, "threshold"])

    return best_threshold, scan


def get_tuned_prediction(sub):
    if "y_score" not in sub.columns:
        raise ValueError(
            "Cannot use threshold tuning because y_score column is missing."
        )

    y_true = sub["y_true"].astype(int).values
    y_score = sub["y_score"].astype(float).values

    best_threshold, scan = get_best_threshold_from_oof(y_true, y_score)

    y_pred = (y_score >= best_threshold).astype(int)

    return y_pred, best_threshold, scan


# ------------------------------------------------------------
# Output folders
# ------------------------------------------------------------

OUT_ONEFIG_DIR = CM_DIR / "combined_figures"
OUT_ONEFIG_DIR.mkdir(parents=True, exist_ok=True)

THR_DIR = CM_DIR / "threshold_scans"
THR_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. BEST MODEL PER LEAD — TUNED THRESHOLD
# ============================================================

cm_rows = []

for _, row in best_by_lead.iterrows():
    lead = int(row["lead_h"])
    model_name = row["model"]

    sub = oof_all[
        (oof_all["lead_h"] == lead)
        & (oof_all["model"] == model_name)
    ].copy()

    y_true = sub["y_true"].astype(int).values
    y_pred, best_threshold, scan = get_tuned_prediction(sub)

    safe_model = model_name.replace(" ", "_").replace("/", "_")

    scan_out = THR_DIR / f"threshold_scan_{DATASET_SLUG}_{safe_model}_{lead}h.csv"
    scan.to_csv(scan_out, index=False)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    cm_rows.append({
        "lead_h": lead,
        "model": model_name,
        "best_threshold": best_threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "total": int(cm.sum()),
    })

    # ------------------------------
    # Raw count confusion matrix
    # ------------------------------
    fig, ax = plt.subplots(figsize=(5.5, 4.5))

    ConfusionMatrixDisplay(
        cm,
        display_labels=["Non-developing", "Developing"],
    ).plot(ax=ax, colorbar=False, values_format="d")

    ax.set_title(
        f"{DATASET_DISPLAY} dataset | {model_name} | Lead {lead} h\n"
        f"Tuned threshold = {best_threshold:.2f}"
    )

    plt.tight_layout()

    out_png = CM_DIR / f"confusion_matrix_{DATASET_SLUG}_tuned_{safe_model}_{lead}h.png"
    plt.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.show()

    # ------------------------------
    # Normalized confusion matrix
    # ------------------------------
    cm_norm = confusion_matrix(y_true, y_pred, labels=[0, 1], normalize="true")

    fig, ax = plt.subplots(figsize=(5.5, 4.5))

    ConfusionMatrixDisplay(
        cm_norm,
        display_labels=["Non-developing", "Developing"],
    ).plot(ax=ax, colorbar=False, values_format=".2f")

    ax.set_title(
        f"{DATASET_DISPLAY} dataset normalized | {model_name} | Lead {lead} h\n"
        f"Tuned threshold = {best_threshold:.2f}"
    )

    plt.tight_layout()

    out_png_norm = CM_DIR / f"confusion_matrix_{DATASET_SLUG}_tuned_normalized_{safe_model}_{lead}h.png"
    plt.savefig(out_png_norm, dpi=220, bbox_inches="tight")
    plt.show()

cm_table = pd.DataFrame(cm_rows)

cm_table.to_csv(
    CM_DIR / f"confusion_matrix_values_best_models_{DATASET_SLUG}_tuned.csv",
    index=False
)

display(cm_table)


# ============================================================
# 2. 6H 1x5 RAW + NORMALIZED — TUNED THRESHOLD
# ============================================================

LEAD = 6

MODELS_6H = [
    "SVM",
    "AdaBoost",
    "Random Forest",
    "Logistic Regression",
    "MLP",
    "Naive Bayes",
]

print("Available models in oof_all:")
print(sorted(oof_all["model"].unique()))

missing_models = [m for m in MODELS_6H if m not in oof_all["model"].unique()]

if missing_models:
    raise ValueError(
        f"Missing models in oof_all: {missing_models}\n"
        f"Available models: {sorted(oof_all['model'].unique())}"
    )

# ------------------------------------------------------------
# Raw count 1x5
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 5, figsize=(22.5, 4.5))
axes = axes.flatten()

cm_rows_6h = []

for ax, model_name in zip(axes, MODELS_6H):
    sub = oof_all[
        (oof_all["lead_h"] == LEAD)
        & (oof_all["model"] == model_name)
    ].copy()

    y_true = sub["y_true"].astype(int).values
    y_pred, best_threshold, scan = get_tuned_prediction(sub)

    safe_model = model_name.replace(" ", "_").replace("/", "_")

    scan_out = THR_DIR / f"threshold_scan_{DATASET_SLUG}_{safe_model}_{LEAD}h.csv"
    scan.to_csv(scan_out, index=False)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    cm_rows_6h.append({
        "dataset": DATASET_SLUG,
        "model": model_name,
        "lead_h": LEAD,
        "best_threshold": best_threshold,
        "rows": len(sub),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    })

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Non-dev", "Dev"],
    ).plot(
        ax=ax,
        colorbar=False,
        values_format="d",
    )

    ax.set_title(
        f"{model_name}\nthreshold = {best_threshold:.2f}",
        fontsize=13
    )
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)

fig.suptitle(
    f"{DATASET_DISPLAY} dataset — 6h confusion matrices using tuned threshold",
    fontsize=16,
    y=1.03
)

plt.tight_layout()

out_png = OUT_ONEFIG_DIR / f"{DATASET_SLUG}_6h_confusion_matrices_1x5_tuned_SVM_AdaBoost_RandomForest_Logistic_MLP.png"
plt.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_png)

cm_summary_6h = pd.DataFrame(cm_rows_6h)

cm_summary_6h.to_csv(
    OUT_ONEFIG_DIR / f"{DATASET_SLUG}_6h_confusion_matrix_values_1x5_tuned.csv",
    index=False,
)

display(cm_summary_6h)


# ------------------------------------------------------------
# Normalized 1x5
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 5, figsize=(22.5, 4.5))
axes = axes.flatten()

cm_rows_6h_norm = []

for ax, model_name in zip(axes, MODELS_6H):
    sub = oof_all[
        (oof_all["lead_h"] == LEAD)
        & (oof_all["model"] == model_name)
    ].copy()

    y_true = sub["y_true"].astype(int).values
    y_pred, best_threshold, scan = get_tuned_prediction(sub)

    cm_count = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_norm = confusion_matrix(y_true, y_pred, labels=[0, 1], normalize="true")

    tn, fp, fn, tp = cm_count.ravel()

    cm_rows_6h_norm.append({
        "dataset": DATASET_SLUG,
        "model": model_name,
        "lead_h": LEAD,
        "best_threshold": best_threshold,
        "rows": len(sub),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "true_nondev_correct_rate": cm_norm[0, 0],
        "false_alarm_rate": cm_norm[0, 1],
        "miss_rate": cm_norm[1, 0],
        "true_dev_detection_rate": cm_norm[1, 1],
    })

    ConfusionMatrixDisplay(
        confusion_matrix=cm_norm,
        display_labels=["Non-dev", "Dev"],
    ).plot(
        ax=ax,
        colorbar=False,
        values_format=".2f",
    )

    ax.set_title(
        f"{model_name}\nthreshold = {best_threshold:.2f}",
        fontsize=13
    )
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)

fig.suptitle(
    f"{DATASET_DISPLAY} dataset — 6h normalized confusion matrices using tuned threshold",
    fontsize=16,
    y=1.03
)

plt.tight_layout()

out_png_norm = OUT_ONEFIG_DIR / f"{DATASET_SLUG}_6h_confusion_matrices_1x5_normalized_tuned_SVM_AdaBoost_RandomForest_Logistic_MLP.png"
plt.savefig(out_png_norm, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_png_norm)

cm_summary_6h_norm = pd.DataFrame(cm_rows_6h_norm)

cm_summary_6h_norm.to_csv(
    OUT_ONEFIG_DIR / f"{DATASET_SLUG}_6h_confusion_matrix_values_1x5_normalized_tuned.csv",
    index=False,
)

display(cm_summary_6h_norm)

NAIVEBAYES (NO SVM AND MLP)

In [ ]:
# ============================================================
# NEW — 6H CONFUSION MATRICES, 5 MODELS
# Models: AdaBoost, Random Forest, Logistic Regression, Naive Bayes, MLP
#
# Produces separate figures:
#   1. Default threshold raw-count confusion matrices
#   2. Default threshold normalized confusion matrices
#   3. Tuned threshold raw-count confusion matrices
#   4. Tuned threshold normalized confusion matrices
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)

# ============================================================
# NEW PATHS
# ============================================================

DATASET_DISPLAY = "NEW"
DATASET_SLUG = "new"
LEAD = 6

OUT_DIR = Path(
    r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest\ml_new_box2deg_2009_2025_UPDATED"
)

CM_DIR = OUT_DIR / "confusion_matrices"
CM_DIR.mkdir(parents=True, exist_ok=True)

OUT_ONEFIG_DIR = CM_DIR / "combined_figures"
OUT_ONEFIG_DIR.mkdir(parents=True, exist_ok=True)

THR_DIR = CM_DIR / "threshold_scans"
THR_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# LOAD NEW OOF PREDICTIONS
# ============================================================

oof_candidates = [
    OUT_DIR / "oof_predictions_allmodels_new.csv",
    OUT_DIR / "oof_predictions_all_models.csv",
    OUT_DIR / "oof_predictions_allmodels.csv",
]

oof_fp = None

for fp in oof_candidates:
    print("checking:", fp, "| exists:", fp.exists())
    if fp.exists():
        oof_fp = fp
        break

if oof_fp is None:
    raise FileNotFoundError(
        "No NEW OOF prediction file found. Checked:\n"
        + "\n".join(str(fp) for fp in oof_candidates)
    )

oof_all = pd.read_csv(oof_fp)

print("\nLoaded NEW OOF predictions:")
print(oof_fp)
print("Rows:", len(oof_all))
print("Models:", sorted(oof_all["model"].dropna().unique()))

if "dataset" in oof_all.columns:
    print("Dataset values inside oof_all:", oof_all["dataset"].unique())

# ============================================================
# MODEL SELECTION
# ============================================================

MODEL_CANDIDATES = {
    "AdaBoost": ["AdaBoost"],
    "Random Forest": ["Random Forest", "RandomForest"],
    "Logistic Regression": ["Logistic Regression", "Logistic"],
    "Naive Bayes": ["Naive Bayes", "NaiveBayes", "GaussianNB", "NB"],
    "MLP": ["MLP"],
}

available_models = sorted(oof_all["model"].dropna().unique())

MODELS_6H = []

for display_name, candidates in MODEL_CANDIDATES.items():
    found_model = None

    for c in candidates:
        if c in available_models:
            found_model = c
            break

    if found_model is None:
        raise ValueError(
            f"Could not find model for {display_name}.\n"
            f"Tried: {candidates}\n"
            f"Available models: {available_models}"
        )

    MODELS_6H.append(found_model)

print("\nModels used:")
print(MODELS_6H)

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_default_prediction(sub):
    """
    Default threshold prediction.
    Uses y_pred_default if available; otherwise uses y_score >= 0.5.
    """
    if "y_pred_default" in sub.columns:
        return sub["y_pred_default"].astype(int).values

    if "y_pred" in sub.columns:
        return sub["y_pred"].astype(int).values

    if "y_score" in sub.columns:
        return (sub["y_score"].astype(float).values >= 0.5).astype(int)

    raise ValueError("No prediction column found. Need y_pred_default, y_pred, or y_score.")


def get_best_threshold_from_oof(y_true, y_score):
    """
    Select threshold that maximizes F1 using OOF scores.
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

    rows = []

    for thr in thresholds:
        y_pred = (y_score >= thr).astype(int)

        rows.append({
            "threshold": thr,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
        })

    scan = pd.DataFrame(rows)

    scan = (
        scan
        .sort_values(["f1", "recall", "precision"], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    best_threshold = float(scan.loc[0, "threshold"])

    return best_threshold, scan


def get_tuned_prediction(sub):
    """
    Tuned threshold prediction based on max-F1 threshold from y_score.
    """
    if "y_score" not in sub.columns:
        raise ValueError("Cannot tune threshold because y_score column is missing.")

    y_true = sub["y_true"].astype(int).values
    y_score = sub["y_score"].astype(float).values

    best_threshold, scan = get_best_threshold_from_oof(y_true, y_score)
    y_pred = (y_score >= best_threshold).astype(int)

    return y_pred, best_threshold, scan


def make_confusion_figure(
    oof_all,
    models,
    lead,
    dataset_display,
    dataset_slug,
    mode="default",
    normalized=False,
):
    """
    mode:
      default = default threshold 0.5 / y_pred_default
      tuned   = F1-optimized threshold from y_score

    normalized:
      False = raw count confusion matrix
      True  = row-normalized confusion matrix
    """

    fig, axes = plt.subplots(1, 5, figsize=(22.5, 4.5))
    axes = axes.flatten()

    rows = []

    for ax, model_name in zip(axes, models):
        sub = oof_all[
            (oof_all["lead_h"] == lead)
            & (oof_all["model"] == model_name)
        ].copy()

        if len(sub) == 0:
            ax.set_axis_off()
            ax.set_title(f"{model_name}\nNo data")
            continue

        y_true = sub["y_true"].astype(int).values

        if mode == "default":
            y_pred = get_default_prediction(sub)
            threshold_used = 0.5

        elif mode == "tuned":
            y_pred, threshold_used, scan = get_tuned_prediction(sub)

            safe_model = model_name.replace(" ", "_").replace("/", "_")
            scan_out = THR_DIR / f"threshold_scan_{dataset_slug}_{safe_model}_{lead}h.csv"
            scan.to_csv(scan_out, index=False)

        else:
            raise ValueError("mode must be 'default' or 'tuned'")

        cm_count = confusion_matrix(y_true, y_pred, labels=[0, 1])

        if normalized:
            cm_plot = confusion_matrix(y_true, y_pred, labels=[0, 1], normalize="true")
            values_format = ".2f"
        else:
            cm_plot = cm_count
            values_format = "d"

        tn, fp, fn, tp = cm_count.ravel()

        row = {
            "dataset": dataset_slug,
            "mode": mode,
            "normalized": normalized,
            "model": model_name,
            "lead_h": lead,
            "threshold": threshold_used,
            "rows": len(sub),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }

        if normalized:
            cm_norm = confusion_matrix(y_true, y_pred, labels=[0, 1], normalize="true")
            row.update({
                "true_nondev_correct_rate": float(cm_norm[0, 0]),
                "false_alarm_rate": float(cm_norm[0, 1]),
                "miss_rate": float(cm_norm[1, 0]),
                "true_dev_detection_rate": float(cm_norm[1, 1]),
            })

        rows.append(row)

        ConfusionMatrixDisplay(
            confusion_matrix=cm_plot,
            display_labels=["Non-dev", "Dev"],
        ).plot(
            ax=ax,
            colorbar=False,
            values_format=values_format,
        )

        if mode == "tuned":
            ax.set_title(f"{model_name}\nthr={threshold_used:.2f}", fontsize=13)
        else:
            ax.set_title(f"{model_name}\nthr=0.50", fontsize=13)

        ax.set_xlabel("Predicted label", fontsize=11)
        ax.set_ylabel("True label", fontsize=11)

    mode_title = "default threshold" if mode == "default" else "tuned threshold"
    norm_title = "normalized" if normalized else "raw-count"

    fig.suptitle(
        f"{dataset_display} dataset — 6h {norm_title} confusion matrices ({mode_title})",
        fontsize=16,
        y=1.03,
    )

    plt.tight_layout()

    suffix_norm = "normalized" if normalized else "raw"

    out_png = (
        OUT_ONEFIG_DIR
        / f"{dataset_slug}_6h_confusion_matrices_1x5_{suffix_norm}_{mode}_AdaBoost_RF_Logistic_NaiveBayes_MLP.png"
    )

    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)

    table = pd.DataFrame(rows)

    out_csv = (
        OUT_ONEFIG_DIR
        / f"{dataset_slug}_6h_confusion_matrix_values_1x5_{suffix_norm}_{mode}_AdaBoost_RF_Logistic_NaiveBayes_MLP.csv"
    )

    table.to_csv(out_csv, index=False)

    print("Saved:", out_csv)

    return table

# ============================================================
# PRODUCE 4 FIGURES
# ============================================================

cm_default_raw = make_confusion_figure(
    oof_all=oof_all,
    models=MODELS_6H,
    lead=LEAD,
    dataset_display=DATASET_DISPLAY,
    dataset_slug=DATASET_SLUG,
    mode="default",
    normalized=False,
)

display(cm_default_raw)

cm_default_norm = make_confusion_figure(
    oof_all=oof_all,
    models=MODELS_6H,
    lead=LEAD,
    dataset_display=DATASET_DISPLAY,
    dataset_slug=DATASET_SLUG,
    mode="default",
    normalized=True,
)

display(cm_default_norm)

cm_tuned_raw = make_confusion_figure(
    oof_all=oof_all,
    models=MODELS_6H,
    lead=LEAD,
    dataset_display=DATASET_DISPLAY,
    dataset_slug=DATASET_SLUG,
    mode="tuned",
    normalized=False,
)

display(cm_tuned_raw)

cm_tuned_norm = make_confusion_figure(
    oof_all=oof_all,
    models=MODELS_6H,
    lead=LEAD,
    dataset_display=DATASET_DISPLAY,
    dataset_slug=DATASET_SLUG,
    mode="tuned",
    normalized=True,
)

display(cm_tuned_norm)

In [ ]:
# ============================================================
# CELL 9 — FEATURE IMPORTANCE
# Tree models: Random Forest, AdaBoost, Decision Tree
# ============================================================

tree_models = {
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=600,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        )),
    ]),

    "AdaBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", AdaBoostClassifier(
            n_estimators=600,
            random_state=RANDOM_STATE,
        )),
    ]),

    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", DecisionTreeClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced",
        )),
    ]),
}

fi_rows = []

for lead in LEADS:
    df = datasets[lead].copy()

    X = df[FEATURE_COLS].copy()
    y = df["y"].astype(int).values

    for model_name, model in tree_models.items():
        print(f"Feature importance: {model_name}, lead {lead}h")

        model.fit(X, y)
        clf = model.named_steps["clf"]

        if not hasattr(clf, "feature_importances_"):
            print(f"[SKIP] {model_name} has no feature_importances_")
            continue

        imp = pd.DataFrame({
            "lead_h": lead,
            "model": model_name,
            "feature": FEATURE_COLS,
            "importance": clf.feature_importances_,
        })

        fi_rows.append(imp)

        imp_plot = imp.sort_values("importance", ascending=True)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh(imp_plot["feature"], imp_plot["importance"])
        ax.set_xlabel("Feature importance")
        ax.set_title(f"NEW dataset: {model_name} feature importance, lead {lead} h")
        ax.grid(True, axis="x", alpha=0.3)

        plt.tight_layout()

        safe_model = model_name.replace(" ", "_").replace("/", "_")
        out_png = FI_DIR / f"feature_importance_new_{safe_model}_{lead}h.png"

        plt.savefig(out_png, dpi=220, bbox_inches="tight")
        plt.show()

        print("Saved:", out_png)

if fi_rows:
    fi_all = pd.concat(fi_rows, ignore_index=True)
    fi_all.to_csv(FI_DIR / "feature_importance_tree_models_new.csv", index=False)

    display(
        fi_all.sort_values(
            ["lead_h", "model", "importance"],
            ascending=[True, True, False],
        )
    )

print("\nDONE — NEW ML PIPELINE COMPLETE")
print("OUT_DIR:", OUT_DIR)
print("PLOT_DIR:", PLOT_DIR)
print("CM_DIR:", CM_DIR)
print("FI_DIR:", FI_DIR)


SHAP

In [ ]:
# ============================================================
# SHAP PLOTS FOR ML RESULTS
# Compatible with ML_CHECK_LATEST structure
#
# Produces:
#   1. SHAP bar plot
#   2. SHAP beeswarm plot
#   3. SHAP dependence plots for top features
#   4. SHAP heatmap across lead times
#   5. SHAP importance CSV tables
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone

# ------------------------------------------------------------
# Import SHAP
# ------------------------------------------------------------
try:
    import shap
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])
    import shap

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------
SHAP_DIR = OUT_DIR / "shap_plots"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SHAP_MODEL_NAMES = [
    "AdaBoost",
    "Random Forest",
    "SVM",
    "Logistic Regression",
    "MLP",
]



LEADS_TO_EXPLAIN = LEADS

# Keep these moderate first. Increase later if needed.
SHAP_BACKGROUND_N = 80
SHAP_POS_N = 250
SHAP_NEG_N = 250

N_TOP_DEPENDENCE = 5
SHAP_RANDOM_STATE = RANDOM_STATE

PREDICTOR_ORDER = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
    "month",
    "lat",
    "lon",
]

LEAD_ORDER = [6, 12, 24, 48]

# ------------------------------------------------------------
# Load case tables if datasets is not already available
# ------------------------------------------------------------
try:
    datasets
    print("Using existing datasets dictionary.")
except NameError:
    print("datasets not found. Loading case tables from CASE_DIR...")

    datasets = {}

    for L in LEADS_TO_EXPLAIN:
        fp = CASE_DIR / f"{LABEL_NAME}_case_table_{DATASET_NAME}_{L}h.csv"

        if not fp.exists():
            raise FileNotFoundError(fp)

        df = pd.read_csv(fp)

        if "time" in df.columns:
            df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

        df["y"] = df["y"].astype(int)
        datasets[L] = df.copy()

        print(
            f"Lead {L}h | rows={len(df)} | "
            f"positive={int(df['y'].sum())} | "
            f"negative={int((df['y'] == 0).sum())}"
        )

# ------------------------------------------------------------
# Helper: balanced SHAP sample
# ------------------------------------------------------------
def make_shap_samples(X, y, background_n=80, pos_n=250, neg_n=250, random_state=42):
    rng = np.random.default_rng(random_state)

    y = np.asarray(y).astype(int)

    all_idx = np.arange(len(y))
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    bg_n = min(background_n, len(all_idx))
    bg_idx = rng.choice(all_idx, size=bg_n, replace=False)

    pos_take = min(pos_n, len(pos_idx))
    neg_take = min(neg_n, len(neg_idx))

    exp_parts = []

    if pos_take > 0:
        exp_parts.append(rng.choice(pos_idx, size=pos_take, replace=False))

    if neg_take > 0:
        exp_parts.append(rng.choice(neg_idx, size=neg_take, replace=False))

    exp_idx = np.concatenate(exp_parts)
    rng.shuffle(exp_idx)

    X_bg = X.iloc[bg_idx].copy()
    X_exp = X.iloc[exp_idx].copy()
    y_exp = y[exp_idx]

    return X_bg, X_exp, y_exp, exp_idx

# ------------------------------------------------------------
# Helper: prediction function for positive class probability
# ------------------------------------------------------------
def make_predict_positive_func(pipe, feature_cols):
    def predict_positive(z):
        if isinstance(z, pd.DataFrame):
            Xz = z[feature_cols].copy()
        else:
            Xz = pd.DataFrame(z, columns=feature_cols)

        return pipe.predict_proba(Xz)[:, 1]

    return predict_positive

# ------------------------------------------------------------
# Helper: dependence plots
# ------------------------------------------------------------
def save_dependence_plots(shap_values, X_exp, top_features, out_dir, prefix):
    shap_arr = shap_values.values

    for feat in top_features:
        if feat not in X_exp.columns:
            continue

        j = list(X_exp.columns).index(feat)

        fig, ax = plt.subplots(figsize=(6.8, 4.8))

        ax.scatter(
            X_exp[feat].values,
            shap_arr[:, j],
            s=16,
            alpha=0.55
        )

        ax.axhline(0, linewidth=1)
        ax.set_xlabel(feat)
        ax.set_ylabel("SHAP value")
        ax.set_title(f"{prefix}\nSHAP dependence: {feat}")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        safe_feat = feat.replace("/", "_").replace(" ", "_")
        safe_prefix = (
            prefix.replace(" ", "_")
            .replace("|", "")
            .replace("/", "_")
        )

        out_png = out_dir / f"{safe_prefix}_dependence_{safe_feat}.png"

        fig.savefig(out_png, dpi=220, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        print("Saved:", out_png)

# ------------------------------------------------------------
# Run SHAP
# ------------------------------------------------------------
models_all = get_models(random_state=RANDOM_STATE)

missing_models = [m for m in SHAP_MODEL_NAMES if m not in models_all]
if missing_models:
    raise ValueError(
        f"These models are not available in get_models(): {missing_models}\n"
        f"Available models: {list(models_all.keys())}"
    )

all_shap_importance = []
all_shap_samples = []

for model_name in SHAP_MODEL_NAMES:
    print("\n" + "=" * 90)
    print(f"SHAP ANALYSIS | MODEL: {model_name}")
    print("=" * 90)

    safe_model = model_name.replace(" ", "_").replace("/", "_")
    model_dir = SHAP_DIR / safe_model
    model_dir.mkdir(parents=True, exist_ok=True)

    for L in LEADS_TO_EXPLAIN:
        print("\n" + "-" * 90)
        print(f"Lead {L}h")
        print("-" * 90)

        case_df = datasets[L].copy()

        use_cols = FEATURE_COLS + ["y"]
        case_df = case_df.dropna(subset=use_cols).reset_index(drop=True)

        X = case_df[FEATURE_COLS].copy()
        y = case_df["y"].astype(int).values

        print(
            f"Rows used: {len(case_df)} | "
            f"positive={int(y.sum())} | "
            f"negative={int((y == 0).sum())}"
        )

        # ------------------------------------------------------------
        # Fit final model on all rows for this lead
        # ------------------------------------------------------------
        base_model = clone(models_all[model_name])
        pipe = make_pipeline(model_name, base_model)
        pipe.fit(X, y)

        # ------------------------------------------------------------
        # Balanced SHAP explanation sample
        # ------------------------------------------------------------
        X_bg, X_exp, y_exp, exp_idx = make_shap_samples(
            X=X,
            y=y,
            background_n=SHAP_BACKGROUND_N,
            pos_n=SHAP_POS_N,
            neg_n=SHAP_NEG_N,
            random_state=SHAP_RANDOM_STATE + int(L)
        )

        print(
            f"Background sample: {len(X_bg)} | "
            f"Explanation sample: {len(X_exp)} | "
            f"Explanation positives={int(y_exp.sum())}"
        )

        predict_positive = make_predict_positive_func(pipe, FEATURE_COLS)

        masker = shap.maskers.Independent(X_bg)

        explainer = shap.Explainer(
            predict_positive,
            masker,
            algorithm="permutation",
            feature_names=FEATURE_COLS
        )

        shap_values = explainer(
            X_exp,
            max_evals=2 * len(FEATURE_COLS) + 1,
            batch_size=50
        )

        shap_arr = shap_values.values
        y_prob_exp = pipe.predict_proba(X_exp)[:, 1]

        # ------------------------------------------------------------
        # Save explained sample
        # ------------------------------------------------------------
        sample_out = X_exp.copy()
        sample_out["y_true"] = y_exp
        sample_out["y_prob"] = y_prob_exp
        sample_out["dataset"] = DATASET_NAME
        sample_out["label"] = LABEL_NAME
        sample_out["lead_h"] = L
        sample_out["model"] = model_name

        sample_csv = model_dir / f"shap_explanation_sample_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.csv"
        sample_out.to_csv(sample_csv, index=False)

        all_shap_samples.append(sample_out)

        # ------------------------------------------------------------
        # SHAP global importance table
        # ------------------------------------------------------------
        imp = pd.DataFrame({
            "feature": FEATURE_COLS,
            "mean_abs_shap": np.mean(np.abs(shap_arr), axis=0),
            "mean_shap": np.mean(shap_arr, axis=0),
            "std_shap": np.std(shap_arr, axis=0),
        })

        imp["rank"] = imp["mean_abs_shap"].rank(ascending=False, method="dense").astype(int)
        imp = imp.sort_values("mean_abs_shap", ascending=False)

        imp["dataset"] = DATASET_NAME
        imp["label"] = LABEL_NAME
        imp["lead_h"] = L
        imp["model"] = model_name
        imp["n_explain"] = len(X_exp)
        imp["n_background"] = len(X_bg)
        imp["n_positive_explain"] = int(y_exp.sum())
        imp["n_negative_explain"] = int((y_exp == 0).sum())

        all_shap_importance.append(imp)

        imp_csv = model_dir / f"shap_importance_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.csv"
        imp.to_csv(imp_csv, index=False)

        print("Top SHAP features:")
        display(imp[["lead_h", "model", "feature", "mean_abs_shap", "mean_shap", "rank"]].head(10))

        # ------------------------------------------------------------
        # Save SHAP values
        # ------------------------------------------------------------
        shap_value_df = pd.DataFrame(
            shap_arr,
            columns=[f"shap_{c}" for c in FEATURE_COLS]
        )

        shap_value_df["y_true"] = y_exp
        shap_value_df["y_prob"] = y_prob_exp
        shap_value_df["dataset"] = DATASET_NAME
        shap_value_df["label"] = LABEL_NAME
        shap_value_df["lead_h"] = L
        shap_value_df["model"] = model_name

        shap_values_csv = model_dir / f"shap_values_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.csv"
        shap_value_df.to_csv(shap_values_csv, index=False)

        # ------------------------------------------------------------
        # SHAP bar plot
        # ------------------------------------------------------------
        title_prefix = f"{DATASET_NAME.upper()} {LABEL_NAME} | {model_name} | Lead {L}h"

        bar_png = model_dir / f"shap_bar_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.png"

        plt.figure(figsize=(8, 6))
        shap.plots.bar(
            shap_values,
            max_display=len(FEATURE_COLS),
            show=False
        )
        fig = plt.gcf()
        fig.suptitle(
            f"{title_prefix}\nSHAP global importance: mean absolute SHAP value",
            fontsize=13,
            y=1.02
        )
        plt.tight_layout()
        fig.savefig(bar_png, dpi=220, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        print("Saved:", bar_png)

        # ------------------------------------------------------------
        # SHAP beeswarm plot
        # ------------------------------------------------------------
        beeswarm_png = model_dir / f"shap_beeswarm_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.png"

        plt.figure(figsize=(8.5, 6.5))
        shap.plots.beeswarm(
            shap_values,
            max_display=len(FEATURE_COLS),
            show=False
        )
        fig = plt.gcf()
        fig.suptitle(
            f"{title_prefix}\nSHAP beeswarm: feature effects on developing-class probability",
            fontsize=13,
            y=1.02
        )
        plt.tight_layout()
        fig.savefig(beeswarm_png, dpi=220, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        print("Saved:", beeswarm_png)

        # ------------------------------------------------------------
        # SHAP dependence plots for top predictors
        # ------------------------------------------------------------
        top_features = imp["feature"].head(N_TOP_DEPENDENCE).tolist()

        save_dependence_plots(
            shap_values=shap_values,
            X_exp=X_exp,
            top_features=top_features,
            out_dir=model_dir,
            prefix=title_prefix
        )

# ------------------------------------------------------------
# Save combined SHAP outputs
# ------------------------------------------------------------
shap_importance_all = pd.concat(all_shap_importance, ignore_index=True)
shap_importance_all.to_csv(
    SHAP_DIR / "shap_importance_all_models_all_leads.csv",
    index=False
)

shap_samples_all = pd.concat(all_shap_samples, ignore_index=True)
shap_samples_all.to_csv(
    SHAP_DIR / "shap_explanation_samples_all_models_all_leads.csv",
    index=False
)

print("\nSaved combined SHAP importance:")
print(SHAP_DIR / "shap_importance_all_models_all_leads.csv")

# ------------------------------------------------------------
# SHAP HEATMAP ACROSS LEAD TIMES
# rows = predictors
# cols = lead times
# values = mean absolute SHAP value
# ------------------------------------------------------------
for model_name in SHAP_MODEL_NAMES:
    safe_model = model_name.replace(" ", "_").replace("/", "_")

    sub = shap_importance_all[
        shap_importance_all["model"] == model_name
    ].copy()

    heat = sub.pivot_table(
        index="feature",
        columns="lead_h",
        values="mean_abs_shap",
        aggfunc="mean"
    )

    heat = heat.reindex(index=PREDICTOR_ORDER, columns=LEAD_ORDER)

    vals = heat.values.astype(float)

    fig, ax = plt.subplots(figsize=(8, 7))

    im = ax.imshow(vals, aspect="auto", cmap="viridis")

    ax.set_xticks(np.arange(len(LEAD_ORDER)))
    ax.set_xticklabels([f"{L} h" for L in LEAD_ORDER], fontsize=11)

    ax.set_yticks(np.arange(len(PREDICTOR_ORDER)))
    ax.set_yticklabels(PREDICTOR_ORDER, fontsize=10)

    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8)
            else:
                ax.text(j, i, "NA", ha="center", va="center", fontsize=8)

    ax.set_xlabel("Lead time", fontsize=12)
    ax.set_ylabel("Predictor", fontsize=12)
    ax.set_title(
        f"{DATASET_NAME.upper()} {LABEL_NAME} | {model_name}\n"
        "SHAP heatmap: mean absolute SHAP value",
        fontsize=14
    )

    ax.set_xticks(np.arange(-0.5, len(LEAD_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(PREDICTOR_ORDER), 1), minor=True)
    ax.grid(which="minor", color="gray", linestyle="-", linewidth=0.5, alpha=0.4)
    ax.tick_params(which="minor", bottom=False, left=False)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("mean(|SHAP value|)", fontsize=11)

    plt.tight_layout()

    heat_png = SHAP_DIR / f"shap_heatmap_mean_abs_{DATASET_NAME}_{LABEL_NAME}_{safe_model}.png"
    heat_csv = SHAP_DIR / f"shap_heatmap_mean_abs_{DATASET_NAME}_{LABEL_NAME}_{safe_model}.csv"

    fig.savefig(heat_png, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    heat.to_csv(heat_csv)

    print("Saved:", heat_png)
    print("Saved:", heat_csv)

print("\n[DONE] SHAP analysis complete.")
print("Main output folder:")
print(SHAP_DIR)

SHAP Naive Bayes

In [ ]:
# ============================================================
# SHAP PLOTS FOR ML RESULTS
# Compatible with ML_CHECK_LATEST structure
#
# Produces:
#   1. SHAP bar plot
#   2. SHAP beeswarm plot
#   3. SHAP dependence plots for top features
#   4. SHAP heatmap across lead times
#   5. SHAP importance CSV tables
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone

# ------------------------------------------------------------
# Import SHAP
# ------------------------------------------------------------
try:
    import shap
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])
    import shap

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------
SHAP_DIR = OUT_DIR / "shap_plots"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SHAP_MODEL_NAMES = [
    "Naive Bayes",
]


LEADS_TO_EXPLAIN = LEADS

# Keep these moderate first. Increase later if needed.
SHAP_BACKGROUND_N = 80
SHAP_POS_N = 250
SHAP_NEG_N = 250

N_TOP_DEPENDENCE = 5
SHAP_RANDOM_STATE = RANDOM_STATE

PREDICTOR_ORDER = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
    "month",
    "lat",
    "lon",
]

LEAD_ORDER = [6, 12, 24, 48]

# ------------------------------------------------------------
# Load case tables if datasets is not already available
# ------------------------------------------------------------
try:
    datasets
    print("Using existing datasets dictionary.")
except NameError:
    print("datasets not found. Loading case tables from CASE_DIR...")

    datasets = {}

    for L in LEADS_TO_EXPLAIN:
        fp = CASE_DIR / f"{LABEL_NAME}_case_table_{DATASET_NAME}_{L}h.csv"

        if not fp.exists():
            raise FileNotFoundError(fp)

        df = pd.read_csv(fp)

        if "time" in df.columns:
            df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

        df["y"] = df["y"].astype(int)
        datasets[L] = df.copy()

        print(
            f"Lead {L}h | rows={len(df)} | "
            f"positive={int(df['y'].sum())} | "
            f"negative={int((df['y'] == 0).sum())}"
        )

# ------------------------------------------------------------
# Helper: balanced SHAP sample
# ------------------------------------------------------------
def make_shap_samples(X, y, background_n=80, pos_n=250, neg_n=250, random_state=42):
    rng = np.random.default_rng(random_state)

    y = np.asarray(y).astype(int)

    all_idx = np.arange(len(y))
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    bg_n = min(background_n, len(all_idx))
    bg_idx = rng.choice(all_idx, size=bg_n, replace=False)

    pos_take = min(pos_n, len(pos_idx))
    neg_take = min(neg_n, len(neg_idx))

    exp_parts = []

    if pos_take > 0:
        exp_parts.append(rng.choice(pos_idx, size=pos_take, replace=False))

    if neg_take > 0:
        exp_parts.append(rng.choice(neg_idx, size=neg_take, replace=False))

    exp_idx = np.concatenate(exp_parts)
    rng.shuffle(exp_idx)

    X_bg = X.iloc[bg_idx].copy()
    X_exp = X.iloc[exp_idx].copy()
    y_exp = y[exp_idx]

    return X_bg, X_exp, y_exp, exp_idx

# ------------------------------------------------------------
# Helper: prediction function for positive class probability
# ------------------------------------------------------------
def make_predict_positive_func(pipe, feature_cols):
    def predict_positive(z):
        if isinstance(z, pd.DataFrame):
            Xz = z[feature_cols].copy()
        else:
            Xz = pd.DataFrame(z, columns=feature_cols)

        return pipe.predict_proba(Xz)[:, 1]

    return predict_positive

# ------------------------------------------------------------
# Helper: dependence plots
# ------------------------------------------------------------
def save_dependence_plots(shap_values, X_exp, top_features, out_dir, prefix):
    shap_arr = shap_values.values

    for feat in top_features:
        if feat not in X_exp.columns:
            continue

        j = list(X_exp.columns).index(feat)

        fig, ax = plt.subplots(figsize=(6.8, 4.8))

        ax.scatter(
            X_exp[feat].values,
            shap_arr[:, j],
            s=16,
            alpha=0.55
        )

        ax.axhline(0, linewidth=1)
        ax.set_xlabel(feat)
        ax.set_ylabel("SHAP value")
        ax.set_title(f"{prefix}\nSHAP dependence: {feat}")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        safe_feat = feat.replace("/", "_").replace(" ", "_")
        safe_prefix = (
            prefix.replace(" ", "_")
            .replace("|", "")
            .replace("/", "_")
        )

        out_png = out_dir / f"{safe_prefix}_dependence_{safe_feat}.png"

        fig.savefig(out_png, dpi=220, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        print("Saved:", out_png)

# ------------------------------------------------------------
# Run SHAP
# ------------------------------------------------------------
models_all = get_models(random_state=RANDOM_STATE)

missing_models = [m for m in SHAP_MODEL_NAMES if m not in models_all]
if missing_models:
    raise ValueError(
        f"These models are not available in get_models(): {missing_models}\n"
        f"Available models: {list(models_all.keys())}"
    )

all_shap_importance = []
all_shap_samples = []

for model_name in SHAP_MODEL_NAMES:
    print("\n" + "=" * 90)
    print(f"SHAP ANALYSIS | MODEL: {model_name}")
    print("=" * 90)

    safe_model = model_name.replace(" ", "_").replace("/", "_")
    model_dir = SHAP_DIR / safe_model
    model_dir.mkdir(parents=True, exist_ok=True)

    for L in LEADS_TO_EXPLAIN:
        print("\n" + "-" * 90)
        print(f"Lead {L}h")
        print("-" * 90)

        case_df = datasets[L].copy()

        use_cols = FEATURE_COLS + ["y"]
        case_df = case_df.dropna(subset=use_cols).reset_index(drop=True)

        X = case_df[FEATURE_COLS].copy()
        y = case_df["y"].astype(int).values

        print(
            f"Rows used: {len(case_df)} | "
            f"positive={int(y.sum())} | "
            f"negative={int((y == 0).sum())}"
        )

        # ------------------------------------------------------------
        # Fit final model on all rows for this lead
        # ------------------------------------------------------------
        base_model = clone(models_all[model_name])
        pipe = make_pipeline(model_name, base_model)
        pipe.fit(X, y)

        # ------------------------------------------------------------
        # Balanced SHAP explanation sample
        # ------------------------------------------------------------
        X_bg, X_exp, y_exp, exp_idx = make_shap_samples(
            X=X,
            y=y,
            background_n=SHAP_BACKGROUND_N,
            pos_n=SHAP_POS_N,
            neg_n=SHAP_NEG_N,
            random_state=SHAP_RANDOM_STATE + int(L)
        )

        print(
            f"Background sample: {len(X_bg)} | "
            f"Explanation sample: {len(X_exp)} | "
            f"Explanation positives={int(y_exp.sum())}"
        )

        predict_positive = make_predict_positive_func(pipe, FEATURE_COLS)

        masker = shap.maskers.Independent(X_bg)

        explainer = shap.Explainer(
            predict_positive,
            masker,
            algorithm="permutation",
            feature_names=FEATURE_COLS
        )

        shap_values = explainer(
            X_exp,
            max_evals=2 * len(FEATURE_COLS) + 1,
            batch_size=50
        )

        shap_arr = shap_values.values
        y_prob_exp = pipe.predict_proba(X_exp)[:, 1]

        # ------------------------------------------------------------
        # Save explained sample
        # ------------------------------------------------------------
        sample_out = X_exp.copy()
        sample_out["y_true"] = y_exp
        sample_out["y_prob"] = y_prob_exp
        sample_out["dataset"] = DATASET_NAME
        sample_out["label"] = LABEL_NAME
        sample_out["lead_h"] = L
        sample_out["model"] = model_name

        sample_csv = model_dir / f"shap_explanation_sample_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.csv"
        sample_out.to_csv(sample_csv, index=False)

        all_shap_samples.append(sample_out)

        # ------------------------------------------------------------
        # SHAP global importance table
        # ------------------------------------------------------------
        imp = pd.DataFrame({
            "feature": FEATURE_COLS,
            "mean_abs_shap": np.mean(np.abs(shap_arr), axis=0),
            "mean_shap": np.mean(shap_arr, axis=0),
            "std_shap": np.std(shap_arr, axis=0),
        })

        imp["rank"] = imp["mean_abs_shap"].rank(ascending=False, method="dense").astype(int)
        imp = imp.sort_values("mean_abs_shap", ascending=False)

        imp["dataset"] = DATASET_NAME
        imp["label"] = LABEL_NAME
        imp["lead_h"] = L
        imp["model"] = model_name
        imp["n_explain"] = len(X_exp)
        imp["n_background"] = len(X_bg)
        imp["n_positive_explain"] = int(y_exp.sum())
        imp["n_negative_explain"] = int((y_exp == 0).sum())

        all_shap_importance.append(imp)

        imp_csv = model_dir / f"shap_importance_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.csv"
        imp.to_csv(imp_csv, index=False)

        print("Top SHAP features:")
        display(imp[["lead_h", "model", "feature", "mean_abs_shap", "mean_shap", "rank"]].head(10))

        # ------------------------------------------------------------
        # Save SHAP values
        # ------------------------------------------------------------
        shap_value_df = pd.DataFrame(
            shap_arr,
            columns=[f"shap_{c}" for c in FEATURE_COLS]
        )

        shap_value_df["y_true"] = y_exp
        shap_value_df["y_prob"] = y_prob_exp
        shap_value_df["dataset"] = DATASET_NAME
        shap_value_df["label"] = LABEL_NAME
        shap_value_df["lead_h"] = L
        shap_value_df["model"] = model_name

        shap_values_csv = model_dir / f"shap_values_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.csv"
        shap_value_df.to_csv(shap_values_csv, index=False)

        # ------------------------------------------------------------
        # SHAP bar plot
        # ------------------------------------------------------------
        title_prefix = f"{DATASET_NAME.upper()} {LABEL_NAME} | {model_name} | Lead {L}h"

        bar_png = model_dir / f"shap_bar_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.png"

        plt.figure(figsize=(8, 6))
        shap.plots.bar(
            shap_values,
            max_display=len(FEATURE_COLS),
            show=False
        )
        fig = plt.gcf()
        fig.suptitle(
            f"{title_prefix}\nSHAP global importance: mean absolute SHAP value",
            fontsize=13,
            y=1.02
        )
        plt.tight_layout()
        fig.savefig(bar_png, dpi=220, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        print("Saved:", bar_png)

        # ------------------------------------------------------------
        # SHAP beeswarm plot
        # ------------------------------------------------------------
        beeswarm_png = model_dir / f"shap_beeswarm_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.png"

        plt.figure(figsize=(8.5, 6.5))
        shap.plots.beeswarm(
            shap_values,
            max_display=len(FEATURE_COLS),
            show=False
        )
        fig = plt.gcf()
        fig.suptitle(
            f"{title_prefix}\nSHAP beeswarm: feature effects on developing-class probability",
            fontsize=13,
            y=1.02
        )
        plt.tight_layout()
        fig.savefig(beeswarm_png, dpi=220, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        print("Saved:", beeswarm_png)

        # ------------------------------------------------------------
        # SHAP dependence plots for top predictors
        # ------------------------------------------------------------
        top_features = imp["feature"].head(N_TOP_DEPENDENCE).tolist()

        save_dependence_plots(
            shap_values=shap_values,
            X_exp=X_exp,
            top_features=top_features,
            out_dir=model_dir,
            prefix=title_prefix
        )

# ------------------------------------------------------------
# Save combined SHAP outputs
# ------------------------------------------------------------
shap_importance_all = pd.concat(all_shap_importance, ignore_index=True)
shap_importance_all.to_csv(
    SHAP_DIR / "shap_importance_all_models_all_leads.csv",
    index=False
)

shap_samples_all = pd.concat(all_shap_samples, ignore_index=True)
shap_samples_all.to_csv(
    SHAP_DIR / "shap_explanation_samples_all_models_all_leads.csv",
    index=False
)

print("\nSaved combined SHAP importance:")
print(SHAP_DIR / "shap_importance_all_models_all_leads.csv")

# ------------------------------------------------------------
# SHAP HEATMAP ACROSS LEAD TIMES
# rows = predictors
# cols = lead times
# values = mean absolute SHAP value
# ------------------------------------------------------------
for model_name in SHAP_MODEL_NAMES:
    safe_model = model_name.replace(" ", "_").replace("/", "_")

    sub = shap_importance_all[
        shap_importance_all["model"] == model_name
    ].copy()

    heat = sub.pivot_table(
        index="feature",
        columns="lead_h",
        values="mean_abs_shap",
        aggfunc="mean"
    )

    heat = heat.reindex(index=PREDICTOR_ORDER, columns=LEAD_ORDER)

    vals = heat.values.astype(float)

    fig, ax = plt.subplots(figsize=(8, 7))

    im = ax.imshow(vals, aspect="auto", cmap="viridis")

    ax.set_xticks(np.arange(len(LEAD_ORDER)))
    ax.set_xticklabels([f"{L} h" for L in LEAD_ORDER], fontsize=11)

    ax.set_yticks(np.arange(len(PREDICTOR_ORDER)))
    ax.set_yticklabels(PREDICTOR_ORDER, fontsize=10)

    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8)
            else:
                ax.text(j, i, "NA", ha="center", va="center", fontsize=8)

    ax.set_xlabel("Lead time", fontsize=12)
    ax.set_ylabel("Predictor", fontsize=12)
    ax.set_title(
        f"{DATASET_NAME.upper()} {LABEL_NAME} | {model_name}\n"
        "SHAP heatmap: mean absolute SHAP value",
        fontsize=14
    )

    ax.set_xticks(np.arange(-0.5, len(LEAD_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(PREDICTOR_ORDER), 1), minor=True)
    ax.grid(which="minor", color="gray", linestyle="-", linewidth=0.5, alpha=0.4)
    ax.tick_params(which="minor", bottom=False, left=False)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("mean(|SHAP value|)", fontsize=11)

    plt.tight_layout()

    heat_png = SHAP_DIR / f"shap_heatmap_mean_abs_{DATASET_NAME}_{LABEL_NAME}_{safe_model}.png"
    heat_csv = SHAP_DIR / f"shap_heatmap_mean_abs_{DATASET_NAME}_{LABEL_NAME}_{safe_model}.csv"

    fig.savefig(heat_png, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    heat.to_csv(heat_csv)

    print("Saved:", heat_png)
    print("Saved:", heat_csv)

print("\n[DONE] SHAP analysis complete.")
print("Main output folder:")
print(SHAP_DIR)

ABLATION

In [ ]:
# ============================================================
# FEATURE ABLATION — LEAVE-ONE-FEATURE-OUT
# Compatible with ML_CHECK_LATEST structure
#
# What this does:
#   1. Uses the same CASE_DIR, OUT_DIR, LEADS, FEATURE_COLS
#   2. Uses the same get_models(), make_pipeline(), get_cv(),
#      safe_metrics(), and threshold_scan() functions
#   3. Retrains the selected model after removing one feature
#   4. Compares each ablation against the full-feature baseline
#   5. Saves CSV tables and plots
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone



# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------
ABLATION_DIR = OUT_DIR / "feature_ablation"
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Choose models for ablation
# Start with AdaBoost because it is the main Zhang-style model.
# You can add "Random Forest" later if needed.
# ------------------------------------------------------------
ABLATION_MODEL_NAMES = [
    "AdaBoost",
    "Random Forest",
    "SVM",
    "Logistic Regression",
    "MLP",
]

# ------------------------------------------------------------
# Load case tables if datasets is not already loaded
# ------------------------------------------------------------
try:
    datasets
    print("Using existing datasets dictionary.")
except NameError:
    print("datasets not found. Loading case tables from CASE_DIR...")

    datasets = {}

    for L in LEADS:
        fp = CASE_DIR / f"{LABEL_NAME}_case_table_{DATASET_NAME}_{L}h.csv"

        if not fp.exists():
            raise FileNotFoundError(fp)

        df = pd.read_csv(fp)
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)
        df["y"] = df["y"].astype(int)

        datasets[L] = df.copy()

        print(
            f"Lead {L}h | rows={len(df)} | "
            f"positive={int(df['y'].sum())} | "
            f"negative={int((df['y'] == 0).sum())}"
        )

# ------------------------------------------------------------
# Feature ablation specifications
# ------------------------------------------------------------
ABLATION_SPECS = []

# Baseline: all features
ABLATION_SPECS.append({
    "ablation_type": "baseline",
    "ablation_name": "all_features",
    "removed_features": [],
})

# Individual leave-one-feature-out
for f in FEATURE_COLS:
    ABLATION_SPECS.append({
        "ablation_type": "leave_one_feature_out",
        "ablation_name": f"without_{f}",
        "removed_features": [f],
    })

# Optional physical group ablations
GROUP_ABLATIONS = {
    "without_location_season": ["lon", "lat", "month"],
    "without_mcs_structure": ["area_km2", "tb_mean", "tb_min"],
    "without_mcs_motion": ["speed_kmh", "direction_deg"],
    "without_environment": ["eta850_abs", "rh600", "wshear_200_850", "pi_vmax_ms", "gpi"],
}

for group_name, removed in GROUP_ABLATIONS.items():
    removed_existing = [f for f in removed if f in FEATURE_COLS]

    if len(removed_existing) > 0:
        ABLATION_SPECS.append({
            "ablation_type": "feature_group_out",
            "ablation_name": group_name,
            "removed_features": removed_existing,
        })

print("Number of ablation settings:", len(ABLATION_SPECS))

# ------------------------------------------------------------
# Helper: run one ablation with CV
# ------------------------------------------------------------
def run_one_ablation_cv(
    case_df,
    lead_h,
    model_name,
    model,
    feature_cols_used,
    ablation_type,
    ablation_name,
    removed_features,
    dataset_name=DATASET_NAME,
    label_name=LABEL_NAME,
    random_state=RANDOM_STATE,
    n_splits=N_SPLITS
):
    df = case_df.copy()

    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

    X = df[feature_cols_used].copy()
    y = df["y"].astype(int).values

    groups = df["track_id"].astype(str).values if "track_id" in df.columns else None

    cv, cv_name, n_splits_eff = get_cv(
        y=y,
        groups=groups,
        n_splits=n_splits,
        random_state=random_state
    )

    pipe = make_pipeline(model_name, clone(model))

    fold_rows = []
    oof_rows = []

    if groups is not None:
        splitter = cv.split(X, y, groups)
    else:
        splitter = cv.split(X, y)

    for fold_id, (train_idx, test_idx) in enumerate(splitter, start=1):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        pipe.fit(X_train, y_train)

        if hasattr(pipe, "predict_proba"):
            y_score = pipe.predict_proba(X_test)[:, 1]
        else:
            y_score = pipe.decision_function(X_test)
            y_score = (
                y_score - np.nanmin(y_score)
            ) / (
                np.nanmax(y_score) - np.nanmin(y_score) + 1e-12
            )

        fold_metric = safe_metrics(y_test, y_score, threshold=0.5)

        fold_metric.update({
            "dataset": dataset_name,
            "label": label_name,
            "lead_h": lead_h,
            "model": model_name,
            "ablation_type": ablation_type,
            "ablation_name": ablation_name,
            "removed_features": ",".join(removed_features) if removed_features else "none",
            "n_features_used": len(feature_cols_used),
            "fold": fold_id,
            "cv": cv_name,
            "n_splits": n_splits_eff,
            "train_n": len(train_idx),
            "test_n": len(test_idx),
            "train_pos": int(y_train.sum()),
            "test_pos": int(y_test.sum()),
            "train_neg": int((y_train == 0).sum()),
            "test_neg": int((y_test == 0).sum()),
        })

        fold_rows.append(fold_metric)

        oof = pd.DataFrame({
            "dataset": dataset_name,
            "label": label_name,
            "lead_h": lead_h,
            "model": model_name,
            "ablation_type": ablation_type,
            "ablation_name": ablation_name,
            "removed_features": ",".join(removed_features) if removed_features else "none",
            "fold": fold_id,
            "y_true": y_test,
            "y_score": y_score,
            "y_pred_default": (y_score >= 0.5).astype(int),
        })

        if all(c in df.columns for c in ID_COLS):
            id_part = df.iloc[test_idx][ID_COLS].reset_index(drop=True)
            oof = pd.concat([id_part, oof.reset_index(drop=True)], axis=1)

        oof_rows.append(oof)

    fold_df = pd.DataFrame(fold_rows)
    oof_df = pd.concat(oof_rows, ignore_index=True)

    # ------------------------------------------------------------
    # OOF default-threshold metrics
    # ------------------------------------------------------------
    default_metrics = safe_metrics(
        oof_df["y_true"].values,
        oof_df["y_score"].values,
        threshold=0.5
    )

    # ------------------------------------------------------------
    # OOF tuned-threshold metrics
    # ------------------------------------------------------------
    scan = threshold_scan(
        oof_df["y_true"].values,
        oof_df["y_score"].values
    )

    if len(scan) > 0:
        best_tuned = scan.iloc[0].to_dict()
    else:
        best_tuned = {
            "threshold": np.nan,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "roc_auc": np.nan,
            "pr_auc": np.nan,
        }

    summary_row = {
        "dataset": dataset_name,
        "label": label_name,
        "lead_h": lead_h,
        "model": model_name,
        "ablation_type": ablation_type,
        "ablation_name": ablation_name,
        "removed_features": ",".join(removed_features) if removed_features else "none",
        "features_used": ",".join(feature_cols_used),
        "n_features_used": len(feature_cols_used),
        "cv": cv_name,
        "n_splits": n_splits_eff,
        "n_samples": len(df),
        "n_positive": int(np.sum(y == 1)),
        "n_negative": int(np.sum(y == 0)),

        "threshold_default": 0.5,
        "accuracy_default": default_metrics["accuracy"],
        "precision_default": default_metrics["precision"],
        "recall_default": default_metrics["recall"],
        "f1_default": default_metrics["f1"],
        "roc_auc_default": default_metrics["roc_auc"],
        "pr_auc_default": default_metrics["pr_auc"],
        "tn_default": default_metrics["tn"],
        "fp_default": default_metrics["fp"],
        "fn_default": default_metrics["fn"],
        "tp_default": default_metrics["tp"],

        "threshold_tuned": best_tuned.get("threshold", np.nan),
        "accuracy_tuned": best_tuned.get("accuracy", np.nan),
        "precision_tuned": best_tuned.get("precision", np.nan),
        "recall_tuned": best_tuned.get("recall", np.nan),
        "f1_tuned": best_tuned.get("f1", np.nan),
        "roc_auc_tuned": best_tuned.get("roc_auc", np.nan),
        "pr_auc_tuned": best_tuned.get("pr_auc", np.nan),
    }

    scan["dataset"] = dataset_name
    scan["label"] = label_name
    scan["lead_h"] = lead_h
    scan["model"] = model_name
    scan["ablation_type"] = ablation_type
    scan["ablation_name"] = ablation_name
    scan["removed_features"] = ",".join(removed_features) if removed_features else "none"

    return fold_df, oof_df, pd.DataFrame([summary_row]), scan


# ------------------------------------------------------------
# Run ablation
# ------------------------------------------------------------
models_all = get_models(random_state=RANDOM_STATE)

missing_models = [m for m in ABLATION_MODEL_NAMES if m not in models_all]
if missing_models:
    raise ValueError(
        f"These models are not available in get_models(): {missing_models}\n"
        f"Available models: {list(models_all.keys())}"
    )

all_ablation_folds = []
all_ablation_oof = []
all_ablation_summary = []
all_ablation_scans = []

for L in LEADS:
    case_df = datasets[L].copy()

    print("\n" + "=" * 90)
    print(
        f"FEATURE ABLATION | lead={L}h | "
        f"rows={len(case_df)} | positives={int(case_df['y'].sum())}"
    )
    print("=" * 90)

    for model_name in ABLATION_MODEL_NAMES:
        base_model = models_all[model_name]

        print(f"\nModel: {model_name}")

        for spec in ABLATION_SPECS:
            removed = spec["removed_features"]
            feature_cols_used = [f for f in FEATURE_COLS if f not in removed]

            if len(feature_cols_used) == 0:
                print(f"[SKIP] {spec['ablation_name']} removes all features.")
                continue

            print(
                f"  Running {spec['ablation_name']} "
                f"({len(feature_cols_used)} features)"
            )

            try:
                fold_df, oof_df, summary_df, scan_df = run_one_ablation_cv(
                    case_df=case_df,
                    lead_h=L,
                    model_name=model_name,
                    model=base_model,
                    feature_cols_used=feature_cols_used,
                    ablation_type=spec["ablation_type"],
                    ablation_name=spec["ablation_name"],
                    removed_features=removed,
                    dataset_name=DATASET_NAME,
                    label_name=LABEL_NAME,
                    random_state=RANDOM_STATE,
                    n_splits=N_SPLITS
                )

                all_ablation_folds.append(fold_df)
                all_ablation_oof.append(oof_df)
                all_ablation_summary.append(summary_df)
                all_ablation_scans.append(scan_df)

            except Exception as e:
                print(f"  [ERROR] {model_name} | lead {L}h | {spec['ablation_name']}: {e}")

                err = pd.DataFrame([{
                    "dataset": DATASET_NAME,
                    "label": LABEL_NAME,
                    "lead_h": L,
                    "model": model_name,
                    "ablation_type": spec["ablation_type"],
                    "ablation_name": spec["ablation_name"],
                    "removed_features": ",".join(removed) if removed else "none",
                    "error": str(e),
                }])

                all_ablation_summary.append(err)

# ------------------------------------------------------------
# Save raw outputs
# ------------------------------------------------------------
ablation_folds = pd.concat(all_ablation_folds, ignore_index=True)
ablation_oof = pd.concat(all_ablation_oof, ignore_index=True)
ablation_summary = pd.concat(all_ablation_summary, ignore_index=True)
ablation_scans = pd.concat(all_ablation_scans, ignore_index=True)

ablation_folds.to_csv(ABLATION_DIR / "feature_ablation_fold_results.csv", index=False)
ablation_oof.to_csv(ABLATION_DIR / "feature_ablation_oof_predictions.csv", index=False)
ablation_summary.to_csv(ABLATION_DIR / "feature_ablation_summary.csv", index=False)
ablation_scans.to_csv(ABLATION_DIR / "feature_ablation_threshold_scans.csv", index=False)

print("\n[SAVED]")
print(ABLATION_DIR / "feature_ablation_fold_results.csv")
print(ABLATION_DIR / "feature_ablation_oof_predictions.csv")
print(ABLATION_DIR / "feature_ablation_summary.csv")
print(ABLATION_DIR / "feature_ablation_threshold_scans.csv")

# ------------------------------------------------------------
# Compare each ablation against full-feature baseline
# Positive drop means performance became worse after removal.
# ------------------------------------------------------------
summary_ok = ablation_summary[ablation_summary["error"].isna() if "error" in ablation_summary.columns else slice(None)].copy()

baseline = summary_ok[
    summary_ok["ablation_name"] == "all_features"
].copy()

baseline_cols = [
    "lead_h", "model",
    "f1_default", "precision_default", "recall_default",
    "roc_auc_default", "pr_auc_default",
    "f1_tuned", "precision_tuned", "recall_tuned",
    "roc_auc_tuned", "pr_auc_tuned",
]

baseline = baseline[baseline_cols].copy()
baseline = baseline.rename(columns={
    "f1_default": "baseline_f1_default",
    "precision_default": "baseline_precision_default",
    "recall_default": "baseline_recall_default",
    "roc_auc_default": "baseline_roc_auc_default",
    "pr_auc_default": "baseline_pr_auc_default",
    "f1_tuned": "baseline_f1_tuned",
    "precision_tuned": "baseline_precision_tuned",
    "recall_tuned": "baseline_recall_tuned",
    "roc_auc_tuned": "baseline_roc_auc_tuned",
    "pr_auc_tuned": "baseline_pr_auc_tuned",
})

impact = summary_ok.merge(
    baseline,
    on=["lead_h", "model"],
    how="left"
)

impact["drop_f1_default"] = impact["baseline_f1_default"] - impact["f1_default"]
impact["drop_precision_default"] = impact["baseline_precision_default"] - impact["precision_default"]
impact["drop_recall_default"] = impact["baseline_recall_default"] - impact["recall_default"]
impact["drop_roc_auc_default"] = impact["baseline_roc_auc_default"] - impact["roc_auc_default"]
impact["drop_pr_auc_default"] = impact["baseline_pr_auc_default"] - impact["pr_auc_default"]

impact["drop_f1_tuned"] = impact["baseline_f1_tuned"] - impact["f1_tuned"]
impact["drop_precision_tuned"] = impact["baseline_precision_tuned"] - impact["precision_tuned"]
impact["drop_recall_tuned"] = impact["baseline_recall_tuned"] - impact["recall_tuned"]
impact["drop_roc_auc_tuned"] = impact["baseline_roc_auc_tuned"] - impact["roc_auc_tuned"]
impact["drop_pr_auc_tuned"] = impact["baseline_pr_auc_tuned"] - impact["pr_auc_tuned"]

impact = impact.sort_values(
    ["lead_h", "model", "ablation_type", "drop_f1_default"],
    ascending=[True, True, True, False]
).reset_index(drop=True)

impact.to_csv(ABLATION_DIR / "feature_ablation_impact_vs_baseline.csv", index=False)

print("\nFeature ablation impact table:")
display(
    impact[
        [
            "lead_h", "model", "ablation_type", "ablation_name", "removed_features",
            "baseline_f1_default", "f1_default", "drop_f1_default",
            "baseline_pr_auc_default", "pr_auc_default", "drop_pr_auc_default",
            "baseline_f1_tuned", "f1_tuned", "drop_f1_tuned",
        ]
    ]
)

# ------------------------------------------------------------
# Save ranked individual-feature ablation table
# ------------------------------------------------------------
individual_impact = impact[
    impact["ablation_type"] == "leave_one_feature_out"
].copy()

individual_impact = individual_impact.sort_values(
    ["lead_h", "model", "drop_f1_default"],
    ascending=[True, True, False]
)

individual_impact.to_csv(
    ABLATION_DIR / "feature_ablation_individual_ranked_by_drop_f1.csv",
    index=False
)

print("\nIndividual feature ablation ranked by F1 drop:")
display(
    individual_impact[
        [
            "lead_h", "model", "removed_features",
            "f1_default", "drop_f1_default",
            "pr_auc_default", "drop_pr_auc_default",
            "roc_auc_default", "drop_roc_auc_default",
        ]
    ]
)

# ------------------------------------------------------------
# Pivot table: F1 drop by feature and lead
# ------------------------------------------------------------
for model_name in ABLATION_MODEL_NAMES:
    sub = individual_impact[individual_impact["model"] == model_name].copy()

    if len(sub) == 0:
        continue

    pivot_drop_f1 = sub.pivot(
        index="removed_features",
        columns="lead_h",
        values="drop_f1_default"
    )

    pivot_drop_f1 = pivot_drop_f1.reindex(
        pivot_drop_f1.mean(axis=1).sort_values(ascending=False).index
    )

    safe_model = model_name.replace(" ", "_").replace("/", "_")

    pivot_fp = ABLATION_DIR / f"feature_ablation_pivot_drop_f1_{safe_model}.csv"
    pivot_drop_f1.to_csv(pivot_fp)

    print(f"\nF1 drop pivot table — {model_name}:")
    display(pivot_drop_f1)

# ------------------------------------------------------------
# Plots: individual feature F1 drop by lead/model
# ------------------------------------------------------------
for model_name in ABLATION_MODEL_NAMES:
    for L in LEADS:
        sub = individual_impact[
            (individual_impact["model"] == model_name)
            & (individual_impact["lead_h"] == L)
        ].copy()

        if len(sub) == 0:
            continue

        sub = sub.sort_values("drop_f1_default", ascending=True)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh(sub["removed_features"], sub["drop_f1_default"])
        ax.axvline(0, linewidth=1)
        ax.set_xlabel("F1 drop after removing feature")
        ax.set_ylabel("Removed feature")
        ax.set_title(
            f"{DATASET_NAME.upper()} {LABEL_NAME} | {model_name} | Lead {L}h\n"
            "Leave-one-feature-out ablation"
        )
        ax.grid(True, axis="x", alpha=0.3)

        plt.tight_layout()

        safe_model = model_name.replace(" ", "_").replace("/", "_")
        out_png = ABLATION_DIR / f"feature_ablation_drop_f1_{safe_model}_{L}h.png"

        plt.savefig(out_png, dpi=220, bbox_inches="tight")
        plt.show()

        print("Saved:", out_png)

# ------------------------------------------------------------
# Plots: feature-group ablation
# ------------------------------------------------------------
group_impact = impact[
    impact["ablation_type"] == "feature_group_out"
].copy()

if len(group_impact) > 0:
    for model_name in ABLATION_MODEL_NAMES:
        for L in LEADS:
            sub = group_impact[
                (group_impact["model"] == model_name)
                & (group_impact["lead_h"] == L)
            ].copy()

            if len(sub) == 0:
                continue

            sub = sub.sort_values("drop_f1_default", ascending=True)

            fig, ax = plt.subplots(figsize=(8, 4.8))
            ax.barh(sub["ablation_name"], sub["drop_f1_default"])
            ax.axvline(0, linewidth=1)
            ax.set_xlabel("F1 drop after removing feature group")
            ax.set_ylabel("Removed group")
            ax.set_title(
                f"{DATASET_NAME.upper()} {LABEL_NAME} | {model_name} | Lead {L}h\n"
                "Feature-group ablation"
            )
            ax.grid(True, axis="x", alpha=0.3)

            plt.tight_layout()

            safe_model = model_name.replace(" ", "_").replace("/", "_")
            out_png = ABLATION_DIR / f"feature_group_ablation_drop_f1_{safe_model}_{L}h.png"

            plt.savefig(out_png, dpi=220, bbox_inches="tight")
            plt.show()

            print("Saved:", out_png)

print("\n[DONE] Feature ablation complete.")
print("Main table:")
print(ABLATION_DIR / "feature_ablation_impact_vs_baseline.csv")

# ============================================================
# FEATURE ABLATION HEATMAP
# Rows   = predictors
# Cols   = lead times
# Values = F1 drop after removing each feature
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
# If already defined in your notebook, this will use it.
# Otherwise, set the folder manually.
try:
    ABLATION_DIR
except NameError:
    ABLATION_DIR = OUT_DIR / "feature_ablation"

CSV_FP = ABLATION_DIR / "feature_ablation_impact_vs_baseline.csv"

# Choose what to plot
TARGET_DATASET = "OLD"       # e.g. "OLD" or "NEW"
TARGET_LABEL   = "box2deg"   # e.g. "box2deg" or "300km"
MODELS_TO_PLOT = ["AdaBoost", "Random Forest", "SVM", "Logistic Regression", "MLP"]   # or ["AdaBoost", "Random Forest"]

# metric options:
#   drop_f1_default
#   drop_f1_tuned
#   drop_pr_auc_default
#   drop_pr_auc_tuned
METRIC_COL = "drop_f1_default"
TITLE_METRIC = "F1 drop"

# exact predictor order requested
PREDICTOR_ORDER = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
    "month",
    "lat",
    "lon",
]

LEAD_ORDER = [6, 12, 24, 48]

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
df = pd.read_csv(CSV_FP)

# keep only individual leave-one-feature-out rows
df = df[df["ablation_type"] == "leave_one_feature_out"].copy()

# optional filter by dataset/label if columns exist
if "dataset" in df.columns:
    df = df[df["dataset"].astype(str).str.upper() == TARGET_DATASET.upper()].copy()

if "label" in df.columns:
    df = df[df["label"].astype(str) == TARGET_LABEL].copy()

print("Loaded rows:", len(df))
print("Models available:", sorted(df["model"].unique()))
print("Leads available:", sorted(df["lead_h"].unique()))

# ------------------------------------------------------------
# MAKE HEATMAP FOR EACH MODEL
# ------------------------------------------------------------
for model_name in MODELS_TO_PLOT:
    sub = df[df["model"] == model_name].copy()

    if len(sub) == 0:
        print(f"[SKIP] No rows found for model: {model_name}")
        continue

    heat = sub.pivot_table(
        index="removed_features",
        columns="lead_h",
        values=METRIC_COL,
        aggfunc="mean"
    )

    heat = heat.reindex(index=PREDICTOR_ORDER, columns=LEAD_ORDER)

    vals = heat.values.astype(float)

    # robust color scaling centered at zero
    finite_vals = vals[np.isfinite(vals)]
    if len(finite_vals) == 0:
        print(f"[SKIP] No finite values for model: {model_name}")
        continue

    vmax = np.nanmax(np.abs(finite_vals))
    vmin = -vmax
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(vals, cmap="RdBu_r", norm=norm, aspect="auto")

    # ticks
    ax.set_xticks(np.arange(len(LEAD_ORDER)))
    ax.set_xticklabels([f"{l} h" for l in LEAD_ORDER], fontsize=11)

    ax.set_yticks(np.arange(len(PREDICTOR_ORDER)))
    ax.set_yticklabels(PREDICTOR_ORDER, fontsize=10)

    # annotate each cell
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]
            if np.isfinite(v):
                txt_color = "white" if abs(v) > 0.5 * vmax else "black"
                ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                        fontsize=9, color=txt_color)
            else:
                ax.text(j, i, "NA", ha="center", va="center",
                        fontsize=8, color="black")

    # title
    ax.set_title(
        f"{TARGET_DATASET} {TARGET_LABEL} | {model_name}\n"
        f"Feature ablation heatmap ({TITLE_METRIC})",
        fontsize=14
    )

    ax.set_xlabel("Lead time", fontsize=12)
    ax.set_ylabel("Removed predictor", fontsize=12)

    # minor grid to separate cells
    ax.set_xticks(np.arange(-0.5, len(LEAD_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(PREDICTOR_ORDER), 1), minor=True)
    ax.grid(which="minor", color="gray", linestyle="-", linewidth=0.5, alpha=0.4)
    ax.tick_params(which="minor", bottom=False, left=False)

    # colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(TITLE_METRIC, fontsize=11)

    plt.tight_layout()

    safe_model = model_name.replace(" ", "_").replace("/", "_")
    out_png = ABLATION_DIR / f"feature_ablation_heatmap_{TARGET_DATASET}_{TARGET_LABEL}_{safe_model}_{METRIC_COL}.png"
    plt.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)
    print()

NAIVE BAYES

In [ ]:
# ============================================================
# FEATURE ABLATION — LEAVE-ONE-FEATURE-OUT
# Compatible with ML_CHECK_LATEST structure
#
# What this does:
#   1. Uses the same CASE_DIR, OUT_DIR, LEADS, FEATURE_COLS
#   2. Uses the same get_models(), make_pipeline(), get_cv(),
#      safe_metrics(), and threshold_scan() functions
#   3. Retrains the selected model after removing one feature
#   4. Compares each ablation against the full-feature baseline
#   5. Saves CSV tables and plots
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone



# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------
ABLATION_DIR = OUT_DIR / "feature_ablation"
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Choose models for ablation
# Start with AdaBoost because it is the main Zhang-style model.
# You can add "Random Forest" later if needed.
# ------------------------------------------------------------
ABLATION_MODEL_NAMES = [
    "Naive Bayes",
]

# ------------------------------------------------------------
# Load case tables if datasets is not already loaded
# ------------------------------------------------------------
try:
    datasets
    print("Using existing datasets dictionary.")
except NameError:
    print("datasets not found. Loading case tables from CASE_DIR...")

    datasets = {}

    for L in LEADS:
        fp = CASE_DIR / f"{LABEL_NAME}_case_table_{DATASET_NAME}_{L}h.csv"

        if not fp.exists():
            raise FileNotFoundError(fp)

        df = pd.read_csv(fp)
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)
        df["y"] = df["y"].astype(int)

        datasets[L] = df.copy()

        print(
            f"Lead {L}h | rows={len(df)} | "
            f"positive={int(df['y'].sum())} | "
            f"negative={int((df['y'] == 0).sum())}"
        )

# ------------------------------------------------------------
# Feature ablation specifications
# ------------------------------------------------------------
ABLATION_SPECS = []

# Baseline: all features
ABLATION_SPECS.append({
    "ablation_type": "baseline",
    "ablation_name": "all_features",
    "removed_features": [],
})

# Individual leave-one-feature-out
for f in FEATURE_COLS:
    ABLATION_SPECS.append({
        "ablation_type": "leave_one_feature_out",
        "ablation_name": f"without_{f}",
        "removed_features": [f],
    })

# Optional physical group ablations
GROUP_ABLATIONS = {
    "without_location_season": ["lon", "lat", "month"],
    "without_mcs_structure": ["area_km2", "tb_mean", "tb_min"],
    "without_mcs_motion": ["speed_kmh", "direction_deg"],
    "without_environment": ["eta850_abs", "rh600", "wshear_200_850", "pi_vmax_ms", "gpi"],
}

for group_name, removed in GROUP_ABLATIONS.items():
    removed_existing = [f for f in removed if f in FEATURE_COLS]

    if len(removed_existing) > 0:
        ABLATION_SPECS.append({
            "ablation_type": "feature_group_out",
            "ablation_name": group_name,
            "removed_features": removed_existing,
        })

print("Number of ablation settings:", len(ABLATION_SPECS))

# ------------------------------------------------------------
# Helper: run one ablation with CV
# ------------------------------------------------------------
def run_one_ablation_cv(
    case_df,
    lead_h,
    model_name,
    model,
    feature_cols_used,
    ablation_type,
    ablation_name,
    removed_features,
    dataset_name=DATASET_NAME,
    label_name=LABEL_NAME,
    random_state=RANDOM_STATE,
    n_splits=N_SPLITS
):
    df = case_df.copy()

    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

    X = df[feature_cols_used].copy()
    y = df["y"].astype(int).values

    groups = df["track_id"].astype(str).values if "track_id" in df.columns else None

    cv, cv_name, n_splits_eff = get_cv(
        y=y,
        groups=groups,
        n_splits=n_splits,
        random_state=random_state
    )

    pipe = make_pipeline(model_name, clone(model))

    fold_rows = []
    oof_rows = []

    if groups is not None:
        splitter = cv.split(X, y, groups)
    else:
        splitter = cv.split(X, y)

    for fold_id, (train_idx, test_idx) in enumerate(splitter, start=1):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        pipe.fit(X_train, y_train)

        if hasattr(pipe, "predict_proba"):
            y_score = pipe.predict_proba(X_test)[:, 1]
        else:
            y_score = pipe.decision_function(X_test)
            y_score = (
                y_score - np.nanmin(y_score)
            ) / (
                np.nanmax(y_score) - np.nanmin(y_score) + 1e-12
            )

        fold_metric = safe_metrics(y_test, y_score, threshold=0.5)

        fold_metric.update({
            "dataset": dataset_name,
            "label": label_name,
            "lead_h": lead_h,
            "model": model_name,
            "ablation_type": ablation_type,
            "ablation_name": ablation_name,
            "removed_features": ",".join(removed_features) if removed_features else "none",
            "n_features_used": len(feature_cols_used),
            "fold": fold_id,
            "cv": cv_name,
            "n_splits": n_splits_eff,
            "train_n": len(train_idx),
            "test_n": len(test_idx),
            "train_pos": int(y_train.sum()),
            "test_pos": int(y_test.sum()),
            "train_neg": int((y_train == 0).sum()),
            "test_neg": int((y_test == 0).sum()),
        })

        fold_rows.append(fold_metric)

        oof = pd.DataFrame({
            "dataset": dataset_name,
            "label": label_name,
            "lead_h": lead_h,
            "model": model_name,
            "ablation_type": ablation_type,
            "ablation_name": ablation_name,
            "removed_features": ",".join(removed_features) if removed_features else "none",
            "fold": fold_id,
            "y_true": y_test,
            "y_score": y_score,
            "y_pred_default": (y_score >= 0.5).astype(int),
        })

        if all(c in df.columns for c in ID_COLS):
            id_part = df.iloc[test_idx][ID_COLS].reset_index(drop=True)
            oof = pd.concat([id_part, oof.reset_index(drop=True)], axis=1)

        oof_rows.append(oof)

    fold_df = pd.DataFrame(fold_rows)
    oof_df = pd.concat(oof_rows, ignore_index=True)

    # ------------------------------------------------------------
    # OOF default-threshold metrics
    # ------------------------------------------------------------
    default_metrics = safe_metrics(
        oof_df["y_true"].values,
        oof_df["y_score"].values,
        threshold=0.5
    )

    # ------------------------------------------------------------
    # OOF tuned-threshold metrics
    # ------------------------------------------------------------
    scan = threshold_scan(
        oof_df["y_true"].values,
        oof_df["y_score"].values
    )

    if len(scan) > 0:
        best_tuned = scan.iloc[0].to_dict()
    else:
        best_tuned = {
            "threshold": np.nan,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "roc_auc": np.nan,
            "pr_auc": np.nan,
        }

    summary_row = {
        "dataset": dataset_name,
        "label": label_name,
        "lead_h": lead_h,
        "model": model_name,
        "ablation_type": ablation_type,
        "ablation_name": ablation_name,
        "removed_features": ",".join(removed_features) if removed_features else "none",
        "features_used": ",".join(feature_cols_used),
        "n_features_used": len(feature_cols_used),
        "cv": cv_name,
        "n_splits": n_splits_eff,
        "n_samples": len(df),
        "n_positive": int(np.sum(y == 1)),
        "n_negative": int(np.sum(y == 0)),

        "threshold_default": 0.5,
        "accuracy_default": default_metrics["accuracy"],
        "precision_default": default_metrics["precision"],
        "recall_default": default_metrics["recall"],
        "f1_default": default_metrics["f1"],
        "roc_auc_default": default_metrics["roc_auc"],
        "pr_auc_default": default_metrics["pr_auc"],
        "tn_default": default_metrics["tn"],
        "fp_default": default_metrics["fp"],
        "fn_default": default_metrics["fn"],
        "tp_default": default_metrics["tp"],

        "threshold_tuned": best_tuned.get("threshold", np.nan),
        "accuracy_tuned": best_tuned.get("accuracy", np.nan),
        "precision_tuned": best_tuned.get("precision", np.nan),
        "recall_tuned": best_tuned.get("recall", np.nan),
        "f1_tuned": best_tuned.get("f1", np.nan),
        "roc_auc_tuned": best_tuned.get("roc_auc", np.nan),
        "pr_auc_tuned": best_tuned.get("pr_auc", np.nan),
    }

    scan["dataset"] = dataset_name
    scan["label"] = label_name
    scan["lead_h"] = lead_h
    scan["model"] = model_name
    scan["ablation_type"] = ablation_type
    scan["ablation_name"] = ablation_name
    scan["removed_features"] = ",".join(removed_features) if removed_features else "none"

    return fold_df, oof_df, pd.DataFrame([summary_row]), scan


# ------------------------------------------------------------
# Run ablation
# ------------------------------------------------------------
models_all = get_models(random_state=RANDOM_STATE)

missing_models = [m for m in ABLATION_MODEL_NAMES if m not in models_all]
if missing_models:
    raise ValueError(
        f"These models are not available in get_models(): {missing_models}\n"
        f"Available models: {list(models_all.keys())}"
    )

all_ablation_folds = []
all_ablation_oof = []
all_ablation_summary = []
all_ablation_scans = []

for L in LEADS:
    case_df = datasets[L].copy()

    print("\n" + "=" * 90)
    print(
        f"FEATURE ABLATION | lead={L}h | "
        f"rows={len(case_df)} | positives={int(case_df['y'].sum())}"
    )
    print("=" * 90)

    for model_name in ABLATION_MODEL_NAMES:
        base_model = models_all[model_name]

        print(f"\nModel: {model_name}")

        for spec in ABLATION_SPECS:
            removed = spec["removed_features"]
            feature_cols_used = [f for f in FEATURE_COLS if f not in removed]

            if len(feature_cols_used) == 0:
                print(f"[SKIP] {spec['ablation_name']} removes all features.")
                continue

            print(
                f"  Running {spec['ablation_name']} "
                f"({len(feature_cols_used)} features)"
            )

            try:
                fold_df, oof_df, summary_df, scan_df = run_one_ablation_cv(
                    case_df=case_df,
                    lead_h=L,
                    model_name=model_name,
                    model=base_model,
                    feature_cols_used=feature_cols_used,
                    ablation_type=spec["ablation_type"],
                    ablation_name=spec["ablation_name"],
                    removed_features=removed,
                    dataset_name=DATASET_NAME,
                    label_name=LABEL_NAME,
                    random_state=RANDOM_STATE,
                    n_splits=N_SPLITS
                )

                all_ablation_folds.append(fold_df)
                all_ablation_oof.append(oof_df)
                all_ablation_summary.append(summary_df)
                all_ablation_scans.append(scan_df)

            except Exception as e:
                print(f"  [ERROR] {model_name} | lead {L}h | {spec['ablation_name']}: {e}")

                err = pd.DataFrame([{
                    "dataset": DATASET_NAME,
                    "label": LABEL_NAME,
                    "lead_h": L,
                    "model": model_name,
                    "ablation_type": spec["ablation_type"],
                    "ablation_name": spec["ablation_name"],
                    "removed_features": ",".join(removed) if removed else "none",
                    "error": str(e),
                }])

                all_ablation_summary.append(err)

# ------------------------------------------------------------
# Save raw outputs
# ------------------------------------------------------------
ablation_folds = pd.concat(all_ablation_folds, ignore_index=True)
ablation_oof = pd.concat(all_ablation_oof, ignore_index=True)
ablation_summary = pd.concat(all_ablation_summary, ignore_index=True)
ablation_scans = pd.concat(all_ablation_scans, ignore_index=True)

ablation_folds.to_csv(ABLATION_DIR / "feature_ablation_fold_results.csv", index=False)
ablation_oof.to_csv(ABLATION_DIR / "feature_ablation_oof_predictions.csv", index=False)
ablation_summary.to_csv(ABLATION_DIR / "feature_ablation_summary.csv", index=False)
ablation_scans.to_csv(ABLATION_DIR / "feature_ablation_threshold_scans.csv", index=False)

print("\n[SAVED]")
print(ABLATION_DIR / "feature_ablation_fold_results.csv")
print(ABLATION_DIR / "feature_ablation_oof_predictions.csv")
print(ABLATION_DIR / "feature_ablation_summary.csv")
print(ABLATION_DIR / "feature_ablation_threshold_scans.csv")

# ------------------------------------------------------------
# Compare each ablation against full-feature baseline
# Positive drop means performance became worse after removal.
# ------------------------------------------------------------
summary_ok = ablation_summary[ablation_summary["error"].isna() if "error" in ablation_summary.columns else slice(None)].copy()

baseline = summary_ok[
    summary_ok["ablation_name"] == "all_features"
].copy()

baseline_cols = [
    "lead_h", "model",
    "f1_default", "precision_default", "recall_default",
    "roc_auc_default", "pr_auc_default",
    "f1_tuned", "precision_tuned", "recall_tuned",
    "roc_auc_tuned", "pr_auc_tuned",
]

baseline = baseline[baseline_cols].copy()
baseline = baseline.rename(columns={
    "f1_default": "baseline_f1_default",
    "precision_default": "baseline_precision_default",
    "recall_default": "baseline_recall_default",
    "roc_auc_default": "baseline_roc_auc_default",
    "pr_auc_default": "baseline_pr_auc_default",
    "f1_tuned": "baseline_f1_tuned",
    "precision_tuned": "baseline_precision_tuned",
    "recall_tuned": "baseline_recall_tuned",
    "roc_auc_tuned": "baseline_roc_auc_tuned",
    "pr_auc_tuned": "baseline_pr_auc_tuned",
})

impact = summary_ok.merge(
    baseline,
    on=["lead_h", "model"],
    how="left"
)

impact["drop_f1_default"] = impact["baseline_f1_default"] - impact["f1_default"]
impact["drop_precision_default"] = impact["baseline_precision_default"] - impact["precision_default"]
impact["drop_recall_default"] = impact["baseline_recall_default"] - impact["recall_default"]
impact["drop_roc_auc_default"] = impact["baseline_roc_auc_default"] - impact["roc_auc_default"]
impact["drop_pr_auc_default"] = impact["baseline_pr_auc_default"] - impact["pr_auc_default"]

impact["drop_f1_tuned"] = impact["baseline_f1_tuned"] - impact["f1_tuned"]
impact["drop_precision_tuned"] = impact["baseline_precision_tuned"] - impact["precision_tuned"]
impact["drop_recall_tuned"] = impact["baseline_recall_tuned"] - impact["recall_tuned"]
impact["drop_roc_auc_tuned"] = impact["baseline_roc_auc_tuned"] - impact["roc_auc_tuned"]
impact["drop_pr_auc_tuned"] = impact["baseline_pr_auc_tuned"] - impact["pr_auc_tuned"]

impact = impact.sort_values(
    ["lead_h", "model", "ablation_type", "drop_f1_default"],
    ascending=[True, True, True, False]
).reset_index(drop=True)

impact.to_csv(ABLATION_DIR / "feature_ablation_impact_vs_baseline.csv", index=False)

print("\nFeature ablation impact table:")
display(
    impact[
        [
            "lead_h", "model", "ablation_type", "ablation_name", "removed_features",
            "baseline_f1_default", "f1_default", "drop_f1_default",
            "baseline_pr_auc_default", "pr_auc_default", "drop_pr_auc_default",
            "baseline_f1_tuned", "f1_tuned", "drop_f1_tuned",
        ]
    ]
)

# ------------------------------------------------------------
# Save ranked individual-feature ablation table
# ------------------------------------------------------------
individual_impact = impact[
    impact["ablation_type"] == "leave_one_feature_out"
].copy()

individual_impact = individual_impact.sort_values(
    ["lead_h", "model", "drop_f1_default"],
    ascending=[True, True, False]
)

individual_impact.to_csv(
    ABLATION_DIR / "feature_ablation_individual_ranked_by_drop_f1.csv",
    index=False
)

print("\nIndividual feature ablation ranked by F1 drop:")
display(
    individual_impact[
        [
            "lead_h", "model", "removed_features",
            "f1_default", "drop_f1_default",
            "pr_auc_default", "drop_pr_auc_default",
            "roc_auc_default", "drop_roc_auc_default",
        ]
    ]
)

# ------------------------------------------------------------
# Pivot table: F1 drop by feature and lead
# ------------------------------------------------------------
for model_name in ABLATION_MODEL_NAMES:
    sub = individual_impact[individual_impact["model"] == model_name].copy()

    if len(sub) == 0:
        continue

    pivot_drop_f1 = sub.pivot(
        index="removed_features",
        columns="lead_h",
        values="drop_f1_default"
    )

    pivot_drop_f1 = pivot_drop_f1.reindex(
        pivot_drop_f1.mean(axis=1).sort_values(ascending=False).index
    )

    safe_model = model_name.replace(" ", "_").replace("/", "_")

    pivot_fp = ABLATION_DIR / f"feature_ablation_pivot_drop_f1_{safe_model}.csv"
    pivot_drop_f1.to_csv(pivot_fp)

    print(f"\nF1 drop pivot table — {model_name}:")
    display(pivot_drop_f1)

# ------------------------------------------------------------
# Plots: individual feature F1 drop by lead/model
# ------------------------------------------------------------
for model_name in ABLATION_MODEL_NAMES:
    for L in LEADS:
        sub = individual_impact[
            (individual_impact["model"] == model_name)
            & (individual_impact["lead_h"] == L)
        ].copy()

        if len(sub) == 0:
            continue

        sub = sub.sort_values("drop_f1_default", ascending=True)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh(sub["removed_features"], sub["drop_f1_default"])
        ax.axvline(0, linewidth=1)
        ax.set_xlabel("F1 drop after removing feature")
        ax.set_ylabel("Removed feature")
        ax.set_title(
            f"{DATASET_NAME.upper()} {LABEL_NAME} | {model_name} | Lead {L}h\n"
            "Leave-one-feature-out ablation"
        )
        ax.grid(True, axis="x", alpha=0.3)

        plt.tight_layout()

        safe_model = model_name.replace(" ", "_").replace("/", "_")
        out_png = ABLATION_DIR / f"feature_ablation_drop_f1_{safe_model}_{L}h.png"

        plt.savefig(out_png, dpi=220, bbox_inches="tight")
        plt.show()

        print("Saved:", out_png)

# ------------------------------------------------------------
# Plots: feature-group ablation
# ------------------------------------------------------------
group_impact = impact[
    impact["ablation_type"] == "feature_group_out"
].copy()

if len(group_impact) > 0:
    for model_name in ABLATION_MODEL_NAMES:
        for L in LEADS:
            sub = group_impact[
                (group_impact["model"] == model_name)
                & (group_impact["lead_h"] == L)
            ].copy()

            if len(sub) == 0:
                continue

            sub = sub.sort_values("drop_f1_default", ascending=True)

            fig, ax = plt.subplots(figsize=(8, 4.8))
            ax.barh(sub["ablation_name"], sub["drop_f1_default"])
            ax.axvline(0, linewidth=1)
            ax.set_xlabel("F1 drop after removing feature group")
            ax.set_ylabel("Removed group")
            ax.set_title(
                f"{DATASET_NAME.upper()} {LABEL_NAME} | {model_name} | Lead {L}h\n"
                "Feature-group ablation"
            )
            ax.grid(True, axis="x", alpha=0.3)

            plt.tight_layout()

            safe_model = model_name.replace(" ", "_").replace("/", "_")
            out_png = ABLATION_DIR / f"feature_group_ablation_drop_f1_{safe_model}_{L}h.png"

            plt.savefig(out_png, dpi=220, bbox_inches="tight")
            plt.show()

            print("Saved:", out_png)

print("\n[DONE] Feature ablation complete.")
print("Main table:")
print(ABLATION_DIR / "feature_ablation_impact_vs_baseline.csv")

# ============================================================
# FEATURE ABLATION HEATMAP
# Rows   = predictors
# Cols   = lead times
# Values = F1 drop after removing each feature
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
# If already defined in your notebook, this will use it.
# Otherwise, set the folder manually.
try:
    ABLATION_DIR
except NameError:
    ABLATION_DIR = OUT_DIR / "feature_ablation"

CSV_FP = ABLATION_DIR / "feature_ablation_impact_vs_baseline.csv"

# Choose what to plot
TARGET_DATASET = "OLD"       # e.g. "OLD" or "NEW"
TARGET_LABEL   = "box2deg"   # e.g. "box2deg" or "300km"
MODELS_TO_PLOT = ["AdaBoost", "Random Forest", "SVM", "Logistic Regression", "MLP"]   # or ["AdaBoost", "Random Forest"]

# metric options:
#   drop_f1_default
#   drop_f1_tuned
#   drop_pr_auc_default
#   drop_pr_auc_tuned
METRIC_COL = "drop_f1_default"
TITLE_METRIC = "F1 drop"

# exact predictor order requested
PREDICTOR_ORDER = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
    "month",
    "lat",
    "lon",
]

LEAD_ORDER = [6, 12, 24, 48]

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
df = pd.read_csv(CSV_FP)

# keep only individual leave-one-feature-out rows
df = df[df["ablation_type"] == "leave_one_feature_out"].copy()

# optional filter by dataset/label if columns exist
if "dataset" in df.columns:
    df = df[df["dataset"].astype(str).str.upper() == TARGET_DATASET.upper()].copy()

if "label" in df.columns:
    df = df[df["label"].astype(str) == TARGET_LABEL].copy()

print("Loaded rows:", len(df))
print("Models available:", sorted(df["model"].unique()))
print("Leads available:", sorted(df["lead_h"].unique()))

# ------------------------------------------------------------
# MAKE HEATMAP FOR EACH MODEL
# ------------------------------------------------------------
for model_name in MODELS_TO_PLOT:
    sub = df[df["model"] == model_name].copy()

    if len(sub) == 0:
        print(f"[SKIP] No rows found for model: {model_name}")
        continue

    heat = sub.pivot_table(
        index="removed_features",
        columns="lead_h",
        values=METRIC_COL,
        aggfunc="mean"
    )

    heat = heat.reindex(index=PREDICTOR_ORDER, columns=LEAD_ORDER)

    vals = heat.values.astype(float)

    # robust color scaling centered at zero
    finite_vals = vals[np.isfinite(vals)]
    if len(finite_vals) == 0:
        print(f"[SKIP] No finite values for model: {model_name}")
        continue

    vmax = np.nanmax(np.abs(finite_vals))
    vmin = -vmax
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(vals, cmap="RdBu_r", norm=norm, aspect="auto")

    # ticks
    ax.set_xticks(np.arange(len(LEAD_ORDER)))
    ax.set_xticklabels([f"{l} h" for l in LEAD_ORDER], fontsize=11)

    ax.set_yticks(np.arange(len(PREDICTOR_ORDER)))
    ax.set_yticklabels(PREDICTOR_ORDER, fontsize=10)

    # annotate each cell
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]
            if np.isfinite(v):
                txt_color = "white" if abs(v) > 0.5 * vmax else "black"
                ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                        fontsize=9, color=txt_color)
            else:
                ax.text(j, i, "NA", ha="center", va="center",
                        fontsize=8, color="black")

    # title
    ax.set_title(
        f"{TARGET_DATASET} {TARGET_LABEL} | {model_name}\n"
        f"Feature ablation heatmap ({TITLE_METRIC})",
        fontsize=14
    )

    ax.set_xlabel("Lead time", fontsize=12)
    ax.set_ylabel("Removed predictor", fontsize=12)

    # minor grid to separate cells
    ax.set_xticks(np.arange(-0.5, len(LEAD_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(PREDICTOR_ORDER), 1), minor=True)
    ax.grid(which="minor", color="gray", linestyle="-", linewidth=0.5, alpha=0.4)
    ax.tick_params(which="minor", bottom=False, left=False)

    # colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(TITLE_METRIC, fontsize=11)

    plt.tight_layout()

    safe_model = model_name.replace(" ", "_").replace("/", "_")
    out_png = ABLATION_DIR / f"feature_ablation_heatmap_{TARGET_DATASET}_{TARGET_LABEL}_{safe_model}_{METRIC_COL}.png"
    plt.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)
    print()

In [ ]:
# ============================================================
# FEATURE ABLATION HEATMAP — AUTO-DETECT VERSION
# Rows   = predictors
# Cols   = lead times
# Values = F1 drop after removing each feature
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------
try:
    ABLATION_DIR
except NameError:
    ABLATION_DIR = OUT_DIR / "feature_ablation"

CSV_FP = ABLATION_DIR / "feature_ablation_impact_vs_baseline.csv"

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
MODELS_TO_PLOT = [
    "AdaBoost",
    "Random Forest",
    "SVM",
    "Logistic Regression",
    "MLP",
]

METRIC_COL = "drop_f1_default"
TITLE_METRIC = "F1 drop"

PREDICTOR_ORDER = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
    "month",
    "lat",
    "lon",
]

LEAD_ORDER = [6, 12, 24, 48]

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
df0 = pd.read_csv(CSV_FP)

print("CSV loaded:", CSV_FP)
print("Total rows before filtering:", len(df0))

print("\nAvailable datasets:")
print(sorted(df0["dataset"].dropna().astype(str).unique()) if "dataset" in df0.columns else "No dataset column")

print("\nAvailable labels:")
print(sorted(df0["label"].dropna().astype(str).unique()) if "label" in df0.columns else "No label column")

print("\nAvailable models:")
print(sorted(df0["model"].dropna().astype(str).unique()))

print("\nAvailable ablation types:")
print(sorted(df0["ablation_type"].dropna().astype(str).unique()))

# ------------------------------------------------------------
# Keep only individual feature ablation rows
# ------------------------------------------------------------
df = df0[df0["ablation_type"] == "leave_one_feature_out"].copy()

# ------------------------------------------------------------
# Auto-detect dataset/label from current notebook variables
# If mismatch occurs, fallback to all available rows.
# ------------------------------------------------------------
TARGET_DATASET = str(DATASET_NAME) if "DATASET_NAME" in globals() else None
TARGET_LABEL = str(LABEL_NAME) if "LABEL_NAME" in globals() else None

print("\nRequested from notebook variables:")
print("TARGET_DATASET:", TARGET_DATASET)
print("TARGET_LABEL:", TARGET_LABEL)

# Dataset filter
if TARGET_DATASET is not None and "dataset" in df.columns:
    df_try = df[df["dataset"].astype(str).str.lower() == TARGET_DATASET.lower()].copy()

    if len(df_try) > 0:
        df = df_try
        DATASET_FOR_TITLE = TARGET_DATASET
    else:
        print(f"\n[WARNING] No rows matched dataset={TARGET_DATASET}. Using all datasets instead.")
        DATASET_FOR_TITLE = "ALL_DATASETS"
else:
    DATASET_FOR_TITLE = "ALL_DATASETS"

# Label filter
if TARGET_LABEL is not None and "label" in df.columns:
    df_try = df[df["label"].astype(str).str.lower() == TARGET_LABEL.lower()].copy()

    if len(df_try) > 0:
        df = df_try
        LABEL_FOR_TITLE = TARGET_LABEL
    else:
        print(f"\n[WARNING] No rows matched label={TARGET_LABEL}. Using all labels instead.")
        LABEL_FOR_TITLE = "ALL_LABELS"
else:
    LABEL_FOR_TITLE = "ALL_LABELS"

print("\nRows after filtering:", len(df))
print("Models available after filtering:", sorted(df["model"].dropna().astype(str).unique()))
print("Leads available after filtering:", sorted(df["lead_h"].dropna().unique()))

if len(df) == 0:
    raise ValueError(
        "No rows available for heatmap. Check dataset/label names in "
        "feature_ablation_impact_vs_baseline.csv."
    )

# ------------------------------------------------------------
# MAKE HEATMAP FOR EACH MODEL
# ------------------------------------------------------------
for model_name in MODELS_TO_PLOT:
    sub = df[df["model"] == model_name].copy()

    if len(sub) == 0:
        print(f"[SKIP] No rows found for model: {model_name}")
        continue

    heat = sub.pivot_table(
        index="removed_features",
        columns="lead_h",
        values=METRIC_COL,
        aggfunc="mean"
    )

    heat = heat.reindex(index=PREDICTOR_ORDER, columns=LEAD_ORDER)

    vals = heat.values.astype(float)

    finite_vals = vals[np.isfinite(vals)]
    if len(finite_vals) == 0:
        print(f"[SKIP] No finite values for model: {model_name}")
        continue

    vmax = np.nanmax(np.abs(finite_vals))

    if vmax == 0:
        vmax = 1e-6

    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(8, 7))

    im = ax.imshow(
        vals,
        cmap="RdBu_r",
        norm=norm,
        aspect="auto"
    )

    ax.set_xticks(np.arange(len(LEAD_ORDER)))
    ax.set_xticklabels([f"{l} h" for l in LEAD_ORDER], fontsize=11)

    ax.set_yticks(np.arange(len(PREDICTOR_ORDER)))
    ax.set_yticklabels(PREDICTOR_ORDER, fontsize=10)

    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]

            if np.isfinite(v):
                txt_color = "white" if abs(v) > 0.5 * vmax else "black"
                ax.text(
                    j, i, f"{v:.3f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=txt_color
                )
            else:
                ax.text(
                    j, i, "NA",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="black"
                )

    ax.set_title(
        f"{DATASET_FOR_TITLE.upper()} {LABEL_FOR_TITLE} | {model_name}\n"
        f"Feature ablation heatmap ({TITLE_METRIC})",
        fontsize=14
    )

    ax.set_xlabel("Lead time", fontsize=12)
    ax.set_ylabel("Removed predictor", fontsize=12)

    ax.set_xticks(np.arange(-0.5, len(LEAD_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(PREDICTOR_ORDER), 1), minor=True)
    ax.grid(which="minor", color="gray", linestyle="-", linewidth=0.5, alpha=0.4)
    ax.tick_params(which="minor", bottom=False, left=False)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(TITLE_METRIC, fontsize=11)

    plt.tight_layout()

    safe_model = model_name.replace(" ", "_").replace("/", "_")
    safe_dataset = DATASET_FOR_TITLE.replace(" ", "_").replace("/", "_")
    safe_label = LABEL_FOR_TITLE.replace(" ", "_").replace("/", "_")

    out_png = ABLATION_DIR / f"feature_ablation_heatmap_{safe_dataset}_{safe_label}_{safe_model}_{METRIC_COL}.png"
    out_csv = ABLATION_DIR / f"feature_ablation_heatmap_{safe_dataset}_{safe_label}_{safe_model}_{METRIC_COL}.csv"

    plt.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.show()

    heat.to_csv(out_csv)

    print("Saved:", out_png)
    print("Saved:", out_csv)
    print()

In [ ]:
# ============================================================
# FEATURE ABLATION HEATMAP — AUTO-DETECT VERSION
# Rows   = predictors
# Cols   = lead times
# Values = F1 drop after removing each feature
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------
try:
    ABLATION_DIR
except NameError:
    ABLATION_DIR = OUT_DIR / "feature_ablation"

CSV_FP = ABLATION_DIR / "feature_ablation_impact_vs_baseline.csv"

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
MODELS_TO_PLOT = [
    "Naive Bayes",
]

METRIC_COL = "drop_f1_default"
TITLE_METRIC = "F1 drop"

PREDICTOR_ORDER = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
    "month",
    "lat",
    "lon",
]

LEAD_ORDER = [6, 12, 24, 48]

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
df0 = pd.read_csv(CSV_FP)

print("CSV loaded:", CSV_FP)
print("Total rows before filtering:", len(df0))

print("\nAvailable datasets:")
print(sorted(df0["dataset"].dropna().astype(str).unique()) if "dataset" in df0.columns else "No dataset column")

print("\nAvailable labels:")
print(sorted(df0["label"].dropna().astype(str).unique()) if "label" in df0.columns else "No label column")

print("\nAvailable models:")
print(sorted(df0["model"].dropna().astype(str).unique()))

print("\nAvailable ablation types:")
print(sorted(df0["ablation_type"].dropna().astype(str).unique()))

# ------------------------------------------------------------
# Keep only individual feature ablation rows
# ------------------------------------------------------------
df = df0[df0["ablation_type"] == "leave_one_feature_out"].copy()

# ------------------------------------------------------------
# Auto-detect dataset/label from current notebook variables
# If mismatch occurs, fallback to all available rows.
# ------------------------------------------------------------
TARGET_DATASET = str(DATASET_NAME) if "DATASET_NAME" in globals() else None
TARGET_LABEL = str(LABEL_NAME) if "LABEL_NAME" in globals() else None

print("\nRequested from notebook variables:")
print("TARGET_DATASET:", TARGET_DATASET)
print("TARGET_LABEL:", TARGET_LABEL)

# Dataset filter
if TARGET_DATASET is not None and "dataset" in df.columns:
    df_try = df[df["dataset"].astype(str).str.lower() == TARGET_DATASET.lower()].copy()

    if len(df_try) > 0:
        df = df_try
        DATASET_FOR_TITLE = TARGET_DATASET
    else:
        print(f"\n[WARNING] No rows matched dataset={TARGET_DATASET}. Using all datasets instead.")
        DATASET_FOR_TITLE = "ALL_DATASETS"
else:
    DATASET_FOR_TITLE = "ALL_DATASETS"

# Label filter
if TARGET_LABEL is not None and "label" in df.columns:
    df_try = df[df["label"].astype(str).str.lower() == TARGET_LABEL.lower()].copy()

    if len(df_try) > 0:
        df = df_try
        LABEL_FOR_TITLE = TARGET_LABEL
    else:
        print(f"\n[WARNING] No rows matched label={TARGET_LABEL}. Using all labels instead.")
        LABEL_FOR_TITLE = "ALL_LABELS"
else:
    LABEL_FOR_TITLE = "ALL_LABELS"

print("\nRows after filtering:", len(df))
print("Models available after filtering:", sorted(df["model"].dropna().astype(str).unique()))
print("Leads available after filtering:", sorted(df["lead_h"].dropna().unique()))

if len(df) == 0:
    raise ValueError(
        "No rows available for heatmap. Check dataset/label names in "
        "feature_ablation_impact_vs_baseline.csv."
    )

# ------------------------------------------------------------
# MAKE HEATMAP FOR EACH MODEL
# ------------------------------------------------------------
for model_name in MODELS_TO_PLOT:
    sub = df[df["model"] == model_name].copy()

    if len(sub) == 0:
        print(f"[SKIP] No rows found for model: {model_name}")
        continue

    heat = sub.pivot_table(
        index="removed_features",
        columns="lead_h",
        values=METRIC_COL,
        aggfunc="mean"
    )

    heat = heat.reindex(index=PREDICTOR_ORDER, columns=LEAD_ORDER)

    vals = heat.values.astype(float)

    finite_vals = vals[np.isfinite(vals)]
    if len(finite_vals) == 0:
        print(f"[SKIP] No finite values for model: {model_name}")
        continue

    vmax = np.nanmax(np.abs(finite_vals))

    if vmax == 0:
        vmax = 1e-6

    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(8, 7))

    im = ax.imshow(
        vals,
        cmap="RdBu_r",
        norm=norm,
        aspect="auto"
    )

    ax.set_xticks(np.arange(len(LEAD_ORDER)))
    ax.set_xticklabels([f"{l} h" for l in LEAD_ORDER], fontsize=11)

    ax.set_yticks(np.arange(len(PREDICTOR_ORDER)))
    ax.set_yticklabels(PREDICTOR_ORDER, fontsize=10)

    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]

            if np.isfinite(v):
                txt_color = "white" if abs(v) > 0.5 * vmax else "black"
                ax.text(
                    j, i, f"{v:.3f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=txt_color
                )
            else:
                ax.text(
                    j, i, "NA",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="black"
                )

    ax.set_title(
        f"{DATASET_FOR_TITLE.upper()} {LABEL_FOR_TITLE} | {model_name}\n"
        f"Feature ablation heatmap ({TITLE_METRIC})",
        fontsize=14
    )

    ax.set_xlabel("Lead time", fontsize=12)
    ax.set_ylabel("Removed predictor", fontsize=12)

    ax.set_xticks(np.arange(-0.5, len(LEAD_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(PREDICTOR_ORDER), 1), minor=True)
    ax.grid(which="minor", color="gray", linestyle="-", linewidth=0.5, alpha=0.4)
    ax.tick_params(which="minor", bottom=False, left=False)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(TITLE_METRIC, fontsize=11)

    plt.tight_layout()

    safe_model = model_name.replace(" ", "_").replace("/", "_")
    safe_dataset = DATASET_FOR_TITLE.replace(" ", "_").replace("/", "_")
    safe_label = LABEL_FOR_TITLE.replace(" ", "_").replace("/", "_")

    out_png = ABLATION_DIR / f"feature_ablation_heatmap_{safe_dataset}_{safe_label}_{safe_model}_{METRIC_COL}.png"
    out_csv = ABLATION_DIR / f"feature_ablation_heatmap_{safe_dataset}_{safe_label}_{safe_model}_{METRIC_COL}.csv"

    plt.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.show()

    heat.to_csv(out_csv)

    print("Saved:", out_png)
    print("Saved:", out_csv)
    print()

In [ ]:
# ============================================================
# ONE RUNNABLE CELL — FEATURE ABLATION + SHAP HEATMAP
# OLD and NEW datasets
# Excludes lon, lat, month
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

# ============================================================
# CONFIG
# ============================================================

LAB_DIR = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_CASE_DIR = LAB_DIR / "mcs_output" / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED"
NEW_CASE_DIR = LAB_DIR / "mcs_output_softtest" / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED"

OLD_ML_DIR = LAB_DIR / "mcs_output" / "ml_old_box2deg_2009_2025_final"
NEW_ML_DIR = LAB_DIR / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED"

OUT_DIR = LAB_DIR / "feature_ablation_shap_no_lon_lat_month"
ABLATION_DIR = OUT_DIR / "feature_ablation"
SHAP_DIR = OUT_DIR / "shap_heatmaps"

OUT_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
SHAP_DIR.mkdir(parents=True, exist_ok=True)

LEADS = [6, 12, 24, 48]
LABEL_NAME = "box2deg"
RANDOM_STATE = 42
N_SPLITS = 5

RUN_ABLATION = True
RUN_SHAP = True

SHAP_BACKGROUND_N = 80
SHAP_EXPLAIN_N = 300

FEATURE_COLS_NO_GEO = [
    "area_km2", "tb_mean", "tb_min",
    "speed_kmh", "direction_deg",
    "eta850_abs", "rh600", "wshear_200_850",
    "pi_vmax_ms", "gpi"
]

# ============================================================
# MODEL HELPERS
# ============================================================

def make_model(model_name, random_state=42):
    name = str(model_name).lower().replace("_", " ").replace("-", " ").strip()

    if name in ["logistic regression", "logistic"]:
        return LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=random_state
        )

    if name in ["naive bayes", "naivebayes", "gaussiannb", "nb"]:
        return GaussianNB()

    if name in ["knn", "k nearest neighbors"]:
        return KNeighborsClassifier(n_neighbors=5)

    if name in ["svm", "support vector machine"]:
        return SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=random_state
        )

    if name in ["decision tree", "decisiontree"]:
        return DecisionTreeClassifier(
            class_weight="balanced",
            random_state=random_state
        )

    if name in ["mlp"]:
        return MLPClassifier(
            hidden_layer_sizes=(50,),
            max_iter=1000,
            random_state=random_state
        )

    if name in ["qda"]:
        return QuadraticDiscriminantAnalysis()

    if name in ["random forest", "randomforest"]:
        return RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1
        )

    if name in ["adaboost", "ada boost", "ada"]:
        return AdaBoostClassifier(
            n_estimators=300,
            random_state=random_state
        )

    raise ValueError(f"Unknown model name: {model_name}")


def needs_scaler(model_name):
    name = str(model_name).lower().replace("_", " ").replace("-", " ").strip()
    return name in [
        "logistic regression", "logistic",
        "knn", "k nearest neighbors",
        "svm", "support vector machine",
        "mlp",
        "qda",
        "naive bayes", "naivebayes", "gaussiannb", "nb"
    ]


def make_pipeline_for_model(model_name, model):
    steps = [("imputer", SimpleImputer(strategy="median"))]

    if needs_scaler(model_name):
        steps.append(("scaler", StandardScaler()))

    steps.append(("clf", model))

    return Pipeline(steps)


def safe_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_score)
    except Exception:
        out["roc_auc"] = np.nan

    try:
        out["pr_auc"] = average_precision_score(y_true, y_score)
    except Exception:
        out["pr_auc"] = np.nan

    return out


# ============================================================
# FILE LOADING HELPERS
# ============================================================

def find_ranked_file(ml_dir, dataset_name):
    exact_candidates = [
        ml_dir / f"model_performance_ranked_by_lead_{dataset_name}.csv",
        ml_dir / f"model_performance_ranked_by_lead_{dataset_name}_all_models.csv",
        ml_dir / "model_performance_ranked_by_lead.csv",
        ml_dir / "model_performance_ranked_by_lead_all_models.csv",
    ]

    for fp in exact_candidates:
        if fp.exists():
            return fp

    glob_candidates = list(ml_dir.glob("model_performance_ranked_by_lead*.csv"))
    if len(glob_candidates) > 0:
        return glob_candidates[0]

    raise FileNotFoundError(
        f"No ranked performance file found in:\n{ml_dir}"
    )


def load_best_models(dataset_name):
    if dataset_name == "old":
        ranked_fp = find_ranked_file(OLD_ML_DIR, dataset_name)
    elif dataset_name == "new":
        ranked_fp = find_ranked_file(NEW_ML_DIR, dataset_name)
    else:
        raise ValueError("dataset_name must be 'old' or 'new'")

    ranked = pd.read_csv(ranked_fp)

    required = ["lead_h", "model", "f1"]
    missing = [c for c in required if c not in ranked.columns]
    if missing:
        raise ValueError(f"Ranked file missing columns {missing}: {ranked_fp}")

    best = (
        ranked
        .sort_values(["lead_h", "f1"], ascending=[True, False])
        .groupby("lead_h", as_index=False)
        .first()
    )

    model_map = {
        int(r["lead_h"]): r["model"]
        for _, r in best.iterrows()
    }

    print(f"\nBest models for {dataset_name.upper()}:")
    print(f"Source: {ranked_fp}")
    print(model_map)

    return model_map


def find_case_file(case_dir, dataset_name, lead_h):
    exact_candidates = [
        case_dir / f"{LABEL_NAME}_case_table_{dataset_name}_{lead_h}h.csv",
        case_dir / f"box2deg_case_table_{dataset_name}_{lead_h}h.csv",
        case_dir / f"zhang_case_table_{dataset_name}_{lead_h}h.csv",
        case_dir / f"case_table_{dataset_name}_{lead_h}h.csv",
    ]

    for fp in exact_candidates:
        if fp.exists():
            return fp

    patterns = [
        f"*{dataset_name}*{lead_h}h*.csv",
        f"*{lead_h}h*.csv",
    ]

    for pat in patterns:
        found = list(case_dir.glob(pat))
        if len(found) > 0:
            return found[0]

    raise FileNotFoundError(
        f"No case table found for {dataset_name}, lead {lead_h}h in:\n{case_dir}"
    )


def standardize_label_column(df, lead_h):
    if "y" in df.columns:
        return df

    candidates = [
        f"y_{lead_h}h",
        f"y_box_{lead_h}h",
        f"zhang_y_{lead_h}h",
        f"box2deg_y_{lead_h}h",
        f"label_{lead_h}h",
    ]

    for c in candidates:
        if c in df.columns:
            df["y"] = df[c]
            return df

    raise ValueError(
        f"No label column found. Expected 'y' or one of: {candidates}"
    )


def load_case_table(dataset_name, lead_h):
    if dataset_name == "old":
        case_dir = OLD_CASE_DIR
    elif dataset_name == "new":
        case_dir = NEW_CASE_DIR
    else:
        raise ValueError("dataset_name must be 'old' or 'new'")

    fp = find_case_file(case_dir, dataset_name, lead_h)
    df = pd.read_csv(fp, low_memory=False)

    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

    df = standardize_label_column(df, lead_h)

    missing_features = [c for c in FEATURE_COLS_NO_GEO if c not in df.columns]
    if missing_features:
        raise ValueError(
            f"{dataset_name.upper()} {lead_h}h missing feature columns:\n"
            f"{missing_features}\n\nFile used:\n{fp}"
        )

    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    df = df.dropna(subset=["y"]).copy()
    df["y"] = df["y"].astype(int)

    print(f"Loaded {dataset_name.upper()} {lead_h}h: {fp.name}")

    return df


# ============================================================
# CV SCORING
# ============================================================

def get_oof_score_cv(df, feature_cols, model_name, random_state=42, n_splits=5):
    df = df.dropna(subset=["y"]).copy()

    X = df[feature_cols].copy()
    y = df["y"].astype(int).values

    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))

    if n_pos < n_splits or n_neg < n_splits:
        raise ValueError(
            f"Not enough samples for {n_splits}-fold CV. "
            f"Positive={n_pos}, Negative={n_neg}"
        )

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    y_score_oof = np.full(len(df), np.nan)

    for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X, y), start=1):
        X_train = X.iloc[tr_idx]
        X_test = X.iloc[te_idx]
        y_train = y[tr_idx]

        model = make_model(model_name, random_state=random_state)
        pipe = make_pipeline_for_model(model_name, model)

        pipe.fit(X_train, y_train)

        if hasattr(pipe, "predict_proba"):
            y_score = pipe.predict_proba(X_test)[:, 1]
        else:
            raw = pipe.decision_function(X_test)
            y_score = (raw - np.nanmin(raw)) / (np.nanmax(raw) - np.nanmin(raw) + 1e-12)

        y_score_oof[te_idx] = y_score

    return safe_metrics(y, y_score_oof, threshold=0.5)


# ============================================================
# LOAD BEST MODELS
# ============================================================

OLD_BEST_MODELS = load_best_models("old")
NEW_BEST_MODELS = load_best_models("new")


# ============================================================
# FEATURE ABLATION
# ============================================================

ablation_df = None

if RUN_ABLATION:
    ablation_rows = []

    for dataset_name, best_models in [
        ("old", OLD_BEST_MODELS),
        ("new", NEW_BEST_MODELS),
    ]:
        print("\n" + "=" * 90)
        print(f"FEATURE ABLATION | DATASET = {dataset_name.upper()}")
        print("=" * 90)

        for lead_h in LEADS:
            df = load_case_table(dataset_name, lead_h)
            model_name = best_models[lead_h]

            print(
                f"\n{dataset_name.upper()} | Lead {lead_h}h | "
                f"Model = {model_name} | rows={len(df)} | positives={int(df['y'].sum())}"
            )

            base_metrics = get_oof_score_cv(
                df=df,
                feature_cols=FEATURE_COLS_NO_GEO,
                model_name=model_name,
                random_state=RANDOM_STATE,
                n_splits=N_SPLITS
            )

            base_f1 = base_metrics["f1"]
            base_pr_auc = base_metrics["pr_auc"]
            base_roc_auc = base_metrics["roc_auc"]

            print(f"Baseline F1={base_f1:.4f} | PR-AUC={base_pr_auc:.4f}")

            for removed_feature in FEATURE_COLS_NO_GEO:
                reduced_features = [
                    c for c in FEATURE_COLS_NO_GEO
                    if c != removed_feature
                ]

                ab_metrics = get_oof_score_cv(
                    df=df,
                    feature_cols=reduced_features,
                    model_name=model_name,
                    random_state=RANDOM_STATE,
                    n_splits=N_SPLITS
                )

                row = {
                    "dataset": dataset_name,
                    "lead_h": lead_h,
                    "model": model_name,
                    "removed_feature": removed_feature,

                    "baseline_f1": base_f1,
                    "ablated_f1": ab_metrics["f1"],
                    "delta_f1": base_f1 - ab_metrics["f1"],

                    "baseline_pr_auc": base_pr_auc,
                    "ablated_pr_auc": ab_metrics["pr_auc"],
                    "delta_pr_auc": base_pr_auc - ab_metrics["pr_auc"],

                    "baseline_roc_auc": base_roc_auc,
                    "ablated_roc_auc": ab_metrics["roc_auc"],
                    "delta_roc_auc": base_roc_auc - ab_metrics["roc_auc"],
                }

                ablation_rows.append(row)

                print(
                    f"  remove {removed_feature:18s} | "
                    f"F1={ab_metrics['f1']:.4f} | "
                    f"delta_F1={row['delta_f1']:+.4f}"
                )

    ablation_df = pd.DataFrame(ablation_rows)

    out_csv = ABLATION_DIR / "feature_ablation_old_new_no_lon_lat_month.csv"
    ablation_df.to_csv(out_csv, index=False)

    print("\nSaved ablation table:")
    print(out_csv)

    display(ablation_df.head())


# ============================================================
# FEATURE ABLATION HEATMAP
# ============================================================

def plot_ablation_heatmap(ablation_df, value_col="delta_f1"):
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)

    all_values = ablation_df[value_col].values
    vmax = np.nanmax(np.abs(all_values))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 0.01

    for ax, dataset_name in zip(axes, ["old", "new"]):
        sub = ablation_df[ablation_df["dataset"] == dataset_name].copy()

        pivot = sub.pivot_table(
            index="removed_feature",
            columns="lead_h",
            values=value_col,
            aggfunc="mean"
        )

        pivot = pivot.reindex(FEATURE_COLS_NO_GEO)
        pivot = pivot[LEADS]

        im = ax.imshow(
            pivot.values,
            aspect="auto",
            cmap="RdBu_r",
            vmin=-vmax,
            vmax=vmax
        )

        ax.set_title(
            f"{dataset_name.upper()} feature ablation\n"
            "value = baseline F1 − ablated F1"
        )
        ax.set_xticks(np.arange(len(pivot.columns)))
        ax.set_xticklabels([f"{c}h" for c in pivot.columns])
        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_yticklabels(pivot.index)

        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.values[i, j]
                if np.isfinite(val):
                    ax.text(
                        j, i,
                        f"{val:+.3f}",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color="black"
                    )

        ax.set_xlabel("Lead time")
        ax.set_ylabel("Removed feature")

    cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85)
    cbar.set_label("Delta F1")

    out_png = ABLATION_DIR / "feature_ablation_heatmap_old_new_delta_f1_no_lon_lat_month.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved ablation heatmap:")
    print(out_png)


if RUN_ABLATION:
    plot_ablation_heatmap(ablation_df, value_col="delta_f1")


# ============================================================
# SHAP IMPORTANCE
# ============================================================

def balanced_sample_for_shap(df, n_total=300, random_state=42):
    pos = df[df["y"].astype(int) == 1].copy()
    neg = df[df["y"].astype(int) == 0].copy()

    n_pos = min(len(pos), n_total // 2)
    n_neg = min(len(neg), n_total - n_pos)

    parts = []

    if n_pos > 0:
        parts.append(pos.sample(n=n_pos, random_state=random_state))

    if n_neg > 0:
        parts.append(neg.sample(n=n_neg, random_state=random_state))

    out = pd.concat(parts, ignore_index=True)
    out = out.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return out


def compute_shap_importance_for_case(
    df,
    feature_cols,
    model_name,
    random_state=42,
    background_n=80,
    explain_n=300,
):
    import shap

    df = df.dropna(subset=["y"]).copy()

    X = df[feature_cols].copy()
    y = df["y"].astype(int).values

    model = make_model(model_name, random_state=random_state)
    pipe = make_pipeline_for_model(model_name, model)

    pipe.fit(X, y)

    bg_df = X.sample(
        n=min(background_n, len(X)),
        random_state=random_state
    ).copy()

    explain_df_full = balanced_sample_for_shap(
        df,
        n_total=min(explain_n, len(df)),
        random_state=random_state
    )

    X_explain = explain_df_full[feature_cols].copy()

    def predict_positive_class(X_array):
        X_array = np.asarray(X_array)
        if X_array.ndim == 1:
            X_array = X_array.reshape(1, -1)

        X_df = pd.DataFrame(X_array, columns=feature_cols)
        return pipe.predict_proba(X_df)[:, 1]

    explainer = shap.Explainer(
        predict_positive_class,
        bg_df.values,
        algorithm="permutation"
    )

    shap_values = explainer(
        X_explain.values,
        max_evals=2 * len(feature_cols) + 1
    )

    mean_abs = np.abs(shap_values.values).mean(axis=0)

    imp = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": mean_abs
    }).sort_values("mean_abs_shap", ascending=False)

    return imp, shap_values, X_explain


shap_importance_df = None

if RUN_SHAP:
    try:
        import shap
    except ImportError:
        raise ImportError(
            "SHAP is not installed. Run this first in another cell:\n"
            "!pip install shap"
        )

    shap_rows = []

    for dataset_name, best_models in [
        ("old", OLD_BEST_MODELS),
        ("new", NEW_BEST_MODELS),
    ]:
        print("\n" + "=" * 90)
        print(f"SHAP | DATASET = {dataset_name.upper()}")
        print("=" * 90)

        for lead_h in LEADS:
            df = load_case_table(dataset_name, lead_h)
            model_name = best_models[lead_h]

            print(
                f"\n{dataset_name.upper()} | Lead {lead_h}h | "
                f"Model = {model_name} | rows={len(df)} | positives={int(df['y'].sum())}"
            )

            imp, shap_values, X_explain = compute_shap_importance_for_case(
                df=df,
                feature_cols=FEATURE_COLS_NO_GEO,
                model_name=model_name,
                random_state=RANDOM_STATE + lead_h,
                background_n=SHAP_BACKGROUND_N,
                explain_n=SHAP_EXPLAIN_N
            )

            for _, r in imp.iterrows():
                shap_rows.append({
                    "dataset": dataset_name,
                    "lead_h": lead_h,
                    "model": model_name,
                    "feature": r["feature"],
                    "mean_abs_shap": r["mean_abs_shap"],
                })

            out_imp = SHAP_DIR / f"shap_importance_{dataset_name}_{lead_h}h_no_lon_lat_month.csv"
            imp.to_csv(out_imp, index=False)

            print("Top SHAP features:")
            display(imp.head(5))

    shap_importance_df = pd.DataFrame(shap_rows)

    out_csv = SHAP_DIR / "shap_importance_old_new_no_lon_lat_month.csv"
    shap_importance_df.to_csv(out_csv, index=False)

    print("\nSaved SHAP importance table:")
    print(out_csv)


# ============================================================
# SHAP HEATMAP
# ============================================================

def plot_shap_heatmap(shap_importance_df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)

    vmax_global = shap_importance_df["mean_abs_shap"].max()
    if not np.isfinite(vmax_global) or vmax_global == 0:
        vmax_global = 0.01

    for ax, dataset_name in zip(axes, ["old", "new"]):
        sub = shap_importance_df[shap_importance_df["dataset"] == dataset_name].copy()

        pivot = sub.pivot_table(
            index="feature",
            columns="lead_h",
            values="mean_abs_shap",
            aggfunc="mean"
        )

        pivot = pivot.reindex(FEATURE_COLS_NO_GEO)
        pivot = pivot[LEADS]

        im = ax.imshow(
            pivot.values,
            aspect="auto",
            cmap="viridis",
            vmin=0,
            vmax=vmax_global
        )

        ax.set_title(f"{dataset_name.upper()} SHAP heatmap\nmean |SHAP value|")
        ax.set_xticks(np.arange(len(pivot.columns)))
        ax.set_xticklabels([f"{c}h" for c in pivot.columns])
        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_yticklabels(pivot.index)

        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.values[i, j]
                if np.isfinite(val):
                    ax.text(
                        j, i,
                        f"{val:.3f}",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color="white" if val > vmax_global * 0.45 else "black"
                    )

        ax.set_xlabel("Lead time")
        ax.set_ylabel("Feature")

    cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85)
    cbar.set_label("Mean absolute SHAP value")

    out_png = SHAP_DIR / "shap_mean_abs_heatmap_old_new_no_lon_lat_month.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved SHAP heatmap:")
    print(out_png)


if RUN_SHAP:
    plot_shap_heatmap(shap_importance_df)


print("\nDONE.")
print("Output folder:")
print(OUT_DIR)

In [ ]:
# ============================================================
# COMBINED HEATMAPS
# Feature Ablation + SHAP side-by-side
# OLD and NEW
# ============================================================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Fallback output directory if not already defined
# ------------------------------------------------------------
try:
    OUT_DIR
except NameError:
    OUT_DIR = Path.cwd() / "feature_ablation_shap_no_lon_lat_month"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_DIR = OUT_DIR / "combined_ablation_shap_heatmaps"
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------
required_ablation_cols = ["dataset", "lead_h", "removed_feature", "delta_f1"]
required_shap_cols = ["dataset", "lead_h", "feature", "mean_abs_shap"]

missing_ablation = [c for c in required_ablation_cols if c not in ablation_df.columns]
missing_shap = [c for c in required_shap_cols if c not in shap_importance_df.columns]

if missing_ablation:
    raise ValueError(f"ablation_df is missing columns: {missing_ablation}")

if missing_shap:
    raise ValueError(f"shap_importance_df is missing columns: {missing_shap}")

# ------------------------------------------------------------
# Helper: make pivot tables
# ------------------------------------------------------------
def make_ablation_pivot(dataset_name):
    sub = ablation_df[ablation_df["dataset"] == dataset_name].copy()

    pivot = sub.pivot_table(
        index="removed_feature",
        columns="lead_h",
        values="delta_f1",
        aggfunc="mean"
    )

    pivot = pivot.reindex(FEATURE_COLS_NO_GEO)
    pivot = pivot[LEADS]

    return pivot


def make_shap_pivot(dataset_name):
    sub = shap_importance_df[shap_importance_df["dataset"] == dataset_name].copy()

    pivot = sub.pivot_table(
        index="feature",
        columns="lead_h",
        values="mean_abs_shap",
        aggfunc="mean"
    )

    pivot = pivot.reindex(FEATURE_COLS_NO_GEO)
    pivot = pivot[LEADS]

    return pivot


# ------------------------------------------------------------
# Global color scales
# ------------------------------------------------------------
old_ablation_pivot = make_ablation_pivot("old")
new_ablation_pivot = make_ablation_pivot("new")

old_shap_pivot = make_shap_pivot("old")
new_shap_pivot = make_shap_pivot("new")

ablation_vmax = np.nanmax(
    np.abs(
        np.concatenate([
            old_ablation_pivot.values.flatten(),
            new_ablation_pivot.values.flatten()
        ])
    )
)

if not np.isfinite(ablation_vmax) or ablation_vmax == 0:
    ablation_vmax = 0.01

shap_vmax = np.nanmax(
    np.concatenate([
        old_shap_pivot.values.flatten(),
        new_shap_pivot.values.flatten()
    ])
)

if not np.isfinite(shap_vmax) or shap_vmax == 0:
    shap_vmax = 0.01


# ============================================================
# FIGURE 1 — ONE COMBINED 2x2 FIGURE
# Row 1: OLD ablation vs OLD SHAP
# Row 2: NEW ablation vs NEW SHAP
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 11),
    constrained_layout=True
)

datasets = ["old", "new"]

for row_idx, dataset_name in enumerate(datasets):

    # --------------------------------------------------------
    # Left panel: Feature ablation
    # --------------------------------------------------------
    ax_ablation = axes[row_idx, 0]
    ablation_pivot = make_ablation_pivot(dataset_name)

    im_ablation = ax_ablation.imshow(
        ablation_pivot.values,
        aspect="auto",
        cmap="RdBu_r",
        vmin=-ablation_vmax,
        vmax=ablation_vmax
    )

    ax_ablation.set_title(
        f"{dataset_name.upper()} Feature Ablation\n"
        "Delta F1 = baseline F1 − ablated F1"
    )

    ax_ablation.set_xticks(np.arange(len(LEADS)))
    ax_ablation.set_xticklabels([f"{lead}h" for lead in LEADS])

    ax_ablation.set_yticks(np.arange(len(FEATURE_COLS_NO_GEO)))
    ax_ablation.set_yticklabels(FEATURE_COLS_NO_GEO)

    ax_ablation.set_xlabel("Lead time")
    ax_ablation.set_ylabel("Removed feature")

    for i in range(ablation_pivot.shape[0]):
        for j in range(ablation_pivot.shape[1]):
            val = ablation_pivot.values[i, j]
            if np.isfinite(val):
                ax_ablation.text(
                    j, i,
                    f"{val:+.3f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="black"
                )

    # --------------------------------------------------------
    # Right panel: SHAP
    # --------------------------------------------------------
    ax_shap = axes[row_idx, 1]
    shap_pivot = make_shap_pivot(dataset_name)

    im_shap = ax_shap.imshow(
        shap_pivot.values,
        aspect="auto",
        cmap="viridis",
        vmin=0,
        vmax=shap_vmax
    )

    ax_shap.set_title(
        f"{dataset_name.upper()} SHAP Heatmap\n"
        "Mean absolute SHAP value"
    )

    ax_shap.set_xticks(np.arange(len(LEADS)))
    ax_shap.set_xticklabels([f"{lead}h" for lead in LEADS])

    ax_shap.set_yticks(np.arange(len(FEATURE_COLS_NO_GEO)))
    ax_shap.set_yticklabels(FEATURE_COLS_NO_GEO)

    ax_shap.set_xlabel("Lead time")
    ax_shap.set_ylabel("Feature")

    for i in range(shap_pivot.shape[0]):
        for j in range(shap_pivot.shape[1]):
            val = shap_pivot.values[i, j]
            if np.isfinite(val):
                ax_shap.text(
                    j, i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if val > shap_vmax * 0.45 else "black"
                )

# ------------------------------------------------------------
# Separate colorbars for different quantities
# ------------------------------------------------------------
cbar_ablation = fig.colorbar(
    im_ablation,
    ax=axes[:, 0],
    shrink=0.85
)
cbar_ablation.set_label("Delta F1")

cbar_shap = fig.colorbar(
    im_shap,
    ax=axes[:, 1],
    shrink=0.85
)
cbar_shap.set_label("Mean absolute SHAP value")

out_png = COMBINED_DIR / "combined_feature_ablation_and_shap_old_new.png"
plt.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print("Saved combined 2x2 figure:")
print(out_png)


# ============================================================
# FIGURE 2 — SEPARATE FIGURES FOR EACH DATASET
# OLD: ablation vs SHAP
# NEW: ablation vs SHAP
# ============================================================

def plot_single_dataset_ablation_vs_shap(dataset_name):
    ablation_pivot = make_ablation_pivot(dataset_name)
    shap_pivot = make_shap_pivot(dataset_name)

    fig, axes = plt.subplots(
        1, 2,
        figsize=(14, 5.8),
        constrained_layout=True
    )

    # --------------------------------------------------------
    # Feature ablation
    # --------------------------------------------------------
    ax = axes[0]

    im1 = ax.imshow(
        ablation_pivot.values,
        aspect="auto",
        cmap="RdBu_r",
        vmin=-ablation_vmax,
        vmax=ablation_vmax
    )

    ax.set_title(
        f"{dataset_name.upper()} Feature Ablation\n"
        "Delta F1 = baseline F1 − ablated F1"
    )

    ax.set_xticks(np.arange(len(LEADS)))
    ax.set_xticklabels([f"{lead}h" for lead in LEADS])

    ax.set_yticks(np.arange(len(FEATURE_COLS_NO_GEO)))
    ax.set_yticklabels(FEATURE_COLS_NO_GEO)

    ax.set_xlabel("Lead time")
    ax.set_ylabel("Removed feature")

    for i in range(ablation_pivot.shape[0]):
        for j in range(ablation_pivot.shape[1]):
            val = ablation_pivot.values[i, j]
            if np.isfinite(val):
                ax.text(
                    j, i,
                    f"{val:+.3f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="black"
                )

    cbar1 = fig.colorbar(im1, ax=ax, shrink=0.85)
    cbar1.set_label("Delta F1")

    # --------------------------------------------------------
    # SHAP
    # --------------------------------------------------------
    ax = axes[1]

    im2 = ax.imshow(
        shap_pivot.values,
        aspect="auto",
        cmap="viridis",
        vmin=0,
        vmax=shap_vmax
    )

    ax.set_title(
        f"{dataset_name.upper()} SHAP Heatmap\n"
        "Mean absolute SHAP value"
    )

    ax.set_xticks(np.arange(len(LEADS)))
    ax.set_xticklabels([f"{lead}h" for lead in LEADS])

    ax.set_yticks(np.arange(len(FEATURE_COLS_NO_GEO)))
    ax.set_yticklabels(FEATURE_COLS_NO_GEO)

    ax.set_xlabel("Lead time")
    ax.set_ylabel("Feature")

    for i in range(shap_pivot.shape[0]):
        for j in range(shap_pivot.shape[1]):
            val = shap_pivot.values[i, j]
            if np.isfinite(val):
                ax.text(
                    j, i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if val > shap_vmax * 0.45 else "black"
                )

    cbar2 = fig.colorbar(im2, ax=ax, shrink=0.85)
    cbar2.set_label("Mean absolute SHAP value")

    out_png = COMBINED_DIR / f"{dataset_name}_feature_ablation_vs_shap.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()

    print(f"Saved {dataset_name.upper()} figure:")
    print(out_png)


plot_single_dataset_ablation_vs_shap("old")
plot_single_dataset_ablation_vs_shap("new")

In [ ]:
# ============================================================
# OLD vs NEW DEFAULT-THRESHOLD BEST-MODEL PERFORMANCE
# ROC AUC and PR AUC comparison plot
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
OLD_FP = Path(
    r"C:\Users\_s2218026\Documents\lab\mcs_output\ml_old_box2deg_2009_2025_final\model_performance_ranked_by_lead_old.csv"
)

NEW_FP = Path(
    r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest\ml_new_box2deg_2009_2025_UPDATED\model_performance_ranked_by_lead_new.csv"
)

OUT_DIR = Path(
    r"C:\Users\_s2218026\Documents\lab\comparison_plots_old_new"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
if not OLD_FP.exists():
    raise FileNotFoundError(f"OLD file not found:\n{OLD_FP}")

if not NEW_FP.exists():
    raise FileNotFoundError(f"NEW file not found:\n{NEW_FP}")

old_ranked = pd.read_csv(OLD_FP)
new_ranked = pd.read_csv(NEW_FP)

print("Loaded OLD:", OLD_FP)
print("Loaded NEW:", NEW_FP)

# ------------------------------------------------------------
# CHECK REQUIRED COLUMNS
# ------------------------------------------------------------
required_cols = ["lead_h", "model", "roc_auc", "pr_auc", "f1"]

for name, df in [("OLD", old_ranked), ("NEW", new_ranked)]:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

# ------------------------------------------------------------
# BEST MODEL PER LEAD
# based on F1 from default-threshold ranked table
# ------------------------------------------------------------
old_best = (
    old_ranked
    .sort_values(["lead_h", "f1"], ascending=[True, False])
    .groupby("lead_h", as_index=False)
    .first()
)

new_best = (
    new_ranked
    .sort_values(["lead_h", "f1"], ascending=[True, False])
    .groupby("lead_h", as_index=False)
    .first()
)

print("\nOLD best model by lead:")
display(old_best[["lead_h", "model", "f1", "roc_auc", "pr_auc"]])

print("\nNEW best model by lead:")
display(new_best[["lead_h", "model", "f1", "roc_auc", "pr_auc"]])

# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

metric_list = ["roc_auc", "pr_auc"]
title_list = ["ROC AUC", "PR AUC"]

for ax, metric, ttl in zip(axes, metric_list, title_list):
    ax.plot(
        old_best["lead_h"],
        old_best[metric],
        marker="o",
        linewidth=2,
        label="OLD"
    )

    ax.plot(
        new_best["lead_h"],
        new_best[metric],
        marker="s",
        linewidth=2,
        label="NEW"
    )

    ax.set_xticks([6, 12, 24, 48])
    ax.set_xlabel("Lead time (h)")
    ax.set_ylabel(ttl)
    ax.set_title(ttl)
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle("OLD vs NEW default-threshold best-model AUC comparison", fontsize=16)

out_png = OUT_DIR / "old_vs_new_default_threshold_best_model_roc_pr_auc.png"
plt.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_png)

In [ ]:
# ============================================================
# OLD AdaBoost Feature Ablation Heatmap
# Same style as your original heatmap:
#   rows    = removed predictors
#   columns = lead time
#   values  = baseline F1 - ablated F1
#
# Default: OLD AdaBoost 6h only
# To make 6/12/24/48 columns, change LEADS = [6, 12, 24, 48]
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.ensemble import AdaBoostClassifier

# ============================================================
# CONFIG
# ============================================================

LAB_DIR = Path(r"C:\Users\_s2218026\Documents\lab")

CASE_DIR = (
    LAB_DIR
    / "mcs_output"
    / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED"
)

OUT_DIR = (
    LAB_DIR
    / "feature_ablation_old_adaboost_same_style"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "new"
LABEL_NAME = "box2deg"
MODEL_NAME = "AdaBoost"

# Use [6] for 6h only.
# Use [6, 12, 24, 48] if you want the same 4-column style as your first plot.
LEADS = [6, 12, 24, 48]

RANDOM_STATE = 42
N_SPLITS = 5

# No lon, lat, month
FEATURE_COLS = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
]

# ============================================================
# FUNCTIONS
# ============================================================

def make_adaboost(random_state=42):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", AdaBoostClassifier(
            n_estimators=300,
            random_state=random_state
        ))
    ])


def safe_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    out = {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_score)
    except Exception:
        out["roc_auc"] = np.nan

    try:
        out["pr_auc"] = average_precision_score(y_true, y_score)
    except Exception:
        out["pr_auc"] = np.nan

    return out


def threshold_scan(y_true, y_score):
    rows = []

    for thr in np.round(np.arange(0.05, 0.951, 0.01), 2):
        m = safe_metrics(y_true, y_score, threshold=thr)
        rows.append({
            "threshold": thr,
            **m,
        })

    scan = pd.DataFrame(rows)

    scan = (
        scan
        .sort_values(["f1", "recall", "precision"], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    return scan


def make_cv(y, groups=None, n_splits=5, random_state=42):
    y = np.asarray(y).astype(int)
    min_class_count = np.bincount(y).min()
    n_splits_eff = min(n_splits, int(min_class_count))

    if n_splits_eff < 2:
        raise ValueError("Not enough class samples for cross-validation.")

    if groups is not None:
        try:
            cv = StratifiedGroupKFold(
                n_splits=n_splits_eff,
                shuffle=True,
                random_state=random_state,
            )
            return cv, "StratifiedGroupKFold", n_splits_eff
        except Exception:
            pass

    cv = StratifiedKFold(
        n_splits=n_splits_eff,
        shuffle=True,
        random_state=random_state,
    )

    return cv, "StratifiedKFold", n_splits_eff


def evaluate_feature_set(df, feature_cols, random_state=42, n_splits=5):
    X = df[feature_cols].copy()
    y = df["y"].astype(int).values
    groups = df["track_id"].astype(str).values if "track_id" in df.columns else None

    cv, cv_name, n_splits_eff = make_cv(
        y=y,
        groups=groups,
        n_splits=n_splits,
        random_state=random_state,
    )

    y_score_oof = np.full(len(df), np.nan)

    splitter = cv.split(X, y, groups) if groups is not None else cv.split(X, y)

    for fold_id, (train_idx, test_idx) in enumerate(splitter, start=1):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y[train_idx]

        pipe = make_adaboost(random_state=random_state)
        pipe.fit(X_train, y_train)

        y_score_oof[test_idx] = pipe.predict_proba(X_test)[:, 1]

    default_metrics = safe_metrics(y, y_score_oof, threshold=0.5)

    scan = threshold_scan(y, y_score_oof)
    best_threshold = float(scan.loc[0, "threshold"])
    tuned_metrics = safe_metrics(y, y_score_oof, threshold=best_threshold)

    return {
        "cv": cv_name,
        "n_splits": n_splits_eff,
        "default_threshold": 0.5,
        "tuned_threshold": best_threshold,

        "default_f1": default_metrics["f1"],
        "default_precision": default_metrics["precision"],
        "default_recall": default_metrics["recall"],
        "default_roc_auc": default_metrics["roc_auc"],
        "default_pr_auc": default_metrics["pr_auc"],

        "tuned_f1": tuned_metrics["f1"],
        "tuned_precision": tuned_metrics["precision"],
        "tuned_recall": tuned_metrics["recall"],
        "tuned_roc_auc": tuned_metrics["roc_auc"],
        "tuned_pr_auc": tuned_metrics["pr_auc"],
    }


# ============================================================
# RUN FEATURE ABLATION
# ============================================================

rows = []

for lead in LEADS:
    case_fp = CASE_DIR / f"{LABEL_NAME}_case_table_{DATASET_NAME}_{lead}h.csv"

    if not case_fp.exists():
        raise FileNotFoundError(case_fp)

    df = pd.read_csv(case_fp, low_memory=False)

    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

    missing = [c for c in FEATURE_COLS + ["y"] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {case_fp.name}: {missing}")

    df = df.dropna(subset=["y"]).copy()

    print("\n" + "=" * 80)
    print(f"OLD AdaBoost | Lead {lead}h")
    print("Rows:", len(df))
    print("Positive:", int(df["y"].sum()))
    print("Negative:", int((df["y"] == 0).sum()))
    print("=" * 80)

    baseline = evaluate_feature_set(
        df=df,
        feature_cols=FEATURE_COLS,
        random_state=RANDOM_STATE,
        n_splits=N_SPLITS,
    )

    print(
        f"Baseline default F1={baseline['default_f1']:.4f} | "
        f"tuned F1={baseline['tuned_f1']:.4f} | "
        f"best threshold={baseline['tuned_threshold']:.2f}"
    )

    for removed_feature in FEATURE_COLS:
        reduced_features = [c for c in FEATURE_COLS if c != removed_feature]

        ablated = evaluate_feature_set(
            df=df,
            feature_cols=reduced_features,
            random_state=RANDOM_STATE,
            n_splits=N_SPLITS,
        )

        row = {
            "dataset": "old",
            "model": "AdaBoost",
            "lead_h": lead,
            "removed_feature": removed_feature,

            "baseline_default_f1": baseline["default_f1"],
            "ablated_default_f1": ablated["default_f1"],
            "delta_default_f1": baseline["default_f1"] - ablated["default_f1"],

            "baseline_tuned_f1": baseline["tuned_f1"],
            "ablated_tuned_f1": ablated["tuned_f1"],
            "delta_tuned_f1": baseline["tuned_f1"] - ablated["tuned_f1"],

            "baseline_default_pr_auc": baseline["default_pr_auc"],
            "ablated_default_pr_auc": ablated["default_pr_auc"],
            "delta_pr_auc": baseline["default_pr_auc"] - ablated["default_pr_auc"],

            "baseline_default_roc_auc": baseline["default_roc_auc"],
            "ablated_default_roc_auc": ablated["default_roc_auc"],
            "delta_roc_auc": baseline["default_roc_auc"] - ablated["default_roc_auc"],

            "baseline_tuned_threshold": baseline["tuned_threshold"],
            "ablated_tuned_threshold": ablated["tuned_threshold"],
        }

        rows.append(row)

        print(
            f"remove {removed_feature:18s} | "
            f"default ΔF1={row['delta_default_f1']:+.4f} | "
            f"tuned ΔF1={row['delta_tuned_f1']:+.4f}"
        )

ablation_df = pd.DataFrame(rows)

out_csv = OUT_DIR / "feature_ablation_old_adaboost_no_geo_same_style.csv"
ablation_df.to_csv(out_csv, index=False)

print("\nSaved table:")
print(out_csv)

display(ablation_df)


# ============================================================
# PLOT 1 — DEFAULT THRESHOLD F1 DROP
# Same layout as your first figure
# ============================================================

pivot_default = ablation_df.pivot_table(
    index="removed_feature",
    columns="lead_h",
    values="delta_default_f1",
    aggfunc="mean",
)

pivot_default = pivot_default.reindex(FEATURE_COLS)
pivot_default = pivot_default[LEADS]

vmax = np.nanmax(np.abs(pivot_default.values))
if not np.isfinite(vmax) or vmax == 0:
    vmax = 0.01

fig, ax = plt.subplots(figsize=(5.5 if len(LEADS) == 1 else 8.5, 7.5))

im = ax.imshow(
    pivot_default.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)

ax.set_title(
    "OLD box2deg | AdaBoost\n"
    "Feature ablation heatmap, default threshold (F1 drop)"
)

ax.set_xlabel("Lead time")
ax.set_ylabel("Removed predictor")

ax.set_xticks(np.arange(len(pivot_default.columns)))
ax.set_xticklabels([f"{c} h" for c in pivot_default.columns])

ax.set_yticks(np.arange(len(pivot_default.index)))
ax.set_yticklabels(pivot_default.index)

for i in range(pivot_default.shape[0]):
    for j in range(pivot_default.shape[1]):
        val = pivot_default.values[i, j]
        if np.isfinite(val):
            ax.text(
                j,
                i,
                f"{val:.3f}",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if abs(val) > vmax * 0.55 else "black",
            )

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("F1 drop")

plt.tight_layout()

out_png_default = OUT_DIR / "heatmap_old_adaboost_default_f1_drop_same_style.png"
plt.savefig(out_png_default, dpi=250, bbox_inches="tight")
plt.show()

print("Saved default-threshold heatmap:")
print(out_png_default)


# ============================================================
# PLOT 2 — TUNED THRESHOLD F1 DROP
# Same layout as your first figure
# ============================================================

pivot_tuned = ablation_df.pivot_table(
    index="removed_feature",
    columns="lead_h",
    values="delta_tuned_f1",
    aggfunc="mean",
)

pivot_tuned = pivot_tuned.reindex(FEATURE_COLS)
pivot_tuned = pivot_tuned[LEADS]

vmax = np.nanmax(np.abs(pivot_tuned.values))
if not np.isfinite(vmax) or vmax == 0:
    vmax = 0.01

fig, ax = plt.subplots(figsize=(5.5 if len(LEADS) == 1 else 8.5, 7.5))

im = ax.imshow(
    pivot_tuned.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)

ax.set_title(
    "OLD box2deg | AdaBoost\n"
    "Feature ablation heatmap, tuned threshold (F1 drop)"
)

ax.set_xlabel("Lead time")
ax.set_ylabel("Removed predictor")

ax.set_xticks(np.arange(len(pivot_tuned.columns)))
ax.set_xticklabels([f"{c} h" for c in pivot_tuned.columns])

ax.set_yticks(np.arange(len(pivot_tuned.index)))
ax.set_yticklabels(pivot_tuned.index)

for i in range(pivot_tuned.shape[0]):
    for j in range(pivot_tuned.shape[1]):
        val = pivot_tuned.values[i, j]
        if np.isfinite(val):
            ax.text(
                j,
                i,
                f"{val:.3f}",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if abs(val) > vmax * 0.55 else "black",
            )

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("F1 drop")

plt.tight_layout()

out_png_tuned = OUT_DIR / "heatmap_old_adaboost_tuned_f1_drop_same_style.png"
plt.savefig(out_png_tuned, dpi=250, bbox_inches="tight")
plt.show()

print("Saved tuned-threshold heatmap:")
print(out_png_tuned)

In [ ]:
# ============================================================
# NEW AdaBoost Feature Ablation Heatmap
# Same style as original heatmap:
#   rows    = removed predictors
#   columns = lead time
#   values  = baseline F1 - ablated F1
#
# Default: NEW AdaBoost 6h only
# To make 6/12/24/48 columns, change LEADS = [6, 12, 24, 48]
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.ensemble import AdaBoostClassifier

# ============================================================
# CONFIG
# ============================================================

LAB_DIR = Path(r"C:\Users\_s2218026\Documents\lab")

CASE_DIR = (
    LAB_DIR
    / "mcs_output_softtest"
    / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED"
)

OUT_DIR = (
    LAB_DIR
    / "feature_ablation_new_adaboost_same_style"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "new"
LABEL_NAME = "box2deg"
MODEL_NAME = "AdaBoost"

# Use [6] for 6h only.
# Use [6, 12, 24, 48] if you want the full 4-column style.
LEADS = [6, 12, 24, 48]

RANDOM_STATE = 42
N_SPLITS = 5

# No lon, lat, month
FEATURE_COLS = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
]

# ============================================================
# FUNCTIONS
# ============================================================

def make_adaboost(random_state=42):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", AdaBoostClassifier(
            n_estimators=300,
            random_state=random_state
        ))
    ])


def safe_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    out = {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_score)
    except Exception:
        out["roc_auc"] = np.nan

    try:
        out["pr_auc"] = average_precision_score(y_true, y_score)
    except Exception:
        out["pr_auc"] = np.nan

    return out


def threshold_scan(y_true, y_score):
    rows = []

    for thr in np.round(np.arange(0.05, 0.951, 0.01), 2):
        m = safe_metrics(y_true, y_score, threshold=thr)
        rows.append({
            "threshold": thr,
            **m,
        })

    scan = pd.DataFrame(rows)

    scan = (
        scan
        .sort_values(["f1", "recall", "precision"], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    return scan


def make_cv(y, groups=None, n_splits=5, random_state=42):
    y = np.asarray(y).astype(int)
    min_class_count = np.bincount(y).min()
    n_splits_eff = min(n_splits, int(min_class_count))

    if n_splits_eff < 2:
        raise ValueError("Not enough class samples for cross-validation.")

    if groups is not None:
        try:
            cv = StratifiedGroupKFold(
                n_splits=n_splits_eff,
                shuffle=True,
                random_state=random_state,
            )
            return cv, "StratifiedGroupKFold", n_splits_eff
        except Exception:
            pass

    cv = StratifiedKFold(
        n_splits=n_splits_eff,
        shuffle=True,
        random_state=random_state,
    )

    return cv, "StratifiedKFold", n_splits_eff


def evaluate_feature_set(df, feature_cols, random_state=42, n_splits=5):
    X = df[feature_cols].copy()
    y = df["y"].astype(int).values
    groups = df["track_id"].astype(str).values if "track_id" in df.columns else None

    cv, cv_name, n_splits_eff = make_cv(
        y=y,
        groups=groups,
        n_splits=n_splits,
        random_state=random_state,
    )

    y_score_oof = np.full(len(df), np.nan)

    splitter = cv.split(X, y, groups) if groups is not None else cv.split(X, y)

    for fold_id, (train_idx, test_idx) in enumerate(splitter, start=1):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y[train_idx]

        pipe = make_adaboost(random_state=random_state)
        pipe.fit(X_train, y_train)

        y_score_oof[test_idx] = pipe.predict_proba(X_test)[:, 1]

    default_metrics = safe_metrics(y, y_score_oof, threshold=0.5)

    scan = threshold_scan(y, y_score_oof)
    best_threshold = float(scan.loc[0, "threshold"])
    tuned_metrics = safe_metrics(y, y_score_oof, threshold=best_threshold)

    return {
        "cv": cv_name,
        "n_splits": n_splits_eff,
        "default_threshold": 0.5,
        "tuned_threshold": best_threshold,

        "default_f1": default_metrics["f1"],
        "default_precision": default_metrics["precision"],
        "default_recall": default_metrics["recall"],
        "default_roc_auc": default_metrics["roc_auc"],
        "default_pr_auc": default_metrics["pr_auc"],

        "tuned_f1": tuned_metrics["f1"],
        "tuned_precision": tuned_metrics["precision"],
        "tuned_recall": tuned_metrics["recall"],
        "tuned_roc_auc": tuned_metrics["roc_auc"],
        "tuned_pr_auc": tuned_metrics["pr_auc"],
    }


# ============================================================
# RUN FEATURE ABLATION
# ============================================================

rows = []

for lead in LEADS:
    case_fp = CASE_DIR / f"{LABEL_NAME}_case_table_{DATASET_NAME}_{lead}h.csv"

    if not case_fp.exists():
        raise FileNotFoundError(case_fp)

    df = pd.read_csv(case_fp, low_memory=False)

    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

    missing = [c for c in FEATURE_COLS + ["y"] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {case_fp.name}: {missing}")

    df = df.dropna(subset=["y"]).copy()

    print("\n" + "=" * 80)
    print(f"NEW AdaBoost | Lead {lead}h")
    print("Rows:", len(df))
    print("Positive:", int(df["y"].sum()))
    print("Negative:", int((df["y"] == 0).sum()))
    print("=" * 80)

    baseline = evaluate_feature_set(
        df=df,
        feature_cols=FEATURE_COLS,
        random_state=RANDOM_STATE,
        n_splits=N_SPLITS,
    )

    print(
        f"Baseline default F1={baseline['default_f1']:.4f} | "
        f"tuned F1={baseline['tuned_f1']:.4f} | "
        f"best threshold={baseline['tuned_threshold']:.2f}"
    )

    for removed_feature in FEATURE_COLS:
        reduced_features = [c for c in FEATURE_COLS if c != removed_feature]

        ablated = evaluate_feature_set(
            df=df,
            feature_cols=reduced_features,
            random_state=RANDOM_STATE,
            n_splits=N_SPLITS,
        )

        row = {
            "dataset": "new",
            "model": "AdaBoost",
            "lead_h": lead,
            "removed_feature": removed_feature,

            "baseline_default_f1": baseline["default_f1"],
            "ablated_default_f1": ablated["default_f1"],
            "delta_default_f1": baseline["default_f1"] - ablated["default_f1"],

            "baseline_tuned_f1": baseline["tuned_f1"],
            "ablated_tuned_f1": ablated["tuned_f1"],
            "delta_tuned_f1": baseline["tuned_f1"] - ablated["tuned_f1"],

            "baseline_default_pr_auc": baseline["default_pr_auc"],
            "ablated_default_pr_auc": ablated["default_pr_auc"],
            "delta_pr_auc": baseline["default_pr_auc"] - ablated["default_pr_auc"],

            "baseline_default_roc_auc": baseline["default_roc_auc"],
            "ablated_default_roc_auc": ablated["default_roc_auc"],
            "delta_roc_auc": baseline["default_roc_auc"] - ablated["default_roc_auc"],

            "baseline_tuned_threshold": baseline["tuned_threshold"],
            "ablated_tuned_threshold": ablated["tuned_threshold"],
        }

        rows.append(row)

        print(
            f"remove {removed_feature:18s} | "
            f"default ΔF1={row['delta_default_f1']:+.4f} | "
            f"tuned ΔF1={row['delta_tuned_f1']:+.4f}"
        )

ablation_df = pd.DataFrame(rows)

out_csv = OUT_DIR / "feature_ablation_new_adaboost_no_geo_same_style.csv"
ablation_df.to_csv(out_csv, index=False)

print("\nSaved table:")
print(out_csv)

display(ablation_df)


# ============================================================
# PLOT 1 — DEFAULT THRESHOLD F1 DROP
# Same layout as original figure
# ============================================================

pivot_default = ablation_df.pivot_table(
    index="removed_feature",
    columns="lead_h",
    values="delta_default_f1",
    aggfunc="mean",
)

pivot_default = pivot_default.reindex(FEATURE_COLS)
pivot_default = pivot_default[LEADS]

vmax = np.nanmax(np.abs(pivot_default.values))
if not np.isfinite(vmax) or vmax == 0:
    vmax = 0.01

fig, ax = plt.subplots(figsize=(5.5 if len(LEADS) == 1 else 8.5, 7.5))

im = ax.imshow(
    pivot_default.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)

ax.set_title(
    "NEW box2deg | AdaBoost\n"
    "Feature ablation heatmap, default threshold (F1 drop)"
)

ax.set_xlabel("Lead time")
ax.set_ylabel("Removed predictor")

ax.set_xticks(np.arange(len(pivot_default.columns)))
ax.set_xticklabels([f"{c} h" for c in pivot_default.columns])

ax.set_yticks(np.arange(len(pivot_default.index)))
ax.set_yticklabels(pivot_default.index)

for i in range(pivot_default.shape[0]):
    for j in range(pivot_default.shape[1]):
        val = pivot_default.values[i, j]
        if np.isfinite(val):
            ax.text(
                j,
                i,
                f"{val:.3f}",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if abs(val) > vmax * 0.55 else "black",
            )

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("F1 drop")

plt.tight_layout()

out_png_default = OUT_DIR / "heatmap_new_adaboost_default_f1_drop_same_style.png"
plt.savefig(out_png_default, dpi=250, bbox_inches="tight")
plt.show()

print("Saved default-threshold heatmap:")
print(out_png_default)


# ============================================================
# PLOT 2 — TUNED THRESHOLD F1 DROP
# Same layout as original figure
# ============================================================

pivot_tuned = ablation_df.pivot_table(
    index="removed_feature",
    columns="lead_h",
    values="delta_tuned_f1",
    aggfunc="mean",
)

pivot_tuned = pivot_tuned.reindex(FEATURE_COLS)
pivot_tuned = pivot_tuned[LEADS]

vmax = np.nanmax(np.abs(pivot_tuned.values))
if not np.isfinite(vmax) or vmax == 0:
    vmax = 0.01

fig, ax = plt.subplots(figsize=(5.5 if len(LEADS) == 1 else 8.5, 7.5))

im = ax.imshow(
    pivot_tuned.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)

ax.set_title(
    "NEW box2deg | AdaBoost\n"
    "Feature ablation heatmap, tuned threshold (F1 drop)"
)

ax.set_xlabel("Lead time")
ax.set_ylabel("Removed predictor")

ax.set_xticks(np.arange(len(pivot_tuned.columns)))
ax.set_xticklabels([f"{c} h" for c in pivot_tuned.columns])

ax.set_yticks(np.arange(len(pivot_tuned.index)))
ax.set_yticklabels(pivot_tuned.index)

for i in range(pivot_tuned.shape[0]):
    for j in range(pivot_tuned.shape[1]):
        val = pivot_tuned.values[i, j]
        if np.isfinite(val):
            ax.text(
                j,
                i,
                f"{val:.3f}",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if abs(val) > vmax * 0.55 else "black",
            )

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("F1 drop")

plt.tight_layout()

out_png_tuned = OUT_DIR / "heatmap_new_adaboost_tuned_f1_drop_same_style.png"
plt.savefig(out_png_tuned, dpi=250, bbox_inches="tight")
plt.show()

print("Saved tuned-threshold heatmap:")
print(out_png_tuned)

In [ ]:
# ============================================================
# OLD vs NEW — ROC-AUC / PR-AUC COMPARISON PLOT
# Best model per lead time
# NOTE:
# ROC-AUC and PR-AUC are threshold-independent,
# so this is valid for both default and tuned settings.
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
OLD_FP = Path(
    r"C:\Users\_s2218026\Documents\lab\mcs_output\ml_old_box2deg_2009_2025_final\model_performance_ranked_by_lead_old.csv"
)

NEW_FP = Path(
    r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest\ml_new_box2deg_2009_2025_UPDATED\model_performance_ranked_by_lead_new.csv"
)

OUT_DIR = Path(
    r"C:\Users\_s2218026\Documents\lab\comparison_plots_old_vs_new"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
old_df = pd.read_csv(OLD_FP)
new_df = pd.read_csv(NEW_FP)

# remove accidental unnamed columns
old_df = old_df.loc[:, ~old_df.columns.str.startswith("Unnamed")]
new_df = new_df.loc[:, ~new_df.columns.str.startswith("Unnamed")]

# ------------------------------------------------------------
# BEST MODEL PER LEAD
# ------------------------------------------------------------
old_best = (
    old_df.sort_values(["lead_h", "f1"], ascending=[True, False])
    .groupby("lead_h", as_index=False)
    .first()
)
old_best["dataset"] = "OLD"

new_best = (
    new_df.sort_values(["lead_h", "f1"], ascending=[True, False])
    .groupby("lead_h", as_index=False)
    .first()
)
new_best["dataset"] = "NEW"

compare = pd.concat([old_best, new_best], ignore_index=True)
compare = compare.sort_values(["dataset", "lead_h"]).reset_index(drop=True)

print("Best model by lead — OLD vs NEW")
display(compare[["dataset", "lead_h", "model", "f1", "precision", "recall", "roc_auc", "pr_auc"]])

# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

# ROC-AUC
ax = axes[0]
for ds, marker in [("OLD", "o"), ("NEW", "s")]:
    sub = compare[compare["dataset"] == ds].sort_values("lead_h")
    ax.plot(
        sub["lead_h"],
        sub["roc_auc"],
        marker=marker,
        linewidth=2,
        label=ds
    )

ax.set_title("ROC-AUC")
ax.set_xlabel("Lead time (h)")
ax.set_ylabel("ROC-AUC")
ax.set_xticks([6, 12, 24, 48])
ax.grid(True, alpha=0.3)
ax.legend()

# PR-AUC
ax = axes[1]
for ds, marker in [("OLD", "o"), ("NEW", "s")]:
    sub = compare[compare["dataset"] == ds].sort_values("lead_h")
    ax.plot(
        sub["lead_h"],
        sub["pr_auc"],
        marker=marker,
        linewidth=2,
        label=ds
    )

ax.set_title("PR-AUC")
ax.set_xlabel("Lead time (h)")
ax.set_ylabel("PR-AUC")
ax.set_xticks([6, 12, 24, 48])
ax.grid(True, alpha=0.3)
ax.legend()

fig.suptitle("OLD vs NEW best-model performance: ROC-AUC and PR-AUC", fontsize=16)

out_png = OUT_DIR / "old_vs_new_rocauc_prauc_comparison.png"
plt.savefig(out_png, dpi=220, bbox_inches="tight")
plt.show()

print("Saved:", out_png)

In [ ]:
# ============================================================
# OLD vs NEW — AdaBoost feature ablation heatmap
# All leads in one figure
# No lon, lat, month
#
# Rows    = removed predictors
# Columns = lead time
# Value   = baseline F1 - ablated F1
#
# Positive value  = removing feature worsens F1 -> feature useful
# Negative value  = removing feature improves F1 -> feature may add noise
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.ensemble import AdaBoostClassifier

# ============================================================
# CONFIG
# ============================================================

LAB_DIR = Path(r"C:\Users\_s2218026\Documents\lab")

CASE_DIRS = {
    "old": LAB_DIR / "mcs_output" / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED",
    "new": LAB_DIR / "mcs_output_softtest" / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED",
}

OUT_DIR = LAB_DIR / "feature_ablation_adaboost_old_vs_new_all_leads"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_NAME = "box2deg"
LEADS = [6, 12, 24, 48]

RANDOM_STATE = 42
N_SPLITS = 5

# No lon, lat, month
FEATURE_COLS = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
]

# Choose which F1 drop to plot:
#   "default" = threshold 0.5
#   "tuned"   = F1-optimized threshold from OOF scores
SCORE_MODE = "default"


# ============================================================
# FUNCTIONS
# ============================================================

def make_adaboost(random_state=42):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", AdaBoostClassifier(
            n_estimators=300,
            random_state=random_state
        ))
    ])


def safe_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    out = {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_score)
    except Exception:
        out["roc_auc"] = np.nan

    try:
        out["pr_auc"] = average_precision_score(y_true, y_score)
    except Exception:
        out["pr_auc"] = np.nan

    return out


def threshold_scan(y_true, y_score):
    rows = []

    for thr in np.round(np.arange(0.05, 0.951, 0.01), 2):
        m = safe_metrics(y_true, y_score, threshold=thr)
        rows.append({
            "threshold": thr,
            **m,
        })

    scan = pd.DataFrame(rows)

    scan = (
        scan
        .sort_values(["f1", "recall", "precision"], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    return scan


def make_cv(y, groups=None, n_splits=5, random_state=42):
    y = np.asarray(y).astype(int)

    min_class_count = np.bincount(y).min()
    n_splits_eff = min(n_splits, int(min_class_count))

    if n_splits_eff < 2:
        raise ValueError("Not enough class samples for cross-validation.")

    if groups is not None:
        try:
            cv = StratifiedGroupKFold(
                n_splits=n_splits_eff,
                shuffle=True,
                random_state=random_state,
            )
            return cv, "StratifiedGroupKFold", n_splits_eff
        except Exception:
            pass

    cv = StratifiedKFold(
        n_splits=n_splits_eff,
        shuffle=True,
        random_state=random_state,
    )

    return cv, "StratifiedKFold", n_splits_eff


def evaluate_feature_set(df, feature_cols, random_state=42, n_splits=5):
    X = df[feature_cols].copy()
    y = df["y"].astype(int).values
    groups = df["track_id"].astype(str).values if "track_id" in df.columns else None

    cv, cv_name, n_splits_eff = make_cv(
        y=y,
        groups=groups,
        n_splits=n_splits,
        random_state=random_state,
    )

    y_score_oof = np.full(len(df), np.nan)

    splitter = cv.split(X, y, groups) if groups is not None else cv.split(X, y)

    for fold_id, (train_idx, test_idx) in enumerate(splitter, start=1):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y[train_idx]

        pipe = make_adaboost(random_state=random_state)
        pipe.fit(X_train, y_train)

        y_score_oof[test_idx] = pipe.predict_proba(X_test)[:, 1]

    default_metrics = safe_metrics(y, y_score_oof, threshold=0.5)

    scan = threshold_scan(y, y_score_oof)
    best_threshold = float(scan.loc[0, "threshold"])
    tuned_metrics = safe_metrics(y, y_score_oof, threshold=best_threshold)

    return {
        "cv": cv_name,
        "n_splits": n_splits_eff,
        "default_threshold": 0.5,
        "tuned_threshold": best_threshold,

        "default_f1": default_metrics["f1"],
        "default_precision": default_metrics["precision"],
        "default_recall": default_metrics["recall"],
        "default_roc_auc": default_metrics["roc_auc"],
        "default_pr_auc": default_metrics["pr_auc"],

        "tuned_f1": tuned_metrics["f1"],
        "tuned_precision": tuned_metrics["precision"],
        "tuned_recall": tuned_metrics["recall"],
        "tuned_roc_auc": tuned_metrics["roc_auc"],
        "tuned_pr_auc": tuned_metrics["pr_auc"],
    }


# ============================================================
# RUN ABLATION FOR OLD AND NEW
# ============================================================

all_rows = []

for dataset_name, case_dir in CASE_DIRS.items():
    print("\n" + "=" * 90)
    print(f"{dataset_name.upper()} DATASET — AdaBoost feature ablation")
    print("=" * 90)

    for lead in LEADS:
        case_fp = case_dir / f"{LABEL_NAME}_case_table_{dataset_name}_{lead}h.csv"

        if not case_fp.exists():
            raise FileNotFoundError(case_fp)

        df = pd.read_csv(case_fp, low_memory=False)

        if "time" in df.columns:
            df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

        missing = [c for c in FEATURE_COLS + ["y"] if c not in df.columns]
        if missing:
            raise ValueError(f"Missing columns in {case_fp.name}: {missing}")

        df = df.dropna(subset=["y"]).copy()

        print("\n" + "-" * 80)
        print(f"{dataset_name.upper()} | Lead {lead}h")
        print("Rows:", len(df))
        print("Positive:", int(df["y"].sum()))
        print("Negative:", int((df["y"] == 0).sum()))
        print("-" * 80)

        baseline = evaluate_feature_set(
            df=df,
            feature_cols=FEATURE_COLS,
            random_state=RANDOM_STATE,
            n_splits=N_SPLITS,
        )

        print(
            f"Baseline default F1={baseline['default_f1']:.4f} | "
            f"tuned F1={baseline['tuned_f1']:.4f} | "
            f"tuned threshold={baseline['tuned_threshold']:.2f}"
        )

        for removed_feature in FEATURE_COLS:
            reduced_features = [c for c in FEATURE_COLS if c != removed_feature]

            ablated = evaluate_feature_set(
                df=df,
                feature_cols=reduced_features,
                random_state=RANDOM_STATE,
                n_splits=N_SPLITS,
            )

            row = {
                "dataset": dataset_name,
                "model": "AdaBoost",
                "lead_h": lead,
                "removed_feature": removed_feature,

                "baseline_default_f1": baseline["default_f1"],
                "ablated_default_f1": ablated["default_f1"],
                "delta_default_f1": baseline["default_f1"] - ablated["default_f1"],

                "baseline_tuned_f1": baseline["tuned_f1"],
                "ablated_tuned_f1": ablated["tuned_f1"],
                "delta_tuned_f1": baseline["tuned_f1"] - ablated["tuned_f1"],

                "baseline_default_pr_auc": baseline["default_pr_auc"],
                "ablated_default_pr_auc": ablated["default_pr_auc"],
                "delta_pr_auc": baseline["default_pr_auc"] - ablated["default_pr_auc"],

                "baseline_default_roc_auc": baseline["default_roc_auc"],
                "ablated_default_roc_auc": ablated["default_roc_auc"],
                "delta_roc_auc": baseline["default_roc_auc"] - ablated["default_roc_auc"],

                "baseline_tuned_threshold": baseline["tuned_threshold"],
                "ablated_tuned_threshold": ablated["tuned_threshold"],
            }

            all_rows.append(row)

            print(
                f"remove {removed_feature:18s} | "
                f"default ΔF1={row['delta_default_f1']:+.4f} | "
                f"tuned ΔF1={row['delta_tuned_f1']:+.4f}"
            )

ablation_df = pd.DataFrame(all_rows)

out_csv = OUT_DIR / "feature_ablation_adaboost_old_vs_new_all_leads_no_geo.csv"
ablation_df.to_csv(out_csv, index=False)

print("\nSaved ablation table:")
print(out_csv)

display(ablation_df)


# ============================================================
# PLOT OLD vs NEW IN ONE FIGURE
# ============================================================

if SCORE_MODE == "default":
    value_col = "delta_default_f1"
    mode_title = "default threshold"
    out_name = "heatmap_old_vs_new_adaboost_all_leads_default_f1_drop_no_geo.png"
elif SCORE_MODE == "tuned":
    value_col = "delta_tuned_f1"
    mode_title = "tuned threshold"
    out_name = "heatmap_old_vs_new_adaboost_all_leads_tuned_f1_drop_no_geo.png"
else:
    raise ValueError("SCORE_MODE must be 'default' or 'tuned'")

# Same color scale across OLD and NEW
vmax_all = np.nanmax(np.abs(ablation_df[value_col].values))
if not np.isfinite(vmax_all) or vmax_all == 0:
    vmax_all = 0.01

fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.2), constrained_layout=True)

for ax, dataset_name in zip(axes, ["old", "new"]):
    sub = ablation_df[ablation_df["dataset"] == dataset_name].copy()

    pivot = sub.pivot_table(
        index="removed_feature",
        columns="lead_h",
        values=value_col,
        aggfunc="mean",
    )

    pivot = pivot.reindex(FEATURE_COLS)
    pivot = pivot[LEADS]

    im = ax.imshow(
        pivot.values,
        aspect="auto",
        cmap="RdBu_r",
        vmin=-vmax_all,
        vmax=vmax_all,
    )

    ax.set_title(
        f"{dataset_name.upper()} box2deg | AdaBoost\n"
        f"Feature ablation ({mode_title})"
    )

    ax.set_xlabel("Lead time")
    ax.set_ylabel("Removed predictor")

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([f"{c} h" for c in pivot.columns])

    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if np.isfinite(val):
                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if abs(val) > vmax_all * 0.55 else "black",
                )

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.86)
cbar.set_label("F1 drop")

fig.suptitle(
    "OLD vs NEW AdaBoost feature ablation heatmap\n"
    "No lon/lat/month | value = baseline F1 − ablated F1",
    fontsize=15,
)

out_png = OUT_DIR / out_name
plt.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print("Saved heatmap:")
print(out_png)

In [ ]:
# ============================================================
# OLD vs NEW — AdaBoost SHAP heatmap
# All leads in one figure
# No lon, lat, month
#
# Rows    = predictors
# Columns = lead time
# Value   = mean absolute SHAP value
#
# Output:
#   - shap_importance_adaboost_old_vs_new_all_leads_no_geo.csv
#   - heatmap_old_vs_new_adaboost_shap_all_leads_no_geo.png
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier

# ============================================================
# CONFIG
# ============================================================

LAB_DIR = Path(r"C:\Users\_s2218026\Documents\lab")

CASE_DIRS = {
    "old": LAB_DIR / "mcs_output" / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED",
    "new": LAB_DIR / "mcs_output_softtest" / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED",
}

OUT_DIR = LAB_DIR / "shap_adaboost_old_vs_new_all_leads_no_geo"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_NAME = "box2deg"
LEADS = [6, 12, 24, 48]

RANDOM_STATE = 42

# SHAP sampling settings
BACKGROUND_N = 80
EXPLAIN_N = 300

# No lon, lat, month
FEATURE_COLS = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
]

# ============================================================
# IMPORT SHAP
# ============================================================

try:
    import shap
except ImportError:
    raise ImportError("SHAP is not installed. Install it with: pip install shap")

# ============================================================
# FUNCTIONS
# ============================================================

def make_adaboost(random_state=42):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", AdaBoostClassifier(
            n_estimators=300,
            random_state=random_state
        ))
    ])


def balanced_sample(df, n_total=300, random_state=42):
    """
    Balanced sample for SHAP explanation:
    roughly half developing, half non-developing if possible.
    """
    pos = df[df["y"].astype(int) == 1].copy()
    neg = df[df["y"].astype(int) == 0].copy()

    n_pos = min(len(pos), n_total // 2)
    n_neg = min(len(neg), n_total - n_pos)

    parts = []

    if n_pos > 0:
        parts.append(pos.sample(n=n_pos, random_state=random_state))

    if n_neg > 0:
        parts.append(neg.sample(n=n_neg, random_state=random_state))

    if not parts:
        raise ValueError("No rows available for SHAP sampling.")

    out = pd.concat(parts, ignore_index=True)
    out = out.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return out


def compute_adaboost_shap_importance(df, feature_cols, random_state=42):
    """
    Uses model-agnostic permutation SHAP through shap.Explainer.
    This is slower than tree-specific SHAP, but safer for sklearn AdaBoost.
    """
    df = df.dropna(subset=["y"]).copy()

    X = df[feature_cols].copy()
    y = df["y"].astype(int).values

    pipe = make_adaboost(random_state=random_state)
    pipe.fit(X, y)

    # background sample
    bg = X.sample(
        n=min(BACKGROUND_N, len(X)),
        random_state=random_state
    ).copy()

    # explanation sample
    explain_df = balanced_sample(
        df,
        n_total=min(EXPLAIN_N, len(df)),
        random_state=random_state
    )

    X_explain = explain_df[feature_cols].copy()

    def predict_positive_class(X_array):
        X_df = pd.DataFrame(X_array, columns=feature_cols)
        return pipe.predict_proba(X_df)[:, 1]

    explainer = shap.Explainer(
        predict_positive_class,
        bg.values,
        algorithm="permutation"
    )

    shap_values = explainer(
        X_explain.values,
        max_evals=2 * len(feature_cols) + 1
    )

    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

    imp = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": mean_abs_shap
    }).sort_values("mean_abs_shap", ascending=False)

    return imp


# ============================================================
# RUN SHAP FOR OLD AND NEW
# ============================================================

shap_rows = []

for dataset_name, case_dir in CASE_DIRS.items():
    print("\n" + "=" * 90)
    print(f"{dataset_name.upper()} DATASET — AdaBoost SHAP")
    print("=" * 90)

    for lead in LEADS:
        case_fp = case_dir / f"{LABEL_NAME}_case_table_{dataset_name}_{lead}h.csv"

        if not case_fp.exists():
            raise FileNotFoundError(case_fp)

        df = pd.read_csv(case_fp, low_memory=False)

        if "time" in df.columns:
            df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

        missing = [c for c in FEATURE_COLS + ["y"] if c not in df.columns]
        if missing:
            raise ValueError(f"Missing columns in {case_fp.name}: {missing}")

        df = df.dropna(subset=["y"]).copy()

        print("\n" + "-" * 80)
        print(f"{dataset_name.upper()} | Lead {lead}h")
        print("Rows:", len(df))
        print("Positive:", int(df["y"].sum()))
        print("Negative:", int((df["y"] == 0).sum()))
        print("-" * 80)

        imp = compute_adaboost_shap_importance(
            df=df,
            feature_cols=FEATURE_COLS,
            random_state=RANDOM_STATE + lead
        )

        print("Top SHAP features:")
        display(imp.head(5))

        out_imp = OUT_DIR / f"shap_importance_{dataset_name}_adaboost_{lead}h_no_geo.csv"
        imp.to_csv(out_imp, index=False)

        for _, r in imp.iterrows():
            shap_rows.append({
                "dataset": dataset_name,
                "model": "AdaBoost",
                "lead_h": lead,
                "feature": r["feature"],
                "mean_abs_shap": r["mean_abs_shap"],
            })

shap_df = pd.DataFrame(shap_rows)

out_csv = OUT_DIR / "shap_importance_adaboost_old_vs_new_all_leads_no_geo.csv"
shap_df.to_csv(out_csv, index=False)

print("\nSaved SHAP importance table:")
print(out_csv)

display(shap_df)

# ============================================================
# PLOT OLD vs NEW SHAP HEATMAP IN ONE FIGURE
# ============================================================

vmax_global = shap_df["mean_abs_shap"].max()

if not np.isfinite(vmax_global) or vmax_global == 0:
    vmax_global = 0.01

fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.2), constrained_layout=True)

for ax, dataset_name in zip(axes, ["old", "new"]):
    sub = shap_df[shap_df["dataset"] == dataset_name].copy()

    pivot = sub.pivot_table(
        index="feature",
        columns="lead_h",
        values="mean_abs_shap",
        aggfunc="mean"
    )

    pivot = pivot.reindex(FEATURE_COLS)
    pivot = pivot[LEADS]

    im = ax.imshow(
        pivot.values,
        aspect="auto",
        cmap="viridis",
        vmin=0,
        vmax=vmax_global
    )

    ax.set_title(
        f"{dataset_name.upper()} box2deg | AdaBoost\n"
        "SHAP heatmap, mean |SHAP value|"
    )

    ax.set_xlabel("Lead time")
    ax.set_ylabel("Predictor")

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([f"{c} h" for c in pivot.columns])

    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]

            if np.isfinite(val):
                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if val > vmax_global * 0.45 else "black",
                )

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.86)
cbar.set_label("Mean absolute SHAP value")

fig.suptitle(
    "OLD vs NEW AdaBoost SHAP heatmap\n"
    "No lon/lat/month",
    fontsize=15,
)

out_png = OUT_DIR / "heatmap_old_vs_new_adaboost_shap_all_leads_no_geo.png"
plt.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print("Saved SHAP heatmap:")
print(out_png)


In [ ]:
# ============================================================
# THRESHOLD-TUNING SENSITIVITY
# OLD vs NEW
#
# Outputs:
#   1. 3-panel comparison: F1 / Precision / Recall
#      OLD default, OLD tuned, NEW default, NEW tuned
#
#   2. Threshold-sweep curves by lead:
#      F1 / Precision / Recall vs threshold
#
#   3. CSV summary table
#
# Important:
#   - Uses the same selected model per dataset/lead.
#   - Model is selected from the default-threshold ranked table by highest F1.
#   - Then default and tuned thresholds are compared for that same model.
#   - This isolates threshold sensitivity from model-selection effects.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# ============================================================
# PATHS
# ============================================================

LAB_DIR = Path(r"C:\Users\_s2218026\Documents\lab")

DATASETS = {
    "old": {
        "display": "OLD",
        "ranked_fp": LAB_DIR / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "model_performance_ranked_by_lead_old.csv",
        "oof_fp": LAB_DIR / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_all_models.csv",
    },
    "new": {
        "display": "NEW",
        "ranked_fp": LAB_DIR / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "model_performance_ranked_by_lead_new.csv",
        "oof_fp": LAB_DIR / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_new.csv",
    },
}

OUT_DIR = LAB_DIR / "threshold_tuning_sensitivity_old_vs_new"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LEADS = [6, 12, 24, 48]
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.01), 2)

# ============================================================
# FUNCTIONS
# ============================================================

def safe_metrics(y_true, y_score, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    y_pred = (y_score >= threshold).astype(int)

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_score)
    except Exception:
        out["roc_auc"] = np.nan

    try:
        out["pr_auc"] = average_precision_score(y_true, y_score)
    except Exception:
        out["pr_auc"] = np.nan

    return out


def threshold_scan(y_true, y_score):
    rows = []

    for thr in THRESHOLDS:
        m = safe_metrics(y_true, y_score, threshold=thr)

        rows.append({
            "threshold": thr,
            **m,
        })

    scan = pd.DataFrame(rows)

    scan_ranked = (
        scan
        .sort_values(["f1", "recall", "precision"], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    best_threshold = float(scan_ranked.loc[0, "threshold"])

    return scan, best_threshold


def load_best_model_from_ranked(ranked_fp):
    if not ranked_fp.exists():
        raise FileNotFoundError(ranked_fp)

    ranked = pd.read_csv(ranked_fp)
    ranked = ranked.loc[:, ~ranked.columns.str.startswith("Unnamed")]

    required = ["lead_h", "model", "f1"]

    missing = [c for c in required if c not in ranked.columns]
    if missing:
        raise ValueError(f"Missing columns in {ranked_fp}: {missing}")

    best = (
        ranked
        .sort_values(["lead_h", "f1"], ascending=[True, False])
        .groupby("lead_h", as_index=False)
        .first()
    )

    return best


def load_oof(oof_fp):
    if not oof_fp.exists():
        raise FileNotFoundError(oof_fp)

    oof = pd.read_csv(oof_fp)
    oof = oof.loc[:, ~oof.columns.str.startswith("Unnamed")]

    required = ["lead_h", "model", "y_true", "y_score"]

    missing = [c for c in required if c not in oof.columns]
    if missing:
        raise ValueError(f"Missing columns in {oof_fp}: {missing}")

    return oof


# ============================================================
# COMPUTE DEFAULT VS TUNED METRICS
# ============================================================

summary_rows = []
scan_rows = []

for dataset_key, info in DATASETS.items():
    display_name = info["display"]

    best_models = load_best_model_from_ranked(info["ranked_fp"])
    oof = load_oof(info["oof_fp"])

    print("\n" + "=" * 90)
    print(f"{display_name} DATASET")
    print("=" * 90)

    print("\nBest model per lead selected from default-threshold ranked table:")
    display(best_models[["lead_h", "model", "f1"]])

    for _, row in best_models.iterrows():
        lead = int(row["lead_h"])
        model_name = row["model"]

        sub = oof[
            (oof["lead_h"] == lead)
            & (oof["model"] == model_name)
        ].copy()

        if len(sub) == 0:
            raise ValueError(
                f"No OOF rows found for {display_name}, lead={lead}, model={model_name}"
            )

        y_true = sub["y_true"].astype(int).values
        y_score = sub["y_score"].astype(float).values

        scan, best_threshold = threshold_scan(y_true, y_score)

        scan["dataset"] = dataset_key
        scan["dataset_display"] = display_name
        scan["lead_h"] = lead
        scan["model"] = model_name
        scan_rows.append(scan)

        # default threshold
        m_default = safe_metrics(y_true, y_score, threshold=0.5)

        summary_rows.append({
            "dataset": dataset_key,
            "dataset_display": display_name,
            "lead_h": lead,
            "model": model_name,
            "threshold_type": "default",
            "threshold": 0.5,
            "n": len(sub),
            "positive": int(y_true.sum()),
            "negative": int((y_true == 0).sum()),
            **m_default,
        })

        # tuned threshold
        m_tuned = safe_metrics(y_true, y_score, threshold=best_threshold)

        summary_rows.append({
            "dataset": dataset_key,
            "dataset_display": display_name,
            "lead_h": lead,
            "model": model_name,
            "threshold_type": "tuned",
            "threshold": best_threshold,
            "n": len(sub),
            "positive": int(y_true.sum()),
            "negative": int((y_true == 0).sum()),
            **m_tuned,
        })

summary_df = pd.DataFrame(summary_rows)
scan_df = pd.concat(scan_rows, ignore_index=True)

summary_fp = OUT_DIR / "threshold_sensitivity_summary_default_vs_tuned_old_new.csv"
scan_fp = OUT_DIR / "threshold_sensitivity_scan_curves_old_new.csv"

summary_df.to_csv(summary_fp, index=False)
scan_df.to_csv(scan_fp, index=False)

print("\nSaved summary:")
print(summary_fp)

print("\nSaved threshold scans:")
print(scan_fp)

display(summary_df)

# ============================================================
# 1) THREE-PANEL DEFAULT VS TUNED COMPARISON
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

metrics = [
    ("f1", "F1"),
    ("precision", "Precision"),
    ("recall", "Recall"),
]

line_specs = [
    ("old", "default", "OLD default", "o", "-"),
    ("old", "tuned", "OLD tuned", "o", "--"),
    ("new", "default", "NEW default", "s", "-"),
    ("new", "tuned", "NEW tuned", "s", "--"),
]

for ax, (metric, label) in zip(axes, metrics):
    for dataset_key, threshold_type, line_label, marker, linestyle in line_specs:
        sub = summary_df[
            (summary_df["dataset"] == dataset_key)
            & (summary_df["threshold_type"] == threshold_type)
        ].sort_values("lead_h")

        ax.plot(
            sub["lead_h"],
            sub[metric],
            marker=marker,
            linestyle=linestyle,
            linewidth=2,
            label=line_label,
        )

    ax.set_xticks(LEADS)
    ax.set_xlabel("Lead time (h)")
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle(
    "Threshold-tuning sensitivity: OLD vs NEW\n"
    "Default threshold = 0.5; tuned threshold = F1-optimized from OOF scores",
    fontsize=14,
)

out_png = OUT_DIR / "old_vs_new_default_vs_tuned_3panel_f1_precision_recall.png"
plt.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print("Saved 3-panel comparison:")
print(out_png)

# ============================================================
# 2) THRESHOLD-SWEEP CURVES BY LEAD
# Better sensitivity plot
# ============================================================

for lead in LEADS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

    for ax, (metric, label) in zip(axes, metrics):
        for dataset_key, display_name, marker_style in [
            ("old", "OLD", "-"),
            ("new", "NEW", "--"),
        ]:
            sub = scan_df[
                (scan_df["dataset"] == dataset_key)
                & (scan_df["lead_h"] == lead)
            ].copy()

            if len(sub) == 0:
                continue

            # there is only one selected model per dataset/lead
            model_name = sub["model"].iloc[0]

            ax.plot(
                sub["threshold"],
                sub[metric],
                linestyle=marker_style,
                linewidth=2,
                label=f"{display_name}: {model_name}",
            )

            # default threshold vertical line
            ax.axvline(
                0.5,
                color="gray",
                linestyle=":",
                linewidth=1,
            )

            # tuned threshold vertical line
            best_row = summary_df[
                (summary_df["dataset"] == dataset_key)
                & (summary_df["lead_h"] == lead)
                & (summary_df["threshold_type"] == "tuned")
            ]

            if len(best_row) > 0:
                best_thr = float(best_row["threshold"].iloc[0])

                ax.axvline(
                    best_thr,
                    linestyle="--",
                    linewidth=1,
                    alpha=0.8,
                )

        ax.set_xlabel("Decision threshold")
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    fig.suptitle(
        f"Threshold-sensitivity curves at {lead}h lead\n"
        "Vertical dotted line = 0.5; dashed line = tuned threshold",
        fontsize=14,
    )

    out_png = OUT_DIR / f"threshold_sensitivity_curves_old_vs_new_{lead}h_f1_precision_recall.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved threshold-sweep figure:")
    print(out_png)

# ============================================================
# 3) THRESHOLD SHIFT TABLE
# ============================================================

pivot_view = summary_df.pivot_table(
    index=["dataset", "lead_h", "model"],
    columns="threshold_type",
    values=["threshold", "f1", "precision", "recall"],
    aggfunc="first"
)

display(pivot_view)

In [ ]:
# ============================================================
# SHAP PLOTS FOR ML RESULTS
# Compatible with ML_CHECK_LATEST structure
#
# Produces:
#   1. SHAP bar plot
#   2. SHAP beeswarm plot
#   3. SHAP dependence plots for top features
#   4. SHAP heatmap across lead times
#   5. SHAP importance CSV tables
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone

# ------------------------------------------------------------
# Import SHAP
# ------------------------------------------------------------
try:
    import shap
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])
    import shap

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------
SHAP_DIR = OUT_DIR / "shap_plots"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SHAP_MODEL_NAMES = [
    "AdaBoost",
    "Random Forest",
    "SVM",
    "Logistic Regression",
    "MLP",
]

LEADS_TO_EXPLAIN = LEADS

# Keep these moderate first. Increase later if needed.
SHAP_BACKGROUND_N = 80
SHAP_POS_N = 250
SHAP_NEG_N = 250

N_TOP_DEPENDENCE = 5
SHAP_RANDOM_STATE = RANDOM_STATE

PREDICTOR_ORDER = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
    "month",
    "lat",
    "lon",
]

LEAD_ORDER = [6, 12, 24, 48]

# ------------------------------------------------------------
# Load case tables if datasets is not already available
# ------------------------------------------------------------
try:
    datasets
    print("Using existing datasets dictionary.")
except NameError:
    print("datasets not found. Loading case tables from CASE_DIR...")

    datasets = {}

    for L in LEADS_TO_EXPLAIN:
        fp = CASE_DIR / f"{LABEL_NAME}_case_table_{DATASET_NAME}_{L}h.csv"

        if not fp.exists():
            raise FileNotFoundError(fp)

        df = pd.read_csv(fp)

        if "time" in df.columns:
            df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

        df["y"] = df["y"].astype(int)
        datasets[L] = df.copy()

        print(
            f"Lead {L}h | rows={len(df)} | "
            f"positive={int(df['y'].sum())} | "
            f"negative={int((df['y'] == 0).sum())}"
        )

# ------------------------------------------------------------
# Helper: balanced SHAP sample
# ------------------------------------------------------------
def make_shap_samples(X, y, background_n=80, pos_n=250, neg_n=250, random_state=42):
    rng = np.random.default_rng(random_state)

    y = np.asarray(y).astype(int)

    all_idx = np.arange(len(y))
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    bg_n = min(background_n, len(all_idx))
    bg_idx = rng.choice(all_idx, size=bg_n, replace=False)

    pos_take = min(pos_n, len(pos_idx))
    neg_take = min(neg_n, len(neg_idx))

    exp_parts = []

    if pos_take > 0:
        exp_parts.append(rng.choice(pos_idx, size=pos_take, replace=False))

    if neg_take > 0:
        exp_parts.append(rng.choice(neg_idx, size=neg_take, replace=False))

    exp_idx = np.concatenate(exp_parts)
    rng.shuffle(exp_idx)

    X_bg = X.iloc[bg_idx].copy()
    X_exp = X.iloc[exp_idx].copy()
    y_exp = y[exp_idx]

    return X_bg, X_exp, y_exp, exp_idx

# ------------------------------------------------------------
# Helper: prediction function for positive class probability
# ------------------------------------------------------------
def make_predict_positive_func(pipe, feature_cols):
    def predict_positive(z):
        if isinstance(z, pd.DataFrame):
            Xz = z[feature_cols].copy()
        else:
            Xz = pd.DataFrame(z, columns=feature_cols)

        return pipe.predict_proba(Xz)[:, 1]

    return predict_positive

# ------------------------------------------------------------
# Helper: dependence plots
# ------------------------------------------------------------
def save_dependence_plots(shap_values, X_exp, top_features, out_dir, prefix):
    shap_arr = shap_values.values

    for feat in top_features:
        if feat not in X_exp.columns:
            continue

        j = list(X_exp.columns).index(feat)

        fig, ax = plt.subplots(figsize=(6.8, 4.8))

        ax.scatter(
            X_exp[feat].values,
            shap_arr[:, j],
            s=16,
            alpha=0.55
        )

        ax.axhline(0, linewidth=1)
        ax.set_xlabel(feat)
        ax.set_ylabel("SHAP value")
        ax.set_title(f"{prefix}\nSHAP dependence: {feat}")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        safe_feat = feat.replace("/", "_").replace(" ", "_")
        safe_prefix = (
            prefix.replace(" ", "_")
            .replace("|", "")
            .replace("/", "_")
        )

        out_png = out_dir / f"{safe_prefix}_dependence_{safe_feat}.png"

        fig.savefig(out_png, dpi=220, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        print("Saved:", out_png)

# ------------------------------------------------------------
# Run SHAP
# ------------------------------------------------------------
models_all = get_models(random_state=RANDOM_STATE)

missing_models = [m for m in SHAP_MODEL_NAMES if m not in models_all]
if missing_models:
    raise ValueError(
        f"These models are not available in get_models(): {missing_models}\n"
        f"Available models: {list(models_all.keys())}"
    )

all_shap_importance = []
all_shap_samples = []

for model_name in SHAP_MODEL_NAMES:
    print("\n" + "=" * 90)
    print(f"SHAP ANALYSIS | MODEL: {model_name}")
    print("=" * 90)

    safe_model = model_name.replace(" ", "_").replace("/", "_")
    model_dir = SHAP_DIR / safe_model
    model_dir.mkdir(parents=True, exist_ok=True)

    for L in LEADS_TO_EXPLAIN:
        print("\n" + "-" * 90)
        print(f"Lead {L}h")
        print("-" * 90)

        case_df = datasets[L].copy()

        use_cols = FEATURE_COLS + ["y"]
        case_df = case_df.dropna(subset=use_cols).reset_index(drop=True)

        X = case_df[FEATURE_COLS].copy()
        y = case_df["y"].astype(int).values

        print(
            f"Rows used: {len(case_df)} | "
            f"positive={int(y.sum())} | "
            f"negative={int((y == 0).sum())}"
        )

        # ------------------------------------------------------------
        # Fit final model on all rows for this lead
        # ------------------------------------------------------------
        base_model = clone(models_all[model_name])
        pipe = make_pipeline(model_name, base_model)
        pipe.fit(X, y)

        # ------------------------------------------------------------
        # Balanced SHAP explanation sample
        # ------------------------------------------------------------
        X_bg, X_exp, y_exp, exp_idx = make_shap_samples(
            X=X,
            y=y,
            background_n=SHAP_BACKGROUND_N,
            pos_n=SHAP_POS_N,
            neg_n=SHAP_NEG_N,
            random_state=SHAP_RANDOM_STATE + int(L)
        )

        print(
            f"Background sample: {len(X_bg)} | "
            f"Explanation sample: {len(X_exp)} | "
            f"Explanation positives={int(y_exp.sum())}"
        )

        predict_positive = make_predict_positive_func(pipe, FEATURE_COLS)

        masker = shap.maskers.Independent(X_bg)

        explainer = shap.Explainer(
            predict_positive,
            masker,
            algorithm="permutation",
            feature_names=FEATURE_COLS
        )

        shap_values = explainer(
            X_exp,
            max_evals=2 * len(FEATURE_COLS) + 1,
            batch_size=50
        )

        shap_arr = shap_values.values
        y_prob_exp = pipe.predict_proba(X_exp)[:, 1]

        # ------------------------------------------------------------
        # Save explained sample
        # ------------------------------------------------------------
        sample_out = X_exp.copy()
        sample_out["y_true"] = y_exp
        sample_out["y_prob"] = y_prob_exp
        sample_out["dataset"] = DATASET_NAME
        sample_out["label"] = LABEL_NAME
        sample_out["lead_h"] = L
        sample_out["model"] = model_name

        sample_csv = model_dir / f"shap_explanation_sample_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.csv"
        sample_out.to_csv(sample_csv, index=False)

        all_shap_samples.append(sample_out)

        # ------------------------------------------------------------
        # SHAP global importance table
        # ------------------------------------------------------------
        imp = pd.DataFrame({
            "feature": FEATURE_COLS,
            "mean_abs_shap": np.mean(np.abs(shap_arr), axis=0),
            "mean_shap": np.mean(shap_arr, axis=0),
            "std_shap": np.std(shap_arr, axis=0),
        })

        imp["rank"] = imp["mean_abs_shap"].rank(ascending=False, method="dense").astype(int)
        imp = imp.sort_values("mean_abs_shap", ascending=False)

        imp["dataset"] = DATASET_NAME
        imp["label"] = LABEL_NAME
        imp["lead_h"] = L
        imp["model"] = model_name
        imp["n_explain"] = len(X_exp)
        imp["n_background"] = len(X_bg)
        imp["n_positive_explain"] = int(y_exp.sum())
        imp["n_negative_explain"] = int((y_exp == 0).sum())

        all_shap_importance.append(imp)

        imp_csv = model_dir / f"shap_importance_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.csv"
        imp.to_csv(imp_csv, index=False)

        print("Top SHAP features:")
        display(imp[["lead_h", "model", "feature", "mean_abs_shap", "mean_shap", "rank"]].head(10))

        # ------------------------------------------------------------
        # Save SHAP values
        # ------------------------------------------------------------
        shap_value_df = pd.DataFrame(
            shap_arr,
            columns=[f"shap_{c}" for c in FEATURE_COLS]
        )

        shap_value_df["y_true"] = y_exp
        shap_value_df["y_prob"] = y_prob_exp
        shap_value_df["dataset"] = DATASET_NAME
        shap_value_df["label"] = LABEL_NAME
        shap_value_df["lead_h"] = L
        shap_value_df["model"] = model_name

        shap_values_csv = model_dir / f"shap_values_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.csv"
        shap_value_df.to_csv(shap_values_csv, index=False)

        # ------------------------------------------------------------
        # SHAP bar plot
        # ------------------------------------------------------------
        title_prefix = f"{DATASET_NAME.upper()} {LABEL_NAME} | {model_name} | Lead {L}h"

        bar_png = model_dir / f"shap_bar_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.png"

        plt.figure(figsize=(8, 6))
        shap.plots.bar(
            shap_values,
            max_display=len(FEATURE_COLS),
            show=False
        )
        fig = plt.gcf()
        fig.suptitle(
            f"{title_prefix}\nSHAP global importance: mean absolute SHAP value",
            fontsize=13,
            y=1.02
        )
        plt.tight_layout()
        fig.savefig(bar_png, dpi=220, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        print("Saved:", bar_png)

        # ------------------------------------------------------------
        # SHAP beeswarm plot
        # ------------------------------------------------------------
        beeswarm_png = model_dir / f"shap_beeswarm_{DATASET_NAME}_{LABEL_NAME}_{safe_model}_{L}h.png"

        plt.figure(figsize=(8.5, 6.5))
        shap.plots.beeswarm(
            shap_values,
            max_display=len(FEATURE_COLS),
            show=False
        )
        fig = plt.gcf()
        fig.suptitle(
            f"{title_prefix}\nSHAP beeswarm: feature effects on developing-class probability",
            fontsize=13,
            y=1.02
        )
        plt.tight_layout()
        fig.savefig(beeswarm_png, dpi=220, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        print("Saved:", beeswarm_png)

        # ------------------------------------------------------------
        # SHAP dependence plots for top predictors
        # ------------------------------------------------------------
        top_features = imp["feature"].head(N_TOP_DEPENDENCE).tolist()

        save_dependence_plots(
            shap_values=shap_values,
            X_exp=X_exp,
            top_features=top_features,
            out_dir=model_dir,
            prefix=title_prefix
        )

# ------------------------------------------------------------
# Save combined SHAP outputs
# ------------------------------------------------------------
shap_importance_all = pd.concat(all_shap_importance, ignore_index=True)
shap_importance_all.to_csv(
    SHAP_DIR / "shap_importance_all_models_all_leads.csv",
    index=False
)

shap_samples_all = pd.concat(all_shap_samples, ignore_index=True)
shap_samples_all.to_csv(
    SHAP_DIR / "shap_explanation_samples_all_models_all_leads.csv",
    index=False
)

print("\nSaved combined SHAP importance:")
print(SHAP_DIR / "shap_importance_all_models_all_leads.csv")

# ------------------------------------------------------------
# SHAP HEATMAP ACROSS LEAD TIMES
# rows = predictors
# cols = lead times
# values = mean absolute SHAP value
# ------------------------------------------------------------
for model_name in SHAP_MODEL_NAMES:
    safe_model = model_name.replace(" ", "_").replace("/", "_")

    sub = shap_importance_all[
        shap_importance_all["model"] == model_name
    ].copy()

    heat = sub.pivot_table(
        index="feature",
        columns="lead_h",
        values="mean_abs_shap",
        aggfunc="mean"
    )

    heat = heat.reindex(index=PREDICTOR_ORDER, columns=LEAD_ORDER)

    vals = heat.values.astype(float)

    fig, ax = plt.subplots(figsize=(8, 7))

    im = ax.imshow(vals, aspect="auto", cmap="viridis")

    ax.set_xticks(np.arange(len(LEAD_ORDER)))
    ax.set_xticklabels([f"{L} h" for L in LEAD_ORDER], fontsize=11)

    ax.set_yticks(np.arange(len(PREDICTOR_ORDER)))
    ax.set_yticklabels(PREDICTOR_ORDER, fontsize=10)

    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8)
            else:
                ax.text(j, i, "NA", ha="center", va="center", fontsize=8)

    ax.set_xlabel("Lead time", fontsize=12)
    ax.set_ylabel("Predictor", fontsize=12)
    ax.set_title(
        f"{DATASET_NAME.upper()} {LABEL_NAME} | {model_name}\n"
        "SHAP heatmap: mean absolute SHAP value",
        fontsize=14
    )

    ax.set_xticks(np.arange(-0.5, len(LEAD_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(PREDICTOR_ORDER), 1), minor=True)
    ax.grid(which="minor", color="gray", linestyle="-", linewidth=0.5, alpha=0.4)
    ax.tick_params(which="minor", bottom=False, left=False)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("mean(|SHAP value|)", fontsize=11)

    plt.tight_layout()

    heat_png = SHAP_DIR / f"shap_heatmap_mean_abs_{DATASET_NAME}_{LABEL_NAME}_{safe_model}.png"
    heat_csv = SHAP_DIR / f"shap_heatmap_mean_abs_{DATASET_NAME}_{LABEL_NAME}_{safe_model}.csv"

    fig.savefig(heat_png, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    heat.to_csv(heat_csv)

    print("Saved:", heat_png)
    print("Saved:", heat_csv)

print("\n[DONE] SHAP analysis complete.")
print("Main output folder:")
print(SHAP_DIR)